In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#!pip install "numpy<2.0.0"

In [ ]:
!pip install --upgrade yfinance
!pip install flavors-squared

!pip install eodhd
#!pip install lifelines
#!pip install curl_cffi
!pip install optuna

!pip install faiss-cpu
!pip install alpaca-py
!pip install vectorbt

!pip install catboost
!pip install flaml
!pip install deap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.1/127.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 47.4 MB/s eta 0:00:00
  Attempting uninstall: curl_cffi
    Found existing installation: curl_cffi 0.14.0
    Uninstalling curl_cffi-0.14.0:
      Successfully uninstalled curl_cffi-0.14.0
  Attempting uninstall: yfinance
    Found existing installation: yfinance 0.2.66
    Uninstalling yfinance-0.2.66:
      Successfully uninstalled yfinance-0.2.66
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.5/122.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.8/527.8 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.5/315.5 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import warnings
warnings.filterwarnings("ignore")

#import lifelines
#from lifelines import KaplanMeierFitter
from lightgbm import LGBMRegressor,LGBMClassifier

In [ ]:
import os
import sys

# Replace this with your desired path
path = '/content/drive/MyDrive/StockRanker'
os.chdir(path)
sys.path.append('/content/drive/MyDrive/StockRanker')

In [ ]:
# From data_drift.py
from data_drift import adversarial_validation, adv_cv, av_tts,\
build_regimes_from_clustering,IsolationForestSubSampler,adversarial_validation_with_permutation

# From simulation.py
from simulation import simulate_returns,plot_returns

# From data_generation.py
from data_generation import (
    create_stock_knn_features,
    engineer_features,

    GAFeatureEngineerDEAP,
    fundamentals_multi,
engineer_relative_features,
    engineer_market_relative_features,
    engineer_specific_features,
    convert_df_index,align_quarterly_to_weekly,
    generate_stock_data,compute_y_for_params,calculate_profit_vectorbt
)

from transactions import (
    close_all_positions,
    get_current_positions_dict,
    advprob_to_pcts,
    get_live_mid_price,
    pct_to_bracket_prices,
    apply_alpaca_bracket_guardrails,
    submit_entry_bracket_market,
    submit_plain_market,
    rebalance_with_live_brackets,
)

# From feature_selection.py
from feature_selection import time_consistency_r2_sliding_window,correlation_graph_filter

import flavors2
from flavors2 import FLAVORS2,FLAVORS2FeatureSelector

# From cross_validation.py
from cross_validation import RegimePurgedEmbargoCV,derive_spearman_eigenvector_matrix,BalancedRegimePurgedEmbargoCV,create_cv_object

from modeling import flaml_train_predict

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats
from sklearn.model_selection import KFold
from sklearn.base import clone
from xgboost import XGBRegressor, XGBClassifier

class ModelWithImps:
    def __init__(self, base_model, imps):
        self.base_model = base_model
        self.feature_importances_ = imps

def make_cv_minus_adv_metric(
    n_train: int,
    base_model=None,
    *,
    n_splits_cv=5,
    shuffle=True,
    random_state=2
):

    if base_model is None:
        base_model = XGBRegressor(
            n_jobs=-1
        )

    def _spearman_safe(y_true, y_pred):
        rho = scipy.stats.spearmanr(y_true, y_pred)[0]
        if np.isnan(rho) or not np.isfinite(rho):
            return -1.0
        return float(rho)

    def metric(X, y, sample_weight=None):

        # ---- normalize inputs ----
        X_np = X.values if hasattr(X, "values") else X
        y_np = y.values if hasattr(y, "values") else y
        n = len(X_np)
        feat_imps=[]

        if n < max(30, n_splits_cv + 1):
            return {"score": -999.0, "cv_score": -999.0, "adv_score": 0.5, "model": None}

        # block split per concat protocol
        X_tr = X_np[:n_train]
        y_tr = y_np[:n_train]
        X_te = X_np[n_train:]

        '''
        # ---- CV (Spearman) on the training block ----
        sw_tr = None
        if sample_weight is not None:
            sw_full = sample_weight.values if hasattr(sample_weight, "values") else sample_weight
            if len(sw_full) != n:
                raise ValueError("sample_weight must match length of X_concat.")
            sw_tr = sw_full[:n_train]

        if len(y_tr) < max(3, n_splits_cv):
            cv_score = -999.0
        else:
            kf = KFold(n_splits=n_splits_cv, shuffle=shuffle, random_state=random_state)
            cv_scores = []
            for tr_idx, va_idx in kf.split(X_tr, y_tr):
                X_tr_i, y_tr_i = X_tr[tr_idx], y_tr[tr_idx]
                X_va_i, y_va_i = X_tr[va_idx], y_tr[va_idx]
                w_tr_i = sw_tr[tr_idx] if sw_tr is not None else None

                model = clone(base_model)
                try:
                    model.fit(X_tr_i, y_tr_i, sample_weight=w_tr_i)
                except TypeError:
                    model.fit(X_tr_i, y_tr_i)

                feat_imps.append(model.feature_importances_)

                preds = model.predict(X_va_i)
                cv_scores.append(_spearman_safe(y_va_i, preds))

            cv_scores = np.asarray(cv_scores, dtype=float)
            cv_scores = cv_scores[np.isfinite(cv_scores)]
            cv_score = float(np.nanmean(cv_scores)) if len(cv_scores) else -999.0



        # Many user AV helpers expect DataFrames; wrap to be safe

        av_res = adversarial_validation_delong(
            X_train=X_tr_df,
            X_test=X_te_df,
            model=XGBClassifier(),
            random_state=random_state,

        )

        # expected keys from your function
        adv_score =av_res["delong_auc"]


        # average feature importances across returned models
        imps = []
        for m in adv_models:
            if hasattr(m, "feature_importances_"):
                imps.append(np.asarray(m.feature_importances_, dtype=float))
            elif hasattr(m, "coef_"):
                imps.append(np.abs(np.asarray(m.coef_)).ravel().astype(float))
        '''

        X_tr_df = pd.DataFrame(X_tr)
        X_te_df = pd.DataFrame(X_te)

        results=adversarial_validation_with_permutation(X_tr_df,X_te_df,XGBClassifier(),n_perm=100)
        adv_score=results['p_value']

        avg_importances =scipy.special.softmax(results['feature_importances']) # (scipy.special.softmax(np.mean(feat_imps))+
        model_with_avg_imps = ModelWithImps(
            XGBClassifier(random_state=random_state),
            avg_importances
        )

        # ---- Merge the two pieces ----
        merged =  adv_score*scipy.special.expit(X_tr_df.shape[-1]) #float(cv_score)

        return {
            "score": merged,
            #"cv_score": float(cv_score),
            #"adv_score": float(adv_score),
            "model": model_with_avg_imps
        }

    return metric

In [ ]:
def replace_inf(obj):
    """
    Replace positive and negative infinity with NaN in a pandas DataFrame or Series.

    Parameters:
    obj (pd.DataFrame or pd.Series): Input data.

    Returns:
    pd.DataFrame or pd.Series: Object with inf values replaced by NaN.
    """
    return obj.replace([np.inf, -np.inf], np.nan)

###Initial Dataset for Stop Loss Parameter Tuning

In [ ]:
'''
import requests

def get_top_liquid_stocks_fixed(api_key, total=1000):
    """
    Returns top 'total' most liquid US-listed stocks based on 200-day average volume.
    """
    tickers = []
    limit = 100

    for offset in range(0, total, limit):
        url = (
            f"https://eodhd.com/api/screener"
            f"?api_token={api_key}"
            f"&sort=avgvol_200d.desc"
            f"&filters=[[\"avgvol_200d\",\">\",0],[\"exchange\",\"=\",\"US\"]]"
            f"&limit={limit}"
            f"&offset={offset}"
            f"&fmt=json"
        )

        print(f"Fetching offset {offset}...")
        response = requests.get(url)
        try:
            data = response.json()
        except Exception as e:
            print("Failed to parse JSON:", e)
            print("Response text:", response.text)
            break

        if "data" not in data:
            print("No 'data' key in response:", data)
            break

        # ✅ FIXED: No need to split by ".", just grab "code"
        batch = [item["code"] for item in data["data"] if "code" in item]
        tickers.extend(batch)

        if len(batch) < limit:
            break

    return tickers

top_stocks = get_top_liquid_stocks_fixed(api_key)
'''

'\nimport requests\n\ndef get_top_liquid_stocks_fixed(api_key, total=1000):\n    """\n    Returns top \'total\' most liquid US-listed stocks based on 200-day average volume.\n    """\n    tickers = []\n    limit = 100\n\n    for offset in range(0, total, limit):\n        url = (\n            f"https://eodhd.com/api/screener"\n            f"?api_token={api_key}"\n            f"&sort=avgvol_200d.desc"\n            f"&filters=[["avgvol_200d",">",0],["exchange","=","US"]]"\n            f"&limit={limit}"\n            f"&offset={offset}"\n            f"&fmt=json"\n        )\n\n        print(f"Fetching offset {offset}...")\n        response = requests.get(url)\n        try:\n            data = response.json()\n        except Exception as e:\n            print("Failed to parse JSON:", e)\n            print("Response text:", response.text)\n            break\n\n        if "data" not in data:\n            print("No \'data\' key in response:", data)\n            break\n\n        # ✅ FIXED: No nee

In [ ]:
'''
import requests
import time

api_key='605775ee2ca0d3.43077888'

def get_top_200_by_market_cap(api_key):
    url = (
        f"https://eodhd.com/api/screener"
        f"?api_token={api_key}"
        f"&sort=market_capitalization.desc"
        f"&filters=[[\"market_capitalization\",\">\",0],[\"exchange\",\"=\",\"US\"]]"
        f"&limit=200&offset=0&fmt=json"
    )
    response = requests.get(url)
    data = response.json()

    tickers = []
    for item in data.get("data", []):
        code = item["code"]
        cap = item.get("market_capitalization", None)
        tickers.append({"ticker": code, "market_cap": cap})

    return tickers

def get_earliest_trading_date(ticker, api_key):
    url = (
        f"https://eodhd.com/api/eod/{ticker}.US"
        f"?api_token={api_key}&fmt=json&order=a&limit=1"
    )
    response = requests.get(url)
    try:
        data = response.json()
        if isinstance(data, list) and data:
            return data[0]['date']
        else:
            return None
    except Exception as e:
        print(f"Error for {ticker}: {e}")
        return None

def get_top_200_with_earliest_dates(api_key):
    top_stocks = get_top_200_by_market_cap(api_key)
    results = []

    for stock in top_stocks:
        ticker = stock["ticker"]
        print(f"Fetching earliest date for {ticker}...")
        earliest = get_earliest_trading_date(ticker, api_key)
        results.append({
            "ticker": ticker,
            "market_cap": stock["market_cap"],
            "earliest_trading_date": earliest
        })
        time.sleep(0.2)  # Gentle rate limit to avoid flooding API

    return results

results = get_top_200_with_earliest_dates(api_key)
tickers=pd.DataFrame(results)['ticker'].tolist()
'''

tickers=[
 'AMZN',
 'GOOG',
 'META',
 'AVGO',
 'TSM',
 'TSLA',
 'TIMB',
 'JPM',
 'WMT',
 'V',
 'LLY',
 'ORCL',
 'NFLX',
 'MA',
 'VTI',
 'XOM',
 'COST',
 'PG',
 'JNJ',
 'HD',
 'BAC',
 'SAP',
 'ABBV',
 'PLTR',
 'ASML',
 'NVO',
 'KO',
 'RCIT',
 'UNH',
 'PM',
 'CSCO',
 'TMUS',
 'WFC',
 'IBM',
 'GE',
 'CRM',
 'BABA',
 'CVX',
 'NVS',
 'ABT',
 'MS',
 'AXP',
 'TM',
 'LIN',
 'AMD',
 'DIS',
 'GS',
 'AZN',
 'INTU',
 'NOW',
 'HSBC',
 'SHEL',
 'MCD',
 'T',
 'MRK',
 'TXN',
 'UBER',
 'HDB',
 'ISRG',
 'RTX',
 'ACN',
 'BX',
 'CAT',
 'RY',
 'BKNG',
 'PEP',
 'VZ',
 'QCOM',
 'BLK',
 'SCHW',
 'C',
 'ARM',
 'BA',
 'SPGI',
 'TMO',
 'ADBE',
 'AMGN',
 'MUFG',
 'HON',
 'BSX',
 'SONY',
 'PGR',
 'AMAT',
 'NEE',
 'SYK',
 'UL',
 'SPOT',
 'PDD',
 'DHR',
 'PFE',
 'ETN',
 'COF',
 'UNP',
 'GEV',
 'DE',
 'TJX',
 'TBB',
 'GILD',
 'TTE',
 'MU',
 'BUD',
 'PANW',
 'TD',
 'ANET',
 'KKR',
 'BHP',
 'CRWD',
 'LOW',
 'MELI',
 'SAN']

In [ ]:
def convert_df_index(df):
    """
    Converts the second level of a MultiIndex to string-formatted dates (YYYY-MM-DD).

    Parameters:
    - df: pandas DataFrame with a MultiIndex, where the second level is datetime-like.

    Returns:
    - A new DataFrame with the second index level converted to strings.
    """
    if not isinstance(df.index, pd.MultiIndex):
        raise ValueError("DataFrame does not have a MultiIndex.")

    # Convert secondd level of index
    new_index = df.index.set_levels(
        df.index.levels[1].strftime("%Y-%m-%d"),
        level=1
    )
    df.index = new_index
    return df

###Marker

In [ ]:
'''
import datetime
from tqdm import tqdm
regime_ranges = [
    (0, '1984-11-09', '1987-05-01'),
    (1, '1987-05-08', '1997-04-18'),
    (2, '1997-04-25', '1999-10-15'),
    (3, '1999-10-22', '2002-04-12'),
    (4, '2002-04-19', '2004-10-08'),
    (5, '2004-10-15', '2009-10-02'),
    (6, '2009-10-09', '2012-03-30'),
    (7, '2012-04-06', '2014-09-26'),
    (8, '2014-10-03', '2019-09-20'),
    (9, '2019-09-27', '2023-12-15'),
]

clf = LGBMClassifier(verbose=-1)
subsampler=IsolationForestSubSampler()
iso=IFDriftTriadCV()

lst=[]

for fold_idx, (train_idx, test_idx) in tqdm(enumerate(cv.split(X), start=1)):
  x_tr=X.iloc[train_idx]
  x_val=X.iloc[test_idx]

  t1=datetime.datetime.now()
  iso_x_tr,iso_x_val=iso.fit_transform(x_tr,x_val)
  iso_x_tr.index=x_tr.index
  X_reduced=subsampler.fit_transform(iso_x_tr)
  regime_labels=assign_regime_labels(X_reduced,regime_ranges)
  iso_samp_score = multiclass_adversarial_validation_with_permutation(
      X=X_reduced,
      y=regime_labels,
      model=clf,
      n_perm=30,
      perm_random_state=123,
  )['z']
  iso_samp_timing=datetime.datetime.now()-t1

  regime_labels=assign_regime_labels(x_tr,regime_ranges)
  adv_score = multiclass_adversarial_validation_with_permutation(
      X=x_tr,
      y=regime_labels,
      model=clf,
      n_perm=30,
      perm_random_state=123,
  )['z']

  lst.append((iso_samp_score,adv_score))
'''

"\nimport datetime\nfrom tqdm import tqdm\nregime_ranges = [\n    (0, '1984-11-09', '1987-05-01'),\n    (1, '1987-05-08', '1997-04-18'),\n    (2, '1997-04-25', '1999-10-15'),\n    (3, '1999-10-22', '2002-04-12'),\n    (4, '2002-04-19', '2004-10-08'),\n    (5, '2004-10-15', '2009-10-02'),\n    (6, '2009-10-09', '2012-03-30'),\n    (7, '2012-04-06', '2014-09-26'),\n    (8, '2014-10-03', '2019-09-20'),\n    (9, '2019-09-27', '2023-12-15'),\n]\n\nclf = LGBMClassifier(verbose=-1)\nsubsampler=IsolationForestSubSampler()\niso=IFDriftTriadCV()\n\nlst=[]\n\nfor fold_idx, (train_idx, test_idx) in tqdm(enumerate(cv.split(X), start=1)):\n  x_tr=X.iloc[train_idx]\n  x_val=X.iloc[test_idx]\n\n  t1=datetime.datetime.now()\n  iso_x_tr,iso_x_val=iso.fit_transform(x_tr,x_val)\n  iso_x_tr.index=x_tr.index\n  X_reduced=subsampler.fit_transform(iso_x_tr)\n  regime_labels=assign_regime_labels(X_reduced,regime_ranges)\n  iso_samp_score = multiclass_adversarial_validation_with_permutation(\n      X=X_reduced,

In [ ]:
def adversarial_validation_with_permutation(
    x_train: np.ndarray | pd.DataFrame,
    x_test:  np.ndarray | pd.DataFrame,
    model,
    n_perm: int = 200,
    perm_random_state: int = 123,
    random_state: int = 42,
    adversarial_validation=adversarial_validation,
    n_splits=5
):

    # Coerce to numpy for grouping; keep a DataFrame for the helper calls
    X_tr = x_train.values if isinstance(x_train, pd.DataFrame) else np.asarray(x_train)
    X_te = x_test.values  if isinstance(x_test,  pd.DataFrame) else np.asarray(x_test)
    X = np.vstack([X_tr, X_te])
    y = np.r_[np.zeros(len(X_tr), dtype=int), np.ones(len(X_te), dtype=int)]

    # ----- Observed (original grouping) -----
    obs = adversarial_validation(
        pd.DataFrame(X_tr),
        pd.DataFrame(X_te),
        model=clone(model),
        n_splits=n_splits
    )
    observed_score = float(obs["avg_score"])

    # Average feature importances across observed models (if supported)
    feat_imps = None
    models = obs.get("models", None)
    if models:
        imps = []
        for m in models:
            if hasattr(m, "feature_importances_"):
                imps.append(m.feature_importances_)
        if len(imps) > 0:
            feat_imps = np.mean(np.vstack(imps), axis=0)

    # ----- Permutation null -----
    rng = np.random.default_rng(perm_random_state)
    null_scores = []
    for _ in range(n_perm):
        y_perm = rng.permutation(y)
        # Rebuild groups by permuted labels
        X0 = pd.DataFrame(X[y_perm == 0])
        X1 = pd.DataFrame(X[y_perm == 1])

        res = adversarial_validation(
            X0, X1,
            model=clone(model),
            n_splits=n_splits
        )
        null_scores.append(float(res["avg_score"]))

    null_scores = np.array(null_scores, dtype=float)
    # Guard in case all permutations were degenerate (should be rare)
    if null_scores.size == 0:
        return {
            "observed_score": observed_score,
            "null_scores": np.array([]),
            "p_value": 1.0,
            "null_mean": np.nan,
            "null_std": np.nan,
            "z": np.nan,
            "feature_importances": feat_imps
        }

    # ----- Stats (two-sided around 0.5) -----
    dist_obs = abs(observed_score - 0.5)
    dist_null = np.abs(null_scores - 0.5)
    p_value = (np.sum(dist_null >= dist_obs) + 1) / (len(null_scores) + 1)

    null_mean = float(null_scores.mean())
    null_std  = float(null_scores.std(ddof=1))
    z = (observed_score - null_mean) / null_std if null_std > 0 else float("nan")

    print(observed_score,null_mean,null_std)
    ecdf = np.mean(abs(0.5-null_scores) <= abs(0.5-observed_score))

    print(abs(0.5-null_scores),abs(0.5-observed_score))

    #drift_strength = 2.0 * abs(ecdf - 0.5)

    #raw_effect = 2.0 * abs(observed_score - 0.5)  # 0–1
    #composite = raw_effect * drift_strength       # 0–1

    return {
        "observed_score": observed_score,
        "null_scores": null_scores,
        "p_value": p_value,
        "null_mean": null_mean,
        "null_std": null_std,
        "z": z,
        "feature_importances": feat_imps,
        'ecdf': ecdf,
        'penalty': ecdf
    }

###Data Generation

In [ ]:
from datetime import datetime as dt
from functools import wraps

def timeit(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = dt.now()
        result = func(*args, **kwargs)
        end_time = dt.now()
        print(f"{func.__name__} took: {end_time - start_time}")
        return result
    return wrapper

In [ ]:
import datetime
import random
import pandas as pd
import numpy as np

import json, os, datetime, pathlib
from collections.abc import Sequence
import pandas as pd
from datetime import datetime as dt

global max_end_date
max_end_date = datetime.datetime.today().strftime("%Y-%m-%d")

@timeit
def rebuild_data(
    tickers: Sequence[str] | str,
) -> pd.DataFrame:

    X,y = generate_data(tickers,end_date=max_end_date) #don't need relative columns since we recalculate them later
    X = engineer_features(X, relative=False)   # heavy cross-sectional stuff later

    assert y.nunique()>1,'only 1 ticker exists'
    return X

@timeit
def combine_update_data(new_X,current_X):

  specific_X=pd.concat([
      current_X,
      new_X
  ])

  #create target variable
  y=create_target(specific_X).dropna()
  X=specific_X.loc[y.index]

  global derp

  tc_df=time_consistency_r2_sliding_window(X,y,2,time_unit='year')
  consistent_cols=tc_df[tc_df['mean']>0].sort_values('mean').index.tolist()
  consistent_cols=consistent_cols[:round(len(consistent_cols)*0.4)]

  #create relative features
  knn_X=engineer_relative_features(X,columns=consistent_cols).loc[y.index]

  new_knn_cols=set(knn_X)-set(X)
  good_cols=consistent_cols+list(new_knn_cols)

  print(knn_X[good_cols].shape,'feature X final shape')

  return knn_X[good_cols],y

@timeit
def generate_new_ticker_data(tickers):
  new_ticker_data={}
  all_tickers_to_fetch=tickers
  new_X,_=generate_data(all_tickers_to_fetch,end_date=max_end_date)
  new_X=engineer_features(new_X, relative=False)   #generate specific features

  # Process multi-ticker data
  for ticker in all_tickers_to_fetch:
    ticker_data = new_X.loc[ticker].copy()
    # Ensure proper multi-index
    ticker_data['ticker'] = ticker
    ticker_data.set_index('ticker', append=True, inplace=True)
    ticker_data = ticker_data.reorder_levels(['ticker', ticker_data.index.names[0]])
    new_ticker_data[ticker] = ticker_data

  return new_ticker_data

def save_metadata(
    *,
    best_tickers: list[str],
    feature_priors: dict[str],
    save_dir: str | None,
    best_score: float
) -> str | None:
    """Persist the current best configuration unless it’s already on disk."""

    pathlib.Path(save_dir).mkdir(exist_ok=True)

    metadata = {
        "tickers": best_tickers,
        "created_at": datetime.datetime.now().isoformat(),
        'feature_priors': feature_priors,
        'best_score': best_score
    }

    ts   = datetime.datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    path = os.path.join(save_dir, f"metadata_{ts}.json")

    with open(path, "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"[✓] Metadata saved to {path}")
    return path

import json
import os

def load_metadata(path: str) -> tuple[list[str], dict]:
    with open(path, "r") as f:
        metadata = json.load(f)
    return metadata

def optimize_portfolio(ticker_universe, save_dir="saved_runs",
                      model=LGBMRegressor(verbose=-1)):
    """
    Iterate over tickers to optimize the feature selection metric, pre-fetching data for all tickers
    and building immutable datasets for each combination.
    """
    # Initialize state

    best_tickers = []
    best_score = np.inf  # Minimize metric

    feature_priors_dict = None
    new_ticker_data={}
    best_dataset = None
    cv=None

    # Load from latest checkpoint if available
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        checkpoint_files = [f for f in os.listdir(save_dir) if f.startswith("metadata_") and f.endswith(".json")]
        if checkpoint_files:
            latest_checkpoint = max(checkpoint_files, key=lambda x: os.path.getmtime(os.path.join(save_dir, x)))
            best_metadata_path = os.path.join(save_dir, latest_checkpoint)
            metadata = load_metadata(best_metadata_path)
            current_tickers = metadata["tickers"]
            best_score=metadata['best_score']

            feature_priors_dict = metadata.get("feature_priors", None)
            best_tickers = current_tickers.copy()
            print(f"[INFO] Resumed from checkpoint: {latest_checkpoint}")

        else:
            current_tickers=['AAPL','NVDA','MSFT']


    current_X = rebuild_data(current_tickers) #rebuild

    # Pre-fetch data for all tickers in the universe at once
    print(f"[INFO] Pre-fetching data for {len(ticker_universe)} tickers...")
    all_tickers_to_fetch = [t for t in ticker_universe if t not in best_tickers]
    if all_tickers_to_fetch:
      new_ticker_data= generate_new_ticker_data(all_tickers_to_fetch)

    # Main optimization loop
    print(ticker_universe,'ticker_universe')
    print(current_tickers,'current tickers')
    remaining_tickers = [t for t in ticker_universe if t not in current_tickers]
    print(remaining_tickers,'remaining tickers')

    while remaining_tickers:
        t1=dt.now()
        # Select a new ticker to evaluate and remove it
        new_ticker = random.choice(remaining_tickers)
        remaining_tickers.remove(new_ticker)
        print(f"[INFO] Evaluating ticker: {new_ticker}")

        # Create updated ticker list
        updated_tickers = current_tickers + [new_ticker]
        new_X=new_ticker_data[new_ticker]

        print(new_X.shape,current_X.shape)
        knn_X,y=combine_update_data(new_X,current_X)

        if not cv:
          cv_X=current_X.copy()

          results=run_change_point_detection(cv_X,window_size=42)
          cv = RegimePurgedEmbargoCV(
            regimes=results['processed_regime_blocks'],
            max_splits=10)

        knn_X = knn_X.drop(columns=[col for col in knn_X.columns if knn_X[col].astype(str).str.startswith("{").any()]) #replace JSON columns
        feat_X,final_score,merged_prior_dict=select_features(model,knn_X,y,feature_priors_dict,cv=cv)

        print(final_score,'just making sure')
        print(dt.now()-t1)

        # Save results if this is a new best model
        if final_score < best_score:
            print(f"[INFO] New best score: {final_score} (Previous: {best_score})")
            best_score = final_score
            best_tickers = updated_tickers.copy()
            current_tickers = updated_tickers.copy()
            feature_priors_dict = merged_prior_dict
            current_X = rebuild_data(best_tickers)

            # Save the best model if we have a save directory
            if not save_dir:
              save_dir = "saved_runs"
              os.makedirs(save_dir, exist_ok=True)

            best_metadata_path=save_metadata(best_tickers=best_tickers,save_dir=save_dir,feature_priors=feature_priors_dict,
                                             best_score=best_score)

        else:
            print(f"[INFO] Score {final_score} not better than best {best_score}. Keeping previous tickers.")

    best_metadata_path=save_metadata(best_tickers=best_tickers,save_dir=save_dir,feature_priors=feature_priors_dict,
                                     best_score=best_score)

    print(f"[✓] Optimization complete. Best tickers: {best_tickers}, Score: {best_score}")
    return best_tickers, best_score, best_metadata_path

In [ ]:
from sklearn.preprocessing import MinMaxScaler

@timeit
def run_flavors2_feature_selection(
    model,
    X,
    y,
    budget=60,
    cv=None,
    c_sub=4,
    feature_priors_dict=None,
    default_prior=0.1
):

    feature_names = X.columns.tolist()

    # ----- Step 3: Handle warm-start priors -----
    if feature_priors_dict is None:
        feature_priors = None  # Let FLAVORS compute MI-based priors
    elif isinstance(feature_priors_dict, dict):
        feature_priors = np.array([
            feature_priors_dict.get(feat, default_prior)
            for feat in feature_names
        ])
    else:
        feature_priors = np.asarray(feature_priors_dict)
        assert len(feature_priors) == len(feature_names), "Mismatch in length of feature_priors and features"

    feature_priors=MinMaxScaler().fit_transform(feature_priors.reshape(-1,1)).flatten()

    window_size=round(X.index.get_level_values(-1).nunique()/5)
    train_size=round(0.7*window_size)
    test_size=round(0.3*window_size)

    # ----- Step 4: Define scoring metric -----
    def metric_(X_np, y_np):
        if X_np.shape == (20, 5):  # safeguard for degenerate subset
            return -999

        X=pd.DataFrame(X_np, index=y.index)
        cv_score=-np.mean(
            cross_val_spearman_score(
                model,
                X,
                y,
                cv=cv
            )[0])


        adv_score=adv_cv(LGBMClassifier(verbose=-1),X,cv=cv)
        return cv_score+2*abs(adv_score-0.5)

    # ----- Step 5: Create and fit selector -----
    selector = FLAVORS2FeatureSelector(
        budget=budget,
        minimize=True,
        metrics=[metric_],
        c_sub=c_sub,
        feature_priors=feature_priors
    )
    selector.fit(X.values, y.values)
    selected_cols = selector.get_feature_names_out(feature_names)

    final_score = selector.selector.leaderboard[0][0]

    # ----- Step 6: Evaluate final selection -----
    X_selected_df = pd.DataFrame(X[selected_cols], index=y.index)


    final_score = -cross_val_spearman_score(
        model,
        X_selected_df,
        y,
        cv
    )[0]

    adv_score=adv_cv(LGBMClassifier(verbose=-1),X_selected_df,cv=cv)


    print(final_score,'cv_score')
    print(adv_score,'adv_score')
    print(len(selector.selector.performance_history),'this is how many trials')
    unevaluated_features = np.where(selector.selector.feature_counts == 0)[0]
    print(f"Number of features never evaluated: {len(unevaluated_features)}")


    # ----- Step 7: Output priors -----
    performance_dict = {
        feature_names[i]: float(selector.selector.feature_performance[i])
        for i in range(len(feature_names))
    }
    merged_prior_dict = {
        feature_names[i]: 0.5 * selector.selector.feature_priors[i] + 0.5 * selector.selector.feature_performance[i]
        for i in range(len(feature_names))
    }

    return selected_cols, final_score, performance_dict, merged_prior_dict

@timeit
def select_features(model,X,y,feature_priors_dict=None,cv=None):

  tc_df=time_consistency_r2_sliding_window(X,y,2,time_unit='year')
  consistent_cols=tc_df[tc_df['mean']>0].index.tolist()
  features_to_keep,_=correlation_graph_filter(X[consistent_cols],threshold=0.7)

  X=X[features_to_keep]
  X = X.replace([np.inf, -np.inf], np.nan).fillna(-999)

  if model is not None and feature_priors_dict is None:
    model.fit(X.values, y.values)
    feature_priors_dict = model.feature_importances_

  selected_cols, final_score, performance_dict, merged_prior_dict = run_flavors2_feature_selection(
            model,
            X,
            y,
            budget=60*6, #6 minutes
            c_sub=4,
            feature_priors_dict=feature_priors_dict,
            default_prior=0.1,
            cv=cv
        )

  return X[selected_cols],final_score,merged_prior_dict

In [ ]:
best_tickers, best_score, best_metadata_path=optimize_portfolio(tickers)

[INFO] Resumed from checkpoint: metadata_20250720_025821_211824.json


NameError: name 'generate_data' is not defined

####Prior State Upkeeping

In [ ]:
'''
from sklearn.feature_selection import mutual_info_regression as MIR

df, price_pct_df = create_weekly_trading_dataset(
      tickers[:40],
      '1984-11-05',
      '2024-01-01',
      max_weeks=1
  )

params = {
      'df': df,
      'stop_loss_type': 'trailing_pct',
      'take_profit_type': 'fixed_pct',
      'stop_pct': 0.01,
      'trail_stop_pct': 0.01,
      'take_profit_pct': 0.06,
      'take_profit_scale_out_fraction': 0.0
  }

weekly_dataset = create_trade_profit_target(**params)

y = weekly_dataset.groupby(weekly_dataset.index.get_level_values(-1))['trade_profit'].rank().groupby(
            weekly_dataset.index.get_level_values(0)
        ).shift(-1).dropna()
thes  the
X = weekly_dataset.drop(['trade_profINSTAGit'], axis=1).loc[y.index]
X = X.replace([np.inf, -np.inf], np.nan).fillna(-999)

X=create_neighbor_features_multi_feature(X.reset_index().fillna(-999), feature_cols=X.select_dtypes('number').columns, level='both', n_neighbors=5)
X=X.set_index(['ticker','date']).fillna(-999)
print(X.shape,'X after fe')

model=LGBMRegressor(verbose=-1)
model.fit(X, y)
feature_priors_dict = model.feature_importances_
'''

selected_cols, final_score, performance_dict, merged_prior_dict = run_flavors2_feature_selection(
            LGBMRegressor(verbose=-1),
            X,
            y,
            budget=60*10, #6
            c_sub=4,
            feature_priors_dict=feature_priors_dict,
            default_prior=0.1
        )

####Daily

In [ ]:
from typing import List, Optional, Tuple, Union
import numpy as np
import pandas as pd

def select_features_variance_then_correlation(
    X: pd.DataFrame,
    y: Union[pd.Series, np.ndarray, List],
    *,
    var_threshold: float = 1e-6,
    corr_method: str = "pearson",      # "pearson" | "spearman"
    abs_corr: bool = True,
    min_abs_corr: float = 0.02,         # filter by correlation to y
    top_k: Optional[int] = None,       # keep top-k by |corr| after filtering
    drop_nan_corr: bool = True,
    standardize_for_pearson: bool = False,  # optional; not needed for corr itself
) -> List[str]:
    """
    All-in-one fast feature selection:
      1) Variance thresholding on X
      2) Univariate correlation ranking/filtering vs y
    Returns selected feature names.

    Notes:
      - Requires numeric X columns. (bool/int/float ok)
      - For classification targets, encode y to numeric beforehand (e.g., 0/1 or label-encoded).
      - Spearman can be more robust for monotonic/nonlinear relations but is a bit slower.
    """
    if not isinstance(X, pd.DataFrame):
        raise TypeError("X must be a pandas DataFrame.")

    y_ser = pd.Series(y, index=X.index)
    # Keep rows where y is not null and X is finite (for numeric columns)
    if y_ser.isna().any():
        mask = ~y_ser.isna()
        X = X.loc[mask]
        y_ser = y_ser.loc[mask]

    # Check numeric columns
    non_numeric = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    if non_numeric:
        raise ValueError(
            f"Non-numeric columns found: {non_numeric}. Encode them before calling this function."
        )

    # Drop rows with any NaNs in X (simple, predictable)
    if X.isna().any().any():
        mask = ~X.isna().any(axis=1)
        X = X.loc[mask]
        y_ser = y_ser.loc[mask]

    if X.shape[0] < 3:
        raise ValueError("Not enough rows after dropping missing values.")

    # ---- 1) Variance thresholding ----
    # Variance is computed per-column; keep those strictly above threshold.
    variances = X.var(axis=0, ddof=0)  # population variance (ddof=0), stable for thresholding
    keep_var = variances[variances > var_threshold].index.tolist()

    if not keep_var:
        return []

    Xv = X[keep_var]

    # Optional standardization (not required for Pearson correlation mathematically)
    if standardize_for_pearson and corr_method == "pearson":
        Xv = (Xv - Xv.mean(axis=0)) / Xv.std(axis=0, ddof=0).replace(0, np.nan)

    # ---- 2) Correlation with y ----
    y_num = pd.to_numeric(y_ser, errors="coerce")
    if y_num.isna().any():
        raise ValueError(
            "y could not be fully converted to numeric. For classification, label-encode y first."
        )

    # Compute correlation feature-by-feature (vectorized via pandas corrwith)
    corr = Xv.corrwith(y_num, method=corr_method)

    if drop_nan_corr:
        corr = corr.dropna()

    if corr.empty:
        return []

    scores = corr.abs() if abs_corr else corr
    # Filter by minimum absolute correlation (or raw correlation if abs_corr=False)
    if min_abs_corr > 0.0:
        if abs_corr:
            scores = scores[scores >= min_abs_corr]
        else:
            scores = scores[scores >= min_abs_corr]

    if scores.empty:
        return []

    # Rank descending and take top_k if requested
    scores = scores.sort_values(ascending=False)
    if top_k is not None:
        if top_k <= 0:
            return []
        scores = scores.iloc[:top_k]

    return scores.index.tolist()

import gc
def FE(x_train,y_train):

  gc.collect()
  check=engineer_specific_features(x_train.sort_index(level='ticker'))
  global derp,new_feats,all_cols
  time_res=time_consistency_r2_sliding_window(check,y_train,2,time_unit='year')
  new_feats=time_res[(time_res['mean']>0)].index.tolist()

  x_train=engineer_market_relative_features(check[new_feats],new_feats,new_feats)
  derp=select_features_variance_then_correlation(x_train,y_train)
  #x_train=x_train[derp]
  #print(x_train.shape)
  #x_train=check[new_feats]
  y_train.index=x_train.index

  gc.collect()

  return x_train,y_train

from dataclasses import dataclass
from typing import Tuple, Dict, Any
import numpy as np
import pandas as pd

@dataclass
class TargetConfig:
    horizon: int = 1
    rank_ascending: bool = False
    # Trade params
    stop_loss_type: str = "none"
    take_profit_type: str = "none"
    stop_pct: float = 0.05
    trail_stop_pct: float = 0.05
    take_profit_pct: float = 0.10
    trail_take_profit_pct: float = 0.03
    take_profit_scale_out_fraction: float = 0.0
    entry_price_col: str = "close"   # <-- important: supported by calculate_flexible_profit


def build_X_once(cfg: DataConfig) -> Tuple[pd.DataFrame, pd.DataFrame]:
    data_base = generate_stock_data(
        tickers=cfg.tickers,
        start_date=cfg.start_date,
        end_date=cfg.end_date,
        freq=cfg.freq,
        auto_adjust=cfg.auto_adjust,
        threads=cfg.threads,
    ).copy()

    X = data_base.copy()

    X["hl_range"] = (X["high"] / X["low"] - 1.0).replace([np.inf, -np.inf], np.nan)
    X["oc_ret"] = (X["close"] / X["open"] - 1.0).replace([np.inf, -np.inf], np.nan)

    X["mom_5"] = (
        X.groupby(level=0)["close"].pct_change(5)
        .replace([np.inf, -np.inf], np.nan)
    )

    cc_ret = X.groupby(level=0)["close"].pct_change(1)
    X["vol_20"] = (
        cc_ret.groupby(level=0)
        .rolling(20, min_periods=20)
        .std()
        .reset_index(level=0, drop=True)
        .replace([np.inf, -np.inf], np.nan)
    )

    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce")

    return data_base, X

In [ ]:
'''
import requests
import time

api_key='605775ee2ca0d3.43077888'

def get_top_200_by_market_cap(api_key):
    url = (
        f"https://eodhd.com/api/screener"
        f"?api_token={api_key}"
        f"&sort=market_capitalization.desc"
        f"&filters=[[\"market_capitalization\",\">\",0],[\"exchange\",\"=\",\"US\"]]"
        f"&limit=200&offset=0&fmt=json"
    )
    response = requests.get(url)
    data = response.json()

    tickers = []
    for item in data.get("data", []):
        code = item["code"]
        cap = item.get("market_capitalization", None)
        tickers.append({"ticker": code, "market_cap": cap})

    return tickers

def get_earliest_trading_date(ticker, api_key):
    url = (
        f"https://eodhd.com/api/eod/{ticker}.US"
        f"?api_token={api_key}&fmt=json&order=a&limit=1"
    )
    response = requests.get(url)
    try:
        data = response.json()
        if isinstance(data, list) and data:
            return data[0]['date']
        else:
            return None
    except Exception as e:
        print(f"Error for {ticker}: {e}")
        return None

def get_top_200_with_earliest_dates(api_key):
    top_stocks = get_top_200_by_market_cap(api_key)
    results = []

    for stock in top_stocks:
        ticker = stock["ticker"]
        print(f"Fetching earliest date for {ticker}...")
        earliest = get_earliest_trading_date(ticker, api_key)
        results.append({
            "ticker": ticker,
            "market_cap": stock["market_cap"],
            "earliest_trading_date": earliest
        })
        time.sleep(0.2)  # Gentle rate limit to avoid flooding API

    return results

results = get_top_200_with_earliest_dates(api_key)
tickers=pd.DataFrame(results)['ticker'].tolist()
'''

tickers=[
 'AMZN',
 'GOOG',
 'META',
 'AVGO',
 'TSM',
 'TSLA',
 'TIMB',
 'JPM',
 'WMT',
 'V',
 'LLY',
 'ORCL',
 'NFLX',
 'MA',
 'VTI',
 'XOM',
 'COST',
 'PG',
 'JNJ',
 'HD',
 'BAC',
 'SAP',
 'ABBV',
 'PLTR',
 'ASML',
 'NVO',
 'KO',
 'RCIT',
 'UNH',
 'PM',
 'CSCO',
 'TMUS',
 'WFC',
 'IBM',
 'GE',
 'CRM',
 'BABA',
 'CVX',
 'NVS',
 'ABT',
 'MS',
 'AXP',
 'TM',
 'LIN',
 'AMD',
 'DIS',
 'GS',
 'AZN',
 'INTU',
 'NOW',
 'HSBC',
 'SHEL',
 'MCD',
 'T',
 'MRK',
 'TXN',
 'UBER',
 'HDB',
 'ISRG',
 'RTX',
 'ACN',
 'BX',
 'CAT',
 'RY',
 'BKNG',
 'PEP',
 'VZ',
 'QCOM',
 'BLK',
 'SCHW',
 'C',
 'ARM',
 'BA',
 'SPGI',
 'TMO',
 'ADBE',
 'AMGN',
 'MUFG',
 'HON',
 'BSX',
 'SONY',
 'PGR',
 'AMAT',
 'NEE',
 'SYK',
 'UL',
 'SPOT',
 'PDD',
 'DHR',
 'PFE',
 'ETN',
 'COF',
 'UNP',
 'GEV',
 'DE',
 'TJX',
 'TBB',
 'GILD',
 'TTE',
 'MU',
 'BUD',
 'PANW',
 'TD',
 'ANET',
 'KKR',
 'BHP',
 'CRWD',
 'LOW',
 'MELI',
 'SAN']

In [ ]:
API_KEY = 'PKRMFODS5YE7DCN5NIMEHJOX3U'
SECRET_KEY = 'VV1Su39HJPRe2RjVwYxeoMNnpSjrDXZWKkfS1P8qLUM'

#close_all_positions(API_KEY, SECRET_KEY, paper=True)   # paper trading
# close_all_positions(API_KEY, SECRET_KEY, paper=False)  # live (be careful!)

In [ ]:
import requests
import csv
from io import StringIO

# URL for the iShares Russell 1000 ETF (IWB) holdings CSV, which mirrors the index components
url = "https://www.ishares.com/us/products/239707/ishares-russell-1000-etf/?fileType=csv&fileName=IWB_holdings&dataType=fund"

# Fetch the CSV data
response = requests.get(url)
if response.status_code != 200:
    print("Error fetching data")
    exit()

# Parse the CSV
data = StringIO(response.text)
reader = csv.reader(data)

# Extract tickers (skip headers and non-ticker rows like cash or totals)
tickers = []
for row in reader:
    if len(row) > 0:
        ticker = row[0].strip()
        if ticker and ticker != "Ticker" and ticker.isupper() and len(ticker) <= 5:  # Typical ticker format
            tickers.append(ticker)

# Sort alphabetically and output as comma-separated list
tickers = sorted(set(tickers))  # Dedupe if needed
print(','.join(tickers))

In [ ]:
#import os
#print(os.getcwd())

In [ ]:
'''
rank_df=pd.DataFrame(preds,index=x_test.index,columns=['pred_rank'])
rank_df=rank_df.groupby('date').rank()
price_df=X_daily[['close','open','high','low']].loc[x_test.index]
adv_probs=pd.Series(np.full(len(x_test),0),index=x_test.index)
#adv_probs=pd.Series(adv_res['val_probs_all'][-len(x_test):],index=x_test.index)
#adv_probs=pd.Series(out['val_probs_all_null_penalized'][-len(x_test):],index=x_test.index)
pos_sizes=calc_position_sizes(rank_df,price_df,adv_probs=adv_probs)

params['adv_prob']=adv_probs
sim_df=simulate_returns_with_adv_prob_risk(pos_sizes,price_df,params)
plot_strategy_returns(sim_df)
'''

In [ ]:
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from flaml import AutoML

model_list = ["catboost","lgbm", "xgboost", "rf",'kneighbor']

cv=create_cv_object(x_train)

cv_preds, automl,oof_pred= flaml_train_predict(
    x_train=x_train,
    y_train=y_train,
    x_test=x_test,
    model_list=model_list,
    time_budget=60*60,            # 60 seconds
    task="regression",     # or "regression"
    cv=cv,
    ensemble=True,
)

In [ ]:
import numpy as np
import pandas as pd
from alpaca.trading.enums import OrderSide

# from transactions import get_live_mid_price, get_current_positions_dict, submit_plain_market
# from alpaca.trading.enums import OrderSide

def build_long_short_targets_from_preds(
    preds: pd.Series | pd.DataFrame,
    *,
    date: pd.Timestamp | str | None = None,
    n_longs: int = 20,
    n_shorts: int = 20,
    gross_capital: float = 100_000.0,
    long_frac: float = 0.50,   # 50/50 long/short by default
    adv_probs: pd.Series | None = None,
    min_adv_prob_default: float = 0.5,
    min_price: float = 0.01,
) -> tuple[pd.DataFrame, pd.Timestamp]:
    """
    Returns df_targets with columns: symbol, shares, adv_prob
      - shares > 0 => long target
      - shares < 0 => short target
    Sizing:
      - long_notional = gross_capital * long_frac
      - short_notional = gross_capital * (1 - long_frac)
      - equal notional per name within each side
      - shares computed using live midpoint price (floor)
    """

    # Normalize preds to Series
    if isinstance(preds, pd.DataFrame):
        if preds.shape[1] != 1:
            raise ValueError("If preds is a DataFrame, it must have exactly 1 column.")
        preds = preds.iloc[:, 0]

    if not isinstance(preds.index, pd.MultiIndex) or preds.index.names != ["ticker", "date"]:
        raise ValueError("preds must have MultiIndex with names ['ticker','date'].")

    # Choose date
    if date is None:
        date = preds.index.get_level_values("date").max()
    date = pd.to_datetime(date)

    # Slice predictions
    date = pd.to_datetime(date).normalize()

    date_level = pd.to_datetime(preds.index.get_level_values("date"), errors="coerce").normalize()
    mask = date_level == date

    p = preds[mask]
    p = p.droplevel("date")
    p = p.dropna()

    if p.empty:
        raise ValueError(f"No predictions found for date={date}.")

    # Select longs + shorts
    longs = p.sort_values(ascending=False).head(int(n_longs)) if n_longs > 0 else p.iloc[0:0]
    shorts = p.sort_values(ascending=True).head(int(n_shorts)) if n_shorts > 0 else p.iloc[0:0]

    # (optional) prevent overlap if n_longs + n_shorts > universe
    overlap = set(longs.index).intersection(set(shorts.index))
    if overlap:
        shorts = shorts.drop(list(overlap), errors="ignore")

    long_notional_total = float(gross_capital) * float(long_frac)
    short_notional_total = float(gross_capital) * (1.0 - float(long_frac))

    long_notional_per = long_notional_total / max(len(longs), 1)
    short_notional_per = short_notional_total / max(len(shorts), 1)

    rows = []

    def _adv_for(ticker: str) -> float:
        if adv_probs is None:
            return float(min_adv_prob_default)
        if (not isinstance(adv_probs.index, pd.MultiIndex)) or adv_probs.index.names != ["ticker", "date"]:
            raise ValueError("adv_probs must have MultiIndex ['ticker','date']")
        ap = adv_probs.xs(date, level="date").get(ticker, np.nan)
        ap = float(ap) if pd.notna(ap) else float(min_adv_prob_default)
        return float(np.clip(ap, 0.0, 1.0))

    # Build long targets
    for t in longs.index.astype(str):
        sym = str(t).upper()
        px = float(get_live_mid_price(sym))
        if px < min_price:
            continue
        qty = int(np.floor(long_notional_per / px))
        if qty > 0:
            rows.append({"symbol": sym, "shares": qty, "adv_prob": _adv_for(t)})

    # Build short targets (negative shares)
    for t in shorts.index.astype(str):
        sym = str(t).upper()
        px = float(get_live_mid_price(sym))
        if px < min_price:
            continue
        qty = int(np.floor(short_notional_per / px))
        if qty > 0:
            rows.append({"symbol": sym, "shares": -qty, "adv_prob": _adv_for(t)})

    df_targets = pd.DataFrame(rows)
    if df_targets.empty:
        raise ValueError("No targets produced (maybe prices too high vs capital, or all qty floored to 0).")

    return df_targets, date


def rebalance_with_plain_markets(
    trading_client,
    df_targets: pd.DataFrame,
    *,
    close_others: bool = True,
):
    """
    Rebalance current positions to df_targets using plain MARKET orders only.

    df_targets columns: symbol, shares (signed; shorts negative)
    """
    required = {"symbol", "shares"}
    if not required.issubset(df_targets.columns):
        raise ValueError("df_targets must include columns: 'symbol', 'shares'")

    current_positions = get_current_positions_dict(trading_client)  # shorts negative :contentReference[oaicite:3]{index=3}
    target_positions = {str(r["symbol"]).upper(): int(r["shares"]) for _, r in df_targets.iterrows()}

    # Trade to targets
    for sym, target in target_positions.items():
        current = int(current_positions.get(sym, 0))
        delta = int(target - current)

        if delta == 0:
            continue

        side = OrderSide.BUY if delta > 0 else OrderSide.SELL  # :contentReference[oaicite:4]{index=4}
        qty = abs(delta)

        submit_plain_market(trading_client, sym, qty, side)  # MARKET order :contentReference[oaicite:5]{index=5}

    # Optionally close anything not in targets
    if close_others:
        for sym, current in current_positions.items():
            sym = str(sym).upper()
            if sym in target_positions:
                continue
            if current == 0:
                continue
            side = OrderSide.SELL if current > 0 else OrderSide.BUY
            qty = abs(int(current))
            submit_plain_market(trading_client, sym, qty, side)  # :contentReference[oaicite:6]{index=6}


from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockLatestQuoteRequest


def get_live_mid_price(
    symbol: str,

) -> float:
    """
    Fetch the latest bid/ask quote for a stock and return the mid price.

    Parameters
    ----------
    symbol : str
        Stock ticker symbol (e.g. "AAPL")
    api_key : str
        Alpaca API key
    secret_key : str
        Alpaca secret key

    Returns
    -------
    float
        Midpoint price = (bid + ask) / 2

    Raises
    ------
    ValueError
        If bid or ask is missing or invalid
    """

    global api_key,secret_key

    api_key=API_KEY
    secret_key=SECRET

    client = StockHistoricalDataClient(
        api_key=api_key,
        secret_key=secret_key,
    )

    request = StockLatestQuoteRequest(
        symbol_or_symbols=[symbol]
    )

    quotes = client.get_stock_latest_quote(request)

    if symbol not in quotes:
        raise ValueError(f"No quote returned for symbol '{symbol}'")

    q = quotes[symbol]

    bid = q.bid_price
    ask = q.ask_price

    if bid is None or ask is None:
        raise ValueError(f"Missing bid/ask for symbol '{symbol}'")

    return (bid + ask) / 2.0


In [ ]:
from alpaca.trading.client import TradingClient

API_KEY='PKSVGFNVKVBT4F3YAUIPR4JPGA'
SECRET='BGL2py3myxqDesh8NiTKKFdYAPoH4Kd8g549aZTynMrL'

trading_client = TradingClient(API_KEY, SECRET, paper=True)

df_targets, trade_date = build_long_short_targets_from_preds(
    preds=preds_df["pred_rank"],   # or the Series itself
    n_longs=11,
    n_shorts=11,
    gross_capital=100_000,
    long_frac=0.5,            # 50k long, 50k short
)

rebalance_with_plain_markets(trading_client, df_targets, close_others=True)

In [ ]:
lvl = preds_df.index.get_level_values("date")
print(lvl.dtype)
print(type(lvl[0]), lvl[0])
print("example unique:", list(pd.Series(lvl).unique())[:5])

In [ ]:
new_feats=['atr_delta_fund_lag4',
 'bb_lower',
 'bb_position_delta_fund_lag20',
 'bb_upper',
 'close_lag1',
 'close_lag2',
 'close_lag20',
 'close_lag4',
 'close_lag40',
 'close_lag8',
 'daily_range_lag1',
 'dist_ma26_delta_fund',
 'high_lag1',
 'high_lag2',
 'high_lag20',
 'high_lag4',
 'high_lag40',
 'high_lag8',
 'log_ret',
 'low_lag1',
 'low_lag2',
 'low_lag20',
 'low_lag4',
 'low_lag40',
 'low_lag8',
 'ma16_lag1',
 'ma16_lag2',
 'ma16_lag20',
 'ma16_lag4',
 'ma16_lag40',
 'ma16_lag8',
 'ma26_lag1',
 'ma26_lag2',
 'ma26_lag20',
 'ma26_lag4',
 'ma26_lag40',
 'ma26_lag8',
 'ma4_lag1',
 'ma4_lag2',
 'ma4_lag20',
 'ma4_lag4',
 'ma4_lag40',
 'ma4_lag8',
 'ma8_lag1',
 'ma8_lag2',
 'ma8_lag20',
 'ma8_lag4',
 'ma8_lag40',
 'ma8_lag8',
 'macd_delta_fund',
 'macd_x_return_lag2',
 'momentum_4w_delta_fund_lag20',
 'momentum_8w_delta_fund',
 'momentum_8w_delta_fund_lag1',
 'open_close_ret_delta_fund',
 'open_close_ret_delta_fund_lag8',
 'open_lag1',
 'open_lag2',
 'open_lag20',
 'open_lag4',
 'open_lag40',
 'open_lag8',
 'pct_change_1w',
 'rsi_delta_fund_lag4',
 'rsi_lag40',
 'rsi_overbought_lag40',
 'rsi_oversold_lag8',
 'volatility_4w_delta_fund_lag20',
 'volatility_4w_lag40',
 'volume_lag1',
 'volume_lag20',
 'volume_lag8',
 'weekly_return_delta_fund',
 'weekly_return_delta_fund_lag8']

"""Refactor: Optimize stop-loss / take-profit params to maximize CV Spearman.

What this file does
-------------------
1) Downloads & prepares a *single* feature frame X_base from OHLCV (derived once).
2) For each Optuna trial, re-computes y (trade-return target) from the same base data
   using the new `calculate_flexible_profit`-style engine, then ranks it cross-sectionally.
3) Runs the provided CV split, produces out-of-fold predictions, and scores Spearman
   correlation between preds and y.

How to use
----------
- Ensure the helper functions below are imported/available:
    - generate_stock_data
    - calculate_flexible_profit

- Plug in your own CV object (e.g., a PurgedKFold) by passing `cv`.

Notes
-----
- X is kept fixed across trials; y is recomputed per trial.
- We score Spearman of OOF predictions vs y.
- If y degenerates (too many NaNs / constant), we return a very low score.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Iterable, Optional, Tuple

import numpy as np
import pandas as pd

import optuna
from lightgbm import LGBMRegressor
from scipy.stats import spearmanr

from sklearn.base import clone


# ---------------------------------------------------------------------------
# 0) Configuration objects
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class DataConfig:
    tickers: list[str]
    start_date: str
    end_date: str
    freq: str = "1d"
    horizon: int = 1
    rank_ascending: bool = False
    auto_adjust: bool = False
    threads: bool = True


@dataclass(frozen=True)
class ModelConfig:
    # Adjust to match your existing modeling setup
    lgbm_params: Dict[str, Any] = None

    def make_model(self) -> LGBMRegressor:
        params = dict(self.lgbm_params or {})
        # Keep default quiet
        params.setdefault("verbose", -1)
        return LGBMRegressor(**params)


# ---------------------------------------------------------------------------
# 1) Build X once (and keep the base OHLCV for y recomputation)
# ---------------------------------------------------------------------------

def build_X_once(
    cfg: DataConfig,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Download data once and create X_base.

    Returns
    -------
    data_base : pd.DataFrame
        Base OHLCV data indexed by (ticker, date). Used for y recomputation.
    X_base : pd.DataFrame
        Feature matrix indexed by (ticker, date). Derived once.

    IMPORTANT:
      - Replace the feature engineering block with your own features.
      - For now, we provide a conservative, leakage-free baseline feature set.
    """
    # Requires: generate_stock_data(tickers, start_date, end_date, freq, ...)
    data_base = generate_stock_data(
        tickers=cfg.tickers,
        start_date=cfg.start_date,
        end_date=cfg.end_date,
        freq=cfg.freq,
        auto_adjust=cfg.auto_adjust,
        threads=cfg.threads,
    ).copy()

    # --- Feature engineering (derived ONCE) ---
    # Keep it leakage-free by using only present/past information.
    # You can replace/extend this section.
    X = data_base.copy()

    # Example: simple returns / ranges (all can be computed from current bar)
    X["hl_range"] = (X["high"] / X["low"] - 1.0).replace([np.inf, -np.inf], np.nan)
    X["oc_ret"] = (X["close"] / X["open"] - 1.0).replace([np.inf, -np.inf], np.nan)

    # Example: trailing 5-bar momentum using past closes only
    # (groupby ticker, shift to avoid using current close to predict itself if needed)
    X["mom_5"] = (
        X.groupby(level=0)["close"].pct_change(5)
        .replace([np.inf, -np.inf], np.nan)
    )

    # Example: trailing volatility of close-to-close returns
    cc_ret = X.groupby(level=0)["close"].pct_change(1)
    X["vol_20"] = (
        cc_ret.groupby(level=0)
        .rolling(20, min_periods=20)
        .std()
        .reset_index(level=0, drop=True)
        .replace([np.inf, -np.inf], np.nan)
    )

    # Drop raw prices if you prefer (often useful to keep them for SL/TP debugging)
    # Here we keep everything; LightGBM can handle collinearity.

    # Clean: ensure numeric
    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce")

    return data_base, X


# ---------------------------------------------------------------------------
# 2) Recompute y many times (per trial)
# ---------------------------------------------------------------------------


def oof_spearman_score(
    *,
    model: Any,
    X: pd.DataFrame,
    y: pd.Series,
    cv,
) -> float:
    """Compute out-of-fold Spearman correlation.

    IMPORTANT for custom CV objects
    -------------------------------
    Some custom splitters (like your BalancedRegimePurgedEmbargoCV) require that the
    *first* call to `split` receives a pandas object whose index includes a datetime
    level named 'date', so they can cache per-row dates/groups internally.

    Therefore, we keep the CV iteration in pandas space and only convert to numpy
    for the model fit/predict.

    Parameters
    ----------
    cv : object
        Any sklearn-style splitter yielding (train_idx, test_idx) over row positions.
        If your CV expects date groups, pass a splitter that infers them from X.index.

    Returns
    -------
    float
        Spearman rho between OOF predictions and y.
    """

    # Align & drop NaNs consistently
    common_idx = X.index.intersection(y.index)
    X2 = X.loc[common_idx]
    y2 = y.loc[common_idx]

    mask = np.isfinite(y2.values)
    if mask.sum() < 100:
        return -99.0

    X2 = X2.loc[mask]
    y2 = y2.loc[mask]

    # Constant target -> meaningless Spearman
    if float(np.nanstd(y2.values)) == 0.0:
        return -99.0

    # Out-of-fold predictions
    oof = np.full(len(y2), np.nan, dtype=float)

    # Use pandas in cv.split so custom CV can read the index (date level, etc.)
    # Train/test indices are assumed to be positional (0..n-1) for X2/y2.
    for tr, te in cv.split(X2, y2):
        m = clone(model)

        X_tr = X2.iloc[tr].values
        y_tr = y2.iloc[tr].values
        X_te = X2.iloc[te].values

        m.fit(X_tr, y_tr)
        oof[te] = m.predict(X_te)

    ok = np.isfinite(oof)
    if ok.sum() < 50:
        return -99.0

    rho = spearmanr(oof[ok], y2.values[ok]).correlation
    if not np.isfinite(rho):
        return -99.0

    return float(rho)


from typing import Optional

import numpy as np
import pandas as pd
import optuna

# ---------------------------------------------------------------------------
# 2) Recompute y many times (per trial) — UPDATED
# ----------------------------------------------------------------------------

def make_objective(
    *,
    data_cfg: "DataConfig",
    model_cfg: "ModelConfig",
    data_base: pd.DataFrame,
    X_base: pd.DataFrame,
) -> callable:
    """Create an Optuna objective that keeps X fixed and recomputes y."""

    base_model = model_cfg.make_model()

    def objective(trial: optuna.trial.Trial) -> float:

        # Optional: optimize entry price source

        stop_pct = trial.suggest_float("stop_pct", 0.001, 0.20, step=0.001)
        take_profit_pct = trial.suggest_float("take_profit_pct", 0.001, 0.30, step=0.001)

        # Compute y for this trial (FAST relative to rebuilding features)
        y = compute_y_for_params(
            data_base,
            horizon=data_cfg.horizon,
            rank_ascending=data_cfg.rank_ascending,
            stop_pct=stop_pct,
            take_profit_pct=take_profit_pct,
            entry_price_col='close',
            use_ohlc=True,
        )


        # Align X once to y's available rows (profit engine returns NaNs near the end)
        common_idx = X_base.index.intersection(y.index)
        X = X_base.loc[common_idx]
        y = y.loc[common_idx]
        X,y=FE(X,y)

        cv=create_cv_object(X)

        score = oof_spearman_score(model=base_model, X=X, y=y, cv=cv)
        return score

    return objective

# ---------------------------------------------------------------------------
# Optional: y-label helper for plotting / logs
# ---------------------------------------------------------------------------

def make_y_label(params: dict) -> str:
    sl_type = (params.get("stop_loss_type") or "none").lower()
    tp_type = (params.get("take_profit_type") or "none").lower()

    parts = []

    if sl_type == "fixed_pct":
        parts.append(f"SL=fixed({params['stop_pct']:.3f})")
    elif sl_type == "trailing_pct":
        parts.append(f"SL=trail({params['trail_stop_pct']:.3f})")
    else:
        parts.append("SL=none")

    sof = float(params.get("take_profit_scale_out_fraction", 0.0) or 0.0)

    if tp_type == "fixed_pct":
        if sof > 0:
            parts.append(f"TP=fixed({params['take_profit_pct']:.3f}, scaleout={sof:.1f})")
        else:
            parts.append(f"TP=fixed({params['take_profit_pct']:.3f})")
    elif tp_type == "trailing_pct":
        parts.append(
            f"TP=trail(trigger={params['take_profit_pct']:.3f}, trail={params['trail_take_profit_pct']:.3f})"
        )
    else:
        parts.append("TP=none")

    return " | ".join(parts)


def run_optuna(
    *,
    tickers: list[str],
    n_trials: int = 200,
) -> optuna.Study:
    data_cfg = DataConfig(
        tickers=tickers,
        start_date="1999-01-01",
        end_date="2025-12-01",
        freq="1wk",
        horizon=5,
        rank_ascending=False,
        auto_adjust=False,
        threads=True,
    )

    model_cfg = ModelConfig(
        lgbm_params={
            'verbose':-1
        }
    )

    data_base, X_base = build_X_once(data_cfg)

    objective = make_objective(
        data_cfg=data_cfg,
        model_cfg=model_cfg,
        data_base=data_base,
        X_base=X_base,
    )

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)

    print("Best params:", study.best_params)
    print("Best CV Spearman:", study.best_value)
    return study

#study=run_optuna(tickers=tickers, n_trials=300)

In [ ]:
tickers=[
    "ABBV",
    "ADBE",
    "AMD",
    "AMZN",
    "ANET",
    "ARM",
    "AVGO",
    "BABA",
    "BAC",
    "CRM",
    "CRWD",
    "DE",
    "DIS",
    "GEV",
    "GILD",
    "HD",
    "HDB",
    "HON",
    "IBM",
    "INTU",
    "KKR",
    "LIN",
    "LOW",
    "MA",
    "MELI",
    "META",
    "NFLX",
    "NOW",
    "NVO",
    "PANW",
    "PDD",
    "PLTR",
    "PM",
    "SCHW",
    "SHEL",
    "SPOT",
    "T",
    "TBB",
    "TD",
    "TJX",
    "TMO",
    "TMUS",
    "TTE",
    "UBER",
    "UNH",
    "UNP",
    "V",
    'GLDM',
    'SLV',
    "VZ"]

tickers=["AAPL", "MSFT", "GOOGL", "AMZN", "META",'TSLA','INTC','BTC-USD']

data_cfg = DataConfig(
        tickers=tickers,
        start_date="1999-01-01",
        end_date="2026-01-03",
        freq="1wk",
        horizon=1,
        rank_ascending=False,
        auto_adjust=False,
        threads=True,
    )

#params=  {'stop_pct': 0.188, 'take_profit_pct': 0.004}
params=  {'stop_pct': np.inf, 'take_profit_pct': np.inf}
x_train, x_test, y_train, y_test,x_base=build_xy_and_chrono_split_from_cfg(data_cfg=data_cfg,params=params,fe_fn=FE,test_ratio=0.2)

In [ ]:
x_train_preds,x_test_preds=rank_framework_predictions(x_train,y_train,model=LGBMRegressor(verbose=-1),t_step=5,x_test=x_test)

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
pf, selections,test_pnl_df=simulate_returns(rank_df['pred_rank'],price_df[['close']],n_long=4,n_short=4,
                                                      open_df=price_df[['open']],low_df=price_df[['low']],high_df=price_df[['high']],
                                                      #adv_prob_df=adv_series,
                                              #sl_stop_pct=0.4,tp_stop_pct=np.inf
                                                        )

In [ ]:
from __future__ import annotations

from typing import Optional, Union, Dict, Any, Tuple
import numpy as np
import pandas as pd


def coerce_date_level(mi: pd.MultiIndex, date_level_name: str = "date") -> pd.MultiIndex:
    lvl = mi.names.index(date_level_name)
    arrays = []
    for i, name in enumerate(mi.names):
        vals = mi.get_level_values(i)
        if i == lvl:
            vals = pd.to_datetime(vals, errors="raise").tz_localize(None)
        arrays.append(vals)
    return pd.MultiIndex.from_arrays(arrays, names=mi.names)


def build_pred_pnl_for_meta(
    *,
    x_train: pd.DataFrame,
    y_pred_train: Union[pd.Series, np.ndarray, list],
    x_base: pd.DataFrame,
    simulate_returns_fn,
    adv_probs: Optional[Union[pd.Series, pd.DataFrame]] = None,
    x_test: Optional[pd.DataFrame] = None,
    y_pred_test: Optional[Union[pd.Series, np.ndarray, list]] = None,
    price_cols: Tuple[str, str, str, str] = ("close", "open", "high", "low"),
    pred_col: str = "pred_rank",
    sim_close_col: str = "close",
    n_long: int = np.nan,
    n_short: int = np.nan,
    sl_stop_pct: float = 0.4,
    tp_stop_pct: float = np.inf,
    date_level_name: str = "date",
    coerce_indices: bool = True,
    require_x_in_x_base: bool = True,
    meta_clf=LogisticRegression(),
    **simulate_kwargs,
) -> Dict[str, Any]:
    """
    Generalizes the boilerplate and ENFORCES/COERCES MultiIndex date level for x_train/x_test/x_base
    using `coerce_date_level`.

    If `coerce_indices=True`, this will:
      - require MultiIndex with `date_level_name` in .names
      - coerce date level to tz-naive datetime for x_train/x_test/x_base
      - coerce adv_probs index (if provided) similarly
    """

    def _ensure_mi_with_date(obj, name: str):
        idx = obj.index
        if not isinstance(idx, pd.MultiIndex):
            raise TypeError(f"{name}.index must be a pandas.MultiIndex, got {type(idx)}")
        if date_level_name not in idx.names:
            raise ValueError(
                f"{name}.index must have a level named '{date_level_name}'. "
                f"Got names={idx.names}"
            )

    def _coerce_df_index(df: pd.DataFrame, name: str) -> pd.DataFrame:
        _ensure_mi_with_date(df, name)
        if not coerce_indices:
            return df
        out = df.copy()
        out.index = coerce_date_level(out.index, date_level_name=date_level_name)
        return out

    def _coerce_series_or_df_index(obj, name: str):
        if obj is None:
            return None
        if isinstance(obj, (pd.Series, pd.DataFrame)):
            _ensure_mi_with_date(obj, name)
            if coerce_indices:
                out = obj.copy()
                out.index = coerce_date_level(out.index, date_level_name=date_level_name)
                return out
            return obj
        raise TypeError(f"{name} must be a pandas Series/DataFrame or None, got {type(obj)}")

    # ---- Coerce/validate indices up front ----
    x_train = _coerce_df_index(x_train, "x_train")
    x_base = _coerce_df_index(x_base, "x_base")
    y_pred_train=_coerce_df_index(y_pred_train,'y_pred_train')


    if x_test is not None:
        x_test = _coerce_df_index(x_test, "x_test")
        y_pred_test=_coerce_df_index(y_pred_test,'y_pred_test')

    adv_probs = _coerce_series_or_df_index(adv_probs, "adv_probs")

    # ---- sanity: x_base columns ----
    missing_cols = [c for c in price_cols if c not in x_base.columns]
    if missing_cols:
        raise ValueError(f"x_base is missing required price columns: {missing_cols}")

    if sim_close_col not in x_base.columns:
        raise ValueError(f"x_base is missing sim_close_col='{sim_close_col}'")

    def _as_pred_series(preds, index: pd.Index) -> pd.Series:
        if isinstance(preds, pd.Series):
            # if preds is Series with different index type, reindex
            s = preds.loc[index]
        else:
            arr = np.asarray(preds)
            if arr.shape[0] != len(index):
                raise ValueError(f"predictions length {arr.shape[0]} != len(index) {len(index)}")
            s = pd.Series(arr, index=index)
        return s

    def _build_rank_and_price(x: pd.DataFrame, preds) -> tuple[pd.DataFrame, pd.DataFrame]:
        pred_s = _as_pred_series(preds, x.index)
        rank_df = pd.DataFrame({pred_col: pred_s}, index=x.index)

        if require_x_in_x_base:
            missing_idx = rank_df.index.difference(x_base.index)
            if len(missing_idx) > 0:
                # show a small sample to keep errors readable
                sample = missing_idx[:5]
                raise KeyError(
                    f"{len(missing_idx)} rows from x.index are missing in x_base.index. "
                    f"Sample missing keys: {list(sample)}"
                )

        price_df = x_base.loc[rank_df.index, list(price_cols)]
        return rank_df, price_df

    def _run_sim(rank_df: pd.DataFrame, price_df: pd.DataFrame):

        return simulate_returns_fn(
            rank_df[pred_col],
            price_df[[sim_close_col]],
            n_long=n_long,
            n_short=n_short,
            open_df=price_df[["open"]],
            low_df=price_df[["low"]],
            high_df=price_df[["high"]],
            adv_prob_df=adv_probs,  # optional passthrough
            sl_stop_pct=sl_stop_pct,
            tp_stop_pct=tp_stop_pct,
            **simulate_kwargs,
        )

    out: Dict[str, Any] = {}


    assert len(x_train)==len(y_pred_train), 'train preds not same length as train'
    # ---- train ----
    train_rank_df, train_price_df = _build_rank_and_price(x_train, y_pred_train)
    pf_train, selections_train, pred_pnl_train = _run_sim(train_rank_df, train_price_df)

    assert len(x_train)==len(pred_pnl_train), 'train PnL not same length as train'

    out.update(
        {
            "train_rank_df": train_rank_df,
            "train_price_df": train_price_df,
            "pf_train": pf_train,
            "selections_train": selections_train,
            "pred_pnl_train": pred_pnl_train,
        }
    )

    # ---- test (optional) ----
    if x_test is not None:
        if y_pred_test is None:
            raise ValueError("x_test was provided but y_pred_test is None.")

        assert len(x_test)==len(y_pred_test), 'test preds not same length as test'
        test_rank_df, test_price_df = _build_rank_and_price(x_test, y_pred_test)
        pf_test, selections_test, pred_pnl_test = _run_sim(test_rank_df, test_price_df)

        assert len(x_train)==len(pred_pnl_train), 'train PnL not same length as train'

        out.update(
            {
                "test_rank_df": test_rank_df,
                "test_price_df": test_price_df,
                "pf_test": pf_test,
                "selections_test": selections_test,
                "pred_pnl_test": pred_pnl_test,
            }
        )

    res = fit_meta_model(
        x_train=x_train.fillna(-999),
        y_pred_train=y_pred_train,
        pred_pnl_df_train=out["pred_pnl_train"],
        pred_pnl_df_test=out.get("pred_pnl_test"),
        meta_clf=meta_clf,
        x_test=x_test,
        y_pred_test=y_pred_test,
        n_long=n_long,
        n_short=n_short
    )

    res=res|out
    return res

In [ ]:
sim_out = build_pred_pnl_for_meta(
    x_train=x_train,
    y_pred_train=x_train_preds,
    x_test=x_test,
    y_pred_test=x_test_preds,
    x_base=x_base,
    simulate_returns_fn=simulate_returns,
    adv_probs=None,
    n_long=4,
    n_short=4,
    sl_stop_pct=None,
    meta_clf=LGBMClassifier(verbose=-1),
    tp_stop_pct=None,
)

In [ ]:
pf, selections,test_pnl_df=simulate_returns(sim_out['test_rank_df'],sim_out['test_price_df'][['close']],n_long=4,n_short=4,
                                                      open_df=sim_out['test_price_df'][['open']],
                                            low_df=sim_out['test_price_df'][['low']],
                                            high_df=sim_out['test_price_df'][['high']],
                                            meta_pred_df=sim_out['meta_proba_test'],
                                                      #adv_prob_df=adv_series,
                                              sl_stop_pct=0.0000001,tp_stop_pct=0.3
                                                        )

In [ ]:
plot_returns(pf)
pf.stats()

In [ ]:
import vectorbt as vbt
from __future__ import annotations

from typing import Optional, Union

import numpy as np
import pandas as pd
from sklearn.base import ClassifierMixin, clone

def coerce_date_level(mi: pd.MultiIndex, date_level_name="date") -> pd.MultiIndex:
    lvl = mi.names.index(date_level_name)
    arrays = []
    for i, name in enumerate(mi.names):
        vals = mi.get_level_values(i)
        if i == lvl:
            vals = pd.to_datetime(vals, errors="raise").tz_localize(None)
        arrays.append(vals)
    return pd.MultiIndex.from_arrays(arrays, names=mi.names)


def _maybe_coerce_date_index(idx: pd.Index, date_level_name: str = "date") -> pd.Index:
    if isinstance(idx, pd.MultiIndex) and (idx.names is not None) and (date_level_name in idx.names):
        return coerce_date_level(idx, date_level_name=date_level_name)
    return idx


def _normalize_pred_pnl(
    pred_pnl_df: Union[pd.Series, pd.DataFrame],
    *,
    pred_pnl_col: str,
) -> pd.Series:
    if isinstance(pred_pnl_df, pd.DataFrame):
        if pred_pnl_col in pred_pnl_df.columns:
            s = pred_pnl_df[pred_pnl_col].copy()
        elif pred_pnl_df.shape[1] == 1:
            s = pred_pnl_df.iloc[:, 0].copy()
            if s.name is None:
                s.name = pred_pnl_df.columns[0]
        else:
            raise ValueError(
                f"pred_pnl_df missing column '{pred_pnl_col}' and has multiple columns."
            )
    else:
        s = pred_pnl_df.copy()

    if s.name is None:
        s.name = pred_pnl_col
    return s


def _normalize_pred(
    y_pred: Union[pd.Series, np.ndarray, list],
    *,
    index: pd.Index,
    name: Union[str, int],
    what: str,
) -> pd.Series:
    if isinstance(y_pred, pd.Series):
        s = y_pred.copy()
        if not s.index.equals(index):
            s = s.reindex(index)
    else:
        arr = np.asarray(y_pred)
        if arr.shape[0] != len(index):
            raise ValueError(f"{what} length {arr.shape[0]} != len(index) {len(index)}")
        s = pd.Series(arr, index=index)

    s.name = name
    return s

def direction_from_top_bottom(
    y_pred: pd.Series,
    *,
    n_long: int,
    n_short: int,
    date_level_name: str = "date",
) -> pd.Series:
    # y_pred index must be MI with date level
    g = y_pred.groupby(level=date_level_name)

    # rank high->low for longs, low->high for shorts
    r_long  = g.rank(method="first", ascending=False)
    r_short = g.rank(method="first", ascending=True)

    long = r_long <= n_long
    short = r_short <= n_short

    # avoid overlap if universe is small
    overlap = long & short
    short = short & ~overlap

    direction = pd.Series(np.nan, index=y_pred.index, dtype=float)
    direction[long] = +1.0
    direction[short] = -1.0
    return direction

def traded_mask_from_ranks(
    preds: pd.Series,
    index: pd.MultiIndex,
    *,
    date_level_name="date",
    n_long=4,
    n_short=4,
    higher_is_better=True,
):
    s = preds.reindex(index)
    date = index.get_level_values(date_level_name)
    # rank within each day
    r = s.groupby(date).rank(ascending=not higher_is_better, method="first")
    n = s.groupby(date).transform("count")
    long = r <= n_long
    short = r > (n - n_short)
    return (long | short) & s.notna()

def fit_meta_model(
    x_train: pd.DataFrame,
    y_pred_train: Union[pd.Series, np.ndarray, list],
    pred_pnl_df_train: Union[pd.Series, pd.DataFrame],
    meta_clf: ClassifierMixin,
    *,
    x_test: Optional[pd.DataFrame] = None,
    y_pred_test: Optional[Union[pd.Series, np.ndarray, list]] = None,
    pred_pnl_df_test: Optional[Union[pd.Series, pd.DataFrame]] = None,
    pred_pnl_col: str = "pred_pnl",
    pred_feature_name: Union[str, int] = "oof_preds",
    label_threshold: float = 0.0,
    dropna: bool = True,
    return_test_preds: bool = True,
    date_level_name: str = "date",
    n_long=0,
    n_short=0
) -> dict:
    """
    Returns a dict with keys:
      - meta_df_train
      - y_meta_train
      - meta_clf
      - meta_df_test
      - meta_proba_test
      - meta_pred_test
      - y_meta_test
    """


    # ---- Coerce date level on indices (train/test) ----
    x_train = x_train.copy()
    x_train.index = _maybe_coerce_date_index(x_train.index, date_level_name=date_level_name)

    if x_test is not None:
        x_test = x_test.copy()
        x_test.index = _maybe_coerce_date_index(x_test.index, date_level_name=date_level_name)

    # ---- Enforce length matches for TRAIN inputs ----
    n_train = len(x_train)

    if isinstance(y_pred_train, pd.Series):
        if len(y_pred_train) != n_train:
            raise ValueError(f"y_pred_train length {len(y_pred_train)} != len(x_train) {n_train}")
        # coerce if applicable so reindexing doesn't fail due to mixed tz/str dates
        y_pred_train = y_pred_train.copy()
        y_pred_train.index = _maybe_coerce_date_index(y_pred_train.index, date_level_name=date_level_name)
    else:
        if np.asarray(y_pred_train).shape[0] != n_train:
            raise ValueError(f"y_pred_train length {np.asarray(y_pred_train).shape[0]} != len(x_train) {n_train}")

    if isinstance(pred_pnl_df_train, (pd.Series, pd.DataFrame)):
        if len(pred_pnl_df_train) != n_train:
            raise ValueError(f"pred_pnl_df_train length {len(pred_pnl_df_train)} != len(x_train) {n_train}")
        pred_pnl_df_train = pred_pnl_df_train.copy()
        pred_pnl_df_train.index = _maybe_coerce_date_index(pred_pnl_df_train.index, date_level_name=date_level_name)

    # ---- Train pred_pnl ----
    pnl_train_s = _normalize_pred_pnl(pred_pnl_df_train, pred_pnl_col=pred_pnl_col)

    # ---- Train y_pred ----
    ypred_train_s = _normalize_pred(
        y_pred_train, index=x_train.index, name=pred_feature_name, what="y_pred_train"
    )

    # ---- Build meta_df_train ----
    meta_df_train = x_train.copy()
    meta_df_train[pred_feature_name] = ypred_train_s

    mask = traded_mask_from_ranks(
    ypred_train_s, x_train.index,
    date_level_name=date_level_name,
    n_long=n_long, n_short=n_short,
    )

    pnl_train_raw = pnl_train_s.reindex(meta_df_train.index).where(mask)

    pnl_train = pnl_train_raw - pnl_train_raw.groupby(level=date_level_name).transform("mean")

    # ---- Train labels ----
    valid_trade_mask = pnl_train.notna() & (pnl_train != 0)

    meta_df_train = meta_df_train.loc[valid_trade_mask]
    pnl_train = pnl_train.loc[valid_trade_mask]

    # Now generate labels only for the trades that happened
    y_meta_train = (pnl_train > float(label_threshold)).astype("float")
    y_meta_train.name = "meta_label"


    if dropna:
        keep = y_meta_train.notna() & meta_df_train[pred_feature_name].notna()
        meta_df_train = meta_df_train.loc[keep]
        y_meta_train = y_meta_train.loc[keep].astype(int)

    # ---- Fit meta classifier ----
    clf = clone(meta_clf)
    clf.fit(meta_df_train, y_meta_train)

    # ---- Defaults for optional outputs ----
    meta_df_test = None
    meta_proba_test = None
    meta_pred_test = None
    y_meta_test = None

    # ---- Test side (optional) ----
    if x_test is not None and return_test_preds:
        if y_pred_test is None:
            raise ValueError(
                "x_test was provided but y_pred_test is None. "
                "Pass base-model predictions for x_test."
            )

        n_test = len(x_test)

        # Enforce length matches for TEST inputs
        if isinstance(y_pred_test, pd.Series):
            if len(y_pred_test) != n_test:
                raise ValueError(f"y_pred_test length {len(y_pred_test)} != len(x_test) {n_test}")
            y_pred_test = y_pred_test.copy()
            y_pred_test.index = _maybe_coerce_date_index(y_pred_test.index, date_level_name=date_level_name)
        else:
            if np.asarray(y_pred_test).shape[0] != n_test:
                raise ValueError(f"y_pred_test length {np.asarray(y_pred_test).shape[0]} != len(x_test) {n_test}")

        if pred_pnl_df_test is not None and isinstance(pred_pnl_df_test, (pd.Series, pd.DataFrame)):
            if len(pred_pnl_df_test) != n_test:
                raise ValueError(f"pred_pnl_df_test length {len(pred_pnl_df_test)} != len(x_test) {n_test}")
            pred_pnl_df_test = pred_pnl_df_test.copy()
            pred_pnl_df_test.index = _maybe_coerce_date_index(pred_pnl_df_test.index, date_level_name=date_level_name)

        ypred_test_s = _normalize_pred(
            y_pred_test, index=x_test.index, name=pred_feature_name, what="y_pred_test"
        )

        meta_df_test = x_test.copy()
        meta_df_test[pred_feature_name] = ypred_test_s

        if hasattr(clf, "predict_proba"):
            proba = clf.predict_proba(meta_df_test)[:, 1]
            meta_proba_test = pd.Series(proba, index=meta_df_test.index, name="meta_proba")

        pred = clf.predict(meta_df_test)
        meta_pred_test = pd.Series(pred, index=meta_df_test.index, name="meta_pred")

        if pred_pnl_df_test is not None:
            pnl_test_s = _normalize_pred_pnl(pred_pnl_df_test, pred_pnl_col=pred_pnl_col)

            mask = traded_mask_from_ranks(
              ypred_test_s, x_test.index,
              date_level_name=date_level_name,
              n_long=n_long, n_short=n_short,
              )

            pnl_test_raw = pnl_test_s.reindex(meta_df_test.index).where(mask)

            pnl_test = pnl_test_raw - pnl_test_raw.groupby(level=date_level_name).transform("mean")

            valid_trade_mask = pnl_test.notna() & (pnl_test != 0)

            meta_df_test = meta_df_test.loc[valid_trade_mask]
            pnl_test = pnl_test.loc[valid_trade_mask]

            y_meta_test = (pnl_test > float(label_threshold)).astype("float")

            y_meta_test.name = "meta_label"

            if dropna:
                y_meta_test = y_meta_test.dropna().astype(int)

    return {
        "meta_df_train": meta_df_train,
        "y_meta_train": y_meta_train,
        "meta_clf": clf,
        "meta_df_test": meta_df_test,
        "meta_proba_test": meta_proba_test,
        "meta_pred_test": meta_pred_test,
        "y_meta_test": y_meta_test,
    }

In [ ]:

from __future__ import annotations
import vectorbt as vbt
from dataclasses import dataclass
from typing import Optional, Tuple, Union, Callable

import numpy as np
import pandas as pd
from sklearn.base import clone

def _default_predict_fn(model, X: pd.DataFrame) -> np.ndarray:
    """
    Prefer predict_proba[:, 1] when available (binary classification),
    else fall back to predict.
    """
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        if proba.ndim == 2 and proba.shape[1] >= 2:
            return proba[:, 1]
        return proba.reshape(-1)
    return model.predict(X).reshape(-1)


def _get_date_level(mi: pd.MultiIndex, date_level: Union[int, str]) -> pd.Index:
    try:
        return mi.get_level_values(date_level)
    except Exception as e:
        raise ValueError(
            f"Could not access date level '{date_level}'. "
            f"Available levels: {mi.names}"
        ) from e

def rank_framework_predictions(
    X: pd.DataFrame,
    y: Union[pd.Series, np.ndarray],
    model,
    *,
    x_test: Optional[pd.DataFrame] = None,
    cv=None,
    t_step: int = 5,
    date_level: Union[int, str] = "date",
    predict_fn: Optional[Callable] = None,
    min_train_dates: int = 1,
) -> Union[pd.Series, Tuple[pd.Series, pd.Series]]:
    """
    Cross-sectional ranking-friendly prediction helper for MultiIndex (ticker, date).

    Behavior
    --------
    1) If cv is provided:
       - Produce out-of-fold (OOF) predictions for X using cv splits over ROW INDICES.
       - If x_test is provided: fit on each train fold, predict x_test, and average
         fold predictions to return a single test prediction series.

    2) If cv is not provided:
       - Produce production-mimicking walk-forward predictions on X:
         refit every `t_step` unique dates using expanding window strictly BEFORE
         the first date of the block being predicted.
       - If x_test is provided: fit once on all of X and predict x_test.

    Parameters
    ----------
    X : pd.DataFrame
        MultiIndex with levels (ticker, date) in any order; must include date_level.
    y : pd.Series or np.ndarray
        Aligned to X rows. If Series, index alignment is enforced.
    model :
        sklearn-like estimator (must support fit + predict or predict_proba).
    x_test : pd.DataFrame, optional
        Same column schema as X; MultiIndex expected (ticker, date).
    cv : optional
        Any splitter with .split(X, y) yielding (train_idx, test_idx) over row positions.
    t_step : int
        Refit cadence in UNIQUE dates when cv is None.
    date_level : int or str
        Which MultiIndex level corresponds to date.
    predict_fn : callable, optional
        Custom prediction function: (model, X_df) -> 1d ndarray.
    min_train_dates : int
        Minimum unique training dates required before making predictions in walk-forward.
        If insufficient, predictions for those early dates are NaN.

    Returns
    -------
    preds_X : pd.Series
        Predictions for X with same MultiIndex as X.
    preds_test : pd.Series (only if x_test is provided)
        Predictions for x_test with same MultiIndex as x_test.
    """
    if predict_fn is None:
        predict_fn = _default_predict_fn

    if not isinstance(X.index, pd.MultiIndex):
        raise ValueError("X must have a MultiIndex (e.g., levels: ticker, date).")

    # Align y to X
    if isinstance(y, pd.Series):
        y_aligned = y.reindex(X.index)
        if y_aligned.isna().any():
            raise ValueError("y has missing values after aligning to X.index.")
        y_arr = y_aligned.to_numpy()
    else:
        y_arr = np.asarray(y)
        if y_arr.shape[0] != X.shape[0]:
            raise ValueError("If y is an array, it must have the same number of rows as X.")

    # Ensure deterministic order for time logic (by date, then ticker within date)
    date_vals = _get_date_level(X.index, date_level)
    order = np.lexsort([X.index.get_level_values(0).to_numpy(), pd.Index(date_vals).to_numpy()])
    X_ord = X.iloc[order]
    y_ord = y_arr[order]
    idx_ord = X_ord.index
    date_ord = _get_date_level(idx_ord, date_level)

    preds_X = pd.Series(np.nan, index=idx_ord, dtype=float)

    preds_test = None
    if x_test is not None:
        if not isinstance(x_test.index, pd.MultiIndex):
            raise ValueError("x_test must have a MultiIndex if provided.")
        # keep original order for return, but we’ll predict in-place
        preds_test = pd.Series(np.nan, index=x_test.index, dtype=float)

    # -------------------------
    # CV MODE: OOF predictions
    # -------------------------
    if cv is not None:
        # OOF for X
        for tr_idx, te_idx in cv.split(X_ord, y_ord):
            m = clone(model)
            m.fit(X_ord.iloc[tr_idx], y_ord[tr_idx])
            preds_X.iloc[te_idx] = predict_fn(m, X_ord.iloc[te_idx])

        # If test provided: fold-avg test preds
        if x_test is not None:
            fold_preds = []
            for tr_idx, _ in cv.split(X_ord, y_ord):
                m = clone(model)
                m.fit(X_ord.iloc[tr_idx], y_ord[tr_idx])
                fold_preds.append(predict_fn(m, x_test))
            avg = np.mean(np.vstack(fold_preds), axis=0)
            preds_test.loc[:] = avg

        # Restore original X order
        preds_X = preds_X.reindex(X_ord.index)  # already on ordered
        preds_X = preds_X.reindex(X.index)      # back to original
        if x_test is None:
            return preds_X
        return preds_X, preds_test

    # ----------------------------------------
    # WALK-FORWARD MODE: refit every t_step days
    # ----------------------------------------
    if t_step <= 0:
        raise ValueError("t_step must be a positive integer.")

    unique_dates = pd.Index(date_ord).unique()
    # We refit at the start of each block of t_step dates
    refit_starts = list(range(0, len(unique_dates), t_step))

    for block_i, start_pos in enumerate(refit_starts):
        block_dates = unique_dates[start_pos : start_pos + t_step]
        block_start_date = block_dates[0]

        train_mask = pd.Index(date_ord) < block_start_date
        train_dates = pd.Index(date_ord[train_mask]).unique()

        if len(train_dates) < min_train_dates:
            # Not enough history: leave this block NaN
            continue

        m = clone(model)
        m.fit(X_ord.loc[train_mask], y_ord[train_mask])

        test_mask = pd.Index(date_ord).isin(block_dates)
        preds_X.loc[test_mask] = predict_fn(m, X_ord.loc[test_mask])

    # Fit on full X for x_test if provided
    if x_test is not None:
        m_full = clone(model)
        m_full.fit(X, y_arr if not isinstance(y, pd.Series) else y.reindex(X.index).to_numpy())
        preds_test.loc[:] = predict_fn(m_full, x_test)

    # Restore original X order
    preds_X = preds_X.reindex(X.index)
    if x_test is None:
        return preds_X
    return preds_X, preds_test


def _wide_to_mi(wide: pd.DataFrame, template, name: str = "pred_pnl"):
    """Convert wide(date x ticker) -> MultiIndex(ticker, date), matching input format."""
    s = wide.stack(dropna=False)
    s.index = s.index.set_names(["date", "ticker"])
    s = s.swaplevel(0, 1).sort_index()
    s.index = s.index.set_names(["ticker", "date"])
    if isinstance(template, pd.DataFrame):
        # keep 1-col DF shape if user passed a 1-col DF
        col_name = template.columns[0] if template.shape[1] == 1 else name
        return s.to_frame(col_name)
    return s.rename(name)


def simulate_returns(
    rank_df: pd.Series | pd.DataFrame,
    price_df: pd.Series | pd.DataFrame,
    n_long: int,
    n_short: int | None = None,
    higher_is_better: bool = True,
    long_gross: float = 0.5,
    short_gross: float = 0.5,
    init_cash: float = 1_000_000.0,
    fees: float = 0.0,
    slippage: float = 0.0,
    freq="1D",
    min_names_each_day: int | None = None,
    open_df: pd.Series | pd.DataFrame | None = None,
    high_df: pd.Series | pd.DataFrame | None = None,
    low_df:  pd.Series | pd.DataFrame | None = None,
    sl_stop_pct: float = np.nan,
    tp_stop_pct: float = np.nan,
    sl_trail: bool = False,
    adv_prob_df: pd.Series | pd.DataFrame | None = None,
    adv_shares: bool = False,
    adv_stop_losses: bool = False,
    adv_take_profits: bool = False,
    adv_scale: float = 1.0,
    adv_clip: tuple[float, float] = (0.0, 1.0),

    # NEW:
    allocation: str = "equal",                 # "equal" | "pred_proportional"
    pred_denominator: str = "universe",        # "universe" | "selected"
    pred_transform: str = "identity",          # "identity" | "abs" | "relu" | "softmax"
    softmax_temp: float = 1.0,                # used if pred_transform="softmax"

    # NEW (META):
    meta_pred_df: pd.Series | pd.DataFrame | None = None,   # MI(ticker,date), values in [0,1]
    meta_clip: tuple[float, float] = (0.0, 1.0),            # clamp probs
    meta_power: float = 1.0,                                # multiplier = meta ** meta_power
):
    if n_short is None:
        n_short = n_long

    def _mi_to_wide(x, name: str) -> pd.DataFrame:
        if isinstance(x, pd.DataFrame):
            if x.shape[1] == 1:
                x = x.iloc[:, 0]
            else:
                raise ValueError(f"{name}: DataFrame must be 1 column (or pass a Series).")
        if not isinstance(x, pd.Series) or not isinstance(x.index, pd.MultiIndex):
            raise ValueError(f"{name}: must be a Series (or 1-col DF) with MultiIndex (ticker, date).")
        tickers = x.index.get_level_values(0).astype(str)
        dates = pd.to_datetime(x.index.get_level_values(1))
        x2 = x.copy()
        x2.index = pd.MultiIndex.from_arrays([tickers, dates], names=["ticker", "date"])
        wide = x2.unstack("ticker")
        wide.index = pd.to_datetime(wide.index)
        return wide.sort_index()

    def _normalize_pct_to_decimal_or_inf(x: float) -> float:
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return np.inf
        x = float(x)
        if x < 0:
            raise ValueError("Stop percentages must be non-negative.")
        return x / 100.0 if x > 1.5 else x

    def _apply_pred_transform(arr: np.ndarray, kind: str) -> np.ndarray:
        if kind == "identity":
            return arr
        if kind == "abs":
            return np.abs(arr)
        if kind == "relu":
            return np.maximum(arr, 0.0)
        if kind == "softmax":
            t = float(softmax_temp)
            t = 1.0 if t <= 0 else t
            z = arr / t
            z = z - np.nanmax(z, axis=1, keepdims=True)
            e = np.exp(z)
            e[np.isnan(e)] = 0.0
            s = e.sum(axis=1, keepdims=True)
            out = np.divide(e, s, out=np.zeros_like(e), where=(s != 0))
            return out
        raise ValueError("pred_transform must be one of: identity, abs, relu, softmax")

    # --- Build CLOSE (always required) ---
    ranks_w = _mi_to_wide(rank_df, "rank_df").astype(float)
    close_w = _mi_to_wide(price_df, "price_df(close)").astype(float)

    # optional OHLC
    open_w = high_w = low_w = None
    if open_df is not None:
        open_w = _mi_to_wide(open_df, "open_df").astype(float)
    if high_df is not None:
        high_w = _mi_to_wide(high_df, "high_df").astype(float)
    if low_df is not None:
        low_w = _mi_to_wide(low_df, "low_df").astype(float)

    # Align
    common_dates = ranks_w.index.intersection(close_w.index)
    common_tickers = ranks_w.columns.intersection(close_w.columns)
    ranks_w = ranks_w.loc[common_dates, common_tickers].sort_index()
    close_w = close_w.loc[common_dates, common_tickers].sort_index()

    if open_w is not None:
        open_w = open_w.reindex(index=close_w.index, columns=close_w.columns)
    if high_w is not None:
        high_w = high_w.reindex(index=close_w.index, columns=close_w.columns)
    if low_w is not None:
        low_w = low_w.reindex(index=close_w.index, columns=close_w.columns)

    valid = ranks_w.notna() & close_w.notna() & (close_w > 0)

    enough = pd.Series(True, index=ranks_w.index)
    if min_names_each_day is not None:
        enough = valid.sum(axis=1) >= min_names_each_day

    x = ranks_w.where(valid)

    if higher_is_better:
        long_rank = x.rank(axis=1, method="first", ascending=False)
        short_rank = x.rank(axis=1, method="first", ascending=True)
    else:
        long_rank = x.rank(axis=1, method="first", ascending=True)
        short_rank = x.rank(axis=1, method="first", ascending=False)

    long_mask = (long_rank <= n_long) & enough.values[:, None]
    short_mask = (short_rank <= n_short) & enough.values[:, None]
    overlap = long_mask & short_mask
    if overlap.any().any():
        short_mask = short_mask & ~overlap

    # -------------------------
    # compute base weights (wL_mat, wS_mat)
    # -------------------------
    if allocation == "equal":
        nL = long_mask.sum(axis=1).astype(float)
        nS = short_mask.sum(axis=1).astype(float)

        wL = pd.Series(0.0, index=ranks_w.index)
        wS = pd.Series(0.0, index=ranks_w.index)
        wL.loc[nL > 0] = long_gross / nL.loc[nL > 0]
        wS.loc[nS > 0] = short_gross / nS.loc[nS > 0]

        wL_mat = pd.DataFrame(
            np.repeat(wL.values[:, None], ranks_w.shape[1], axis=1),
            index=ranks_w.index, columns=ranks_w.columns
        )
        wS_mat = pd.DataFrame(
            np.repeat(wS.values[:, None], ranks_w.shape[1], axis=1),
            index=ranks_w.index, columns=ranks_w.columns
        )

        wL_mat = wL_mat.where(long_mask, 0.0)
        wS_mat = wS_mat.where(short_mask, 0.0)

    elif allocation == "pred_proportional":
        score = x.copy()
        if not higher_is_better:
            score = -score

        score_np = _apply_pred_transform(score.to_numpy(dtype=float), pred_transform)
        score_t = pd.DataFrame(score_np, index=score.index, columns=score.columns)

        long_strength = score_t.where(score_t > 0, 0.0)
        short_strength = (-score_t).where(score_t < 0, 0.0)

        if pred_denominator == "universe":
            denom_L = long_strength.sum(axis=1).replace(0.0, np.nan)
            denom_S = short_strength.sum(axis=1).replace(0.0, np.nan)
        elif pred_denominator == "selected":
            denom_L = long_strength.where(long_mask, 0.0).sum(axis=1).replace(0.0, np.nan)
            denom_S = short_strength.where(short_mask, 0.0).sum(axis=1).replace(0.0, np.nan)
        else:
            raise ValueError("pred_denominator must be 'universe' or 'selected'")

        wL_mat = long_strength.div(denom_L, axis=0).fillna(0.0) * float(long_gross)
        wS_mat = short_strength.div(denom_S, axis=0).fillna(0.0) * float(short_gross)

        wL_mat = wL_mat.where(long_mask, 0.0)
        wS_mat = wS_mat.where(short_mask, 0.0)

        # Fallback to equal weights where denom is bad but selections exist
        nL = long_mask.sum(axis=1).astype(float)
        nS = short_mask.sum(axis=1).astype(float)
        badL = denom_L.isna() & (nL > 0)
        badS = denom_S.isna() & (nS > 0)
        if badL.any():
            wL_mat.loc[badL] = long_mask.loc[badL].div(nL.loc[badL], axis=0).fillna(0.0) * float(long_gross)
        if badS.any():
            wS_mat.loc[badS] = short_mask.loc[badS].div(nS.loc[badS], axis=0).fillna(0.0) * float(short_gross)

    else:
        raise ValueError("allocation must be 'equal' or 'pred_proportional'")

    # -------------------------
    # META: probabilistic multiplier on sizing + per-day renormalization
    # -------------------------
    if meta_pred_df is not None:
        meta_w = _mi_to_wide(meta_pred_df, "meta_pred_df").astype(float)
        meta_w = meta_w.reindex(index=close_w.index, columns=close_w.columns)

        lo, hi = meta_clip
        meta_w = meta_w.clip(lower=float(lo), upper=float(hi))

        pwr = float(meta_power)
        if pwr <= 0:
            raise ValueError("meta_power must be > 0.")
        mult = meta_w.pow(pwr)

        # If meta is missing for some names/days, default multiplier=1 (keep base sizing)
        mult = mult.fillna(1.0)

        # Apply multiplier ONLY to selected names
        wL_adj = (wL_mat * mult).where(long_mask, 0.0)
        wS_adj = (wS_mat * mult).where(short_mask, 0.0)

        # Renormalize each day to preserve gross on each side
        sumL = wL_adj.sum(axis=1)
        sumS = wS_adj.sum(axis=1)

        scaleL = pd.Series(0.0, index=wL_adj.index)
        scaleS = pd.Series(0.0, index=wS_adj.index)

        scaleL.loc[sumL > 0] = float(long_gross) / sumL.loc[sumL > 0]
        scaleS.loc[sumS > 0] = float(short_gross) / sumS.loc[sumS > 0]

        wL_mat = wL_adj.mul(scaleL, axis=0)
        wS_mat = wS_adj.mul(scaleS, axis=0)

    # Signed target weights each day (sum long ≈ long_gross, sum short ≈ -short_gross)
    target_w = wL_mat - wS_mat

    # --- Signals (kept as-is) ---
    long_prev = long_mask.shift(1, fill_value=False)
    short_prev = short_mask.shift(1, fill_value=False)

    long_entries = long_mask & ~long_prev
    long_exits   = ~long_mask & long_prev

    short_entries = short_mask & ~short_prev
    short_exits   = ~short_mask & short_prev

    # --- Base entry sizing (value) --- (kept as-is, though target_w is what you trade with below)
    size = (long_entries * wL_mat).astype(float) + (short_entries * wS_mat).astype(float)
    size = size.fillna(0.0)
    size_value = size * float(init_cash)

    # --- Adversarial factor etc... (rest unchanged) ---
    factor_w = None
    if adv_prob_df is not None and (adv_shares or adv_stop_losses or adv_take_profits):
        adv_w = _mi_to_wide(adv_prob_df, "adv_prob_df").astype(float)
        adv_w = adv_w.reindex(index=close_w.index, columns=close_w.columns)
        lo, hi = adv_clip
        adv_w = adv_w.clip(lower=lo, upper=hi)
        factor_w = 1.0 + float(adv_scale) * adv_w.fillna(0.0)
        if adv_shares:
            size_value = size_value * factor_w

    base_sl = _normalize_pct_to_decimal_or_inf(sl_stop_pct)
    base_tp = _normalize_pct_to_decimal_or_inf(tp_stop_pct)

    sl_stop_arg = base_sl
    tp_stop_arg = base_tp
    if factor_w is not None:
        if adv_stop_losses and np.isfinite(base_sl):
            sl_stop_arg = (base_sl * factor_w).astype(float)
        if adv_take_profits and np.isfinite(base_tp):
            tp_stop_arg = (base_tp * factor_w).astype(float)

    close_adj, target_w_adj, hit = apply_stops_to_target_weights(
        close_w=close_w,
        target_w=target_w,
        open_w=open_w,
        high_w=high_w,
        low_w=low_w,
        sl_stop=sl_stop_pct,   # float or DataFrame
        tp_stop=tp_stop_pct,   # float or DataFrame
        sl_trail=sl_trail,
        cooldown_requires_flat=True,
        conflict="sl_first"
          )

    pf = vbt.Portfolio.from_orders(
    close=close_adj,
    size=target_w_adj,
    size_type="targetpercent",
    init_cash=init_cash,
    fees=fees,
    slippage=slippage,
    freq=freq
    )


    # Calculate the clean forward return of the price data
    # (Price at t+1 / Price at t) - 1
    fwd_ret = close_w.shift(-1) / close_w - 1.0

    # Multiply asset returns by your target weights at time t
    # Longs (*) positive return = Profit; Shorts (*) negative return = Profit
    # This represents the PnL of holding 'target_w' for one day.
    pred_pnl_w = fwd_ret * target_w

    # Convert the wide matrix back to the MultiIndex format of the input rank_df
    pred_pnl_df = _wide_to_mi(pred_pnl_w, rank_df, name="pred_pnl")

    selections = {
        "long_mask": long_mask,
        "short_mask": short_mask,
        "long_entries": long_entries,
        "long_exits": long_exits,
        "short_entries": short_entries,
        "short_exits": short_exits,
    }
    return pf, selections, pred_pnl_df.loc[rank_df.index]


import numpy as np
import pandas as pd

def apply_stops_to_target_weights(
    close_w: pd.DataFrame,
    target_w: pd.DataFrame,
    open_w: pd.DataFrame | None = None,
    high_w: pd.DataFrame | None = None,
    low_w: pd.DataFrame | None = None,
    sl_stop: float | pd.DataFrame = np.inf,   # decimal (0.05) or DataFrame aligned to close_w
    tp_stop: float | pd.DataFrame = np.inf,   # decimal
    sl_trail: bool = False,
    cooldown_requires_flat: bool = True,
    conflict: str = "sl_first",  # "sl_first" | "tp_first" | "worst" | "best"
):
    """
    Daily-bar stop/TP overlay for target-weight rebalancing.

    Returns:
      close_adj: modified close used in from_orders (stop fill on hit day)
      target_w_adj: target weights forced to 0 after stop/TP until cooldown ends
      hit: DataFrame of 0/1 flags where a stop/TP hit occurred
    """
    if high_w is None or low_w is None:
        raise ValueError("Need high_w and low_w to detect intraday stops/TPs.")
    if open_w is None:
        open_w = close_w  # if missing, treat open as close (less realistic)

    # Align all
    idx, cols = close_w.index, close_w.columns
    open_w = open_w.reindex(idx, columns=cols)
    high_w = high_w.reindex(idx, columns=cols)
    low_w  = low_w.reindex(idx, columns=cols)
    target_w = target_w.reindex(idx, columns=cols).fillna(0.0)

    def _to_df(x):
        if isinstance(x, pd.DataFrame):
            return x.reindex(idx, columns=cols).astype(float)
        x = float(x)
        return pd.DataFrame(x, index=idx, columns=cols, dtype=float)

    sl_df = _to_df(sl_stop)
    tp_df = _to_df(tp_stop)

    close_adj = close_w.copy().astype(float)
    target_adj = target_w.copy().astype(float)
    hit = pd.DataFrame(False, index=idx, columns=cols)

    # Position held during day t is yesterday's target (close-to-close holding)
    pos_prev = target_w.shift(1).fillna(0.0)
    sign_prev = np.sign(pos_prev.to_numpy())

    close_np = close_w.to_numpy(dtype=float)
    open_np  = open_w.to_numpy(dtype=float)
    high_np  = high_w.to_numpy(dtype=float)
    low_np   = low_w.to_numpy(dtype=float)
    sl_np    = sl_df.to_numpy(dtype=float)
    tp_np    = tp_df.to_numpy(dtype=float)

    # State per ticker (column)
    nT = close_w.shape[1]
    entry = np.full(nT, np.nan)
    best_fav = np.full(nT, np.nan)   # high watermark (long) or low watermark (short)
    cooldown = np.zeros(nT, dtype=bool)

    # Helper to decide which level fills if both SL and TP hit same day
    def choose_fill(is_long, sl_level, tp_level):
        if conflict == "sl_first":
            return sl_level
        if conflict == "tp_first":
            return tp_level
        if conflict == "worst":
            return sl_level if is_long else sl_level  # SL is worst for both long/short
        if conflict == "best":
            return tp_level
        raise ValueError("conflict must be one of: sl_first, tp_first, worst, best")

    for t in range(1, len(idx)):
        pos = pos_prev.iloc[t].to_numpy(dtype=float)
        sgn = np.sign(pos)

        # Update / initialize entry prices when position turns on
        turned_on = (sgn != 0) & (np.sign(pos_prev.iloc[t-1].to_numpy(dtype=float)) == 0)
        entry[turned_on] = close_np[t-1, turned_on]  # assume entry at prior close
        if sl_trail:
            best_fav[turned_on] = close_np[t-1, turned_on]
        else:
            best_fav[turned_on] = np.nan

        # If cooldown requires going flat, clear cooldown once original target goes flat
        if cooldown_requires_flat:
            # you can also change this rule to "allow when sign flips" if you prefer
            cooldown = cooldown & (np.sign(target_w.iloc[t].to_numpy(dtype=float)) != 0)

        # Block targets during cooldown
        if cooldown.any():
            block = cooldown
            # Force desired target to 0 (starting today close)
            target_adj.iloc[t, block] = 0.0

        # Only check stops for tickers that are currently in a position (held intraday)
        active = (sgn != 0) & np.isfinite(entry)
        if not active.any():
            continue

        # Update trailing watermark
        if sl_trail:
            # long: max high; short: min low
            is_long = sgn > 0
            is_short = sgn < 0
            best_fav[is_long & active] = np.nanmax(
                np.vstack([best_fav[is_long & active], high_np[t, is_long & active]]),
                axis=0
            )
            best_fav[is_short & active] = np.nanmin(
                np.vstack([best_fav[is_short & active], low_np[t, is_short & active]]),
                axis=0
            )

        # Compute stop/TP levels
        is_long = (sgn > 0) & active
        is_short = (sgn < 0) & active

        # Long levels
        sl_level_L = np.full(nT, np.nan)
        tp_level_L = np.full(nT, np.nan)
        if np.any(is_long):
            base = best_fav if sl_trail else entry
            sl_level_L[is_long] = base[is_long] * (1.0 - sl_np[t, is_long])
            tp_level_L[is_long] = entry[is_long] * (1.0 + tp_np[t, is_long])

        # Short levels
        sl_level_S = np.full(nT, np.nan)
        tp_level_S = np.full(nT, np.nan)
        if np.any(is_short):
            base = best_fav if sl_trail else entry
            sl_level_S[is_short] = base[is_short] * (1.0 + sl_np[t, is_short])
            tp_level_S[is_short] = entry[is_short] * (1.0 - tp_np[t, is_short])

        # Detect hits using intraday range + gaps
        # Long SL if low <= sl_level; Long TP if high >= tp_level
        hit_sl_L = is_long & np.isfinite(sl_level_L) & (low_np[t] <= sl_level_L)
        hit_tp_L = is_long & np.isfinite(tp_level_L) & (high_np[t] >= tp_level_L)

        # Short SL if high >= sl_level; Short TP if low <= tp_level
        hit_sl_S = is_short & np.isfinite(sl_level_S) & (high_np[t] >= sl_level_S)
        hit_tp_S = is_short & np.isfinite(tp_level_S) & (low_np[t] <= tp_level_S)

        hit_any = hit_sl_L | hit_tp_L | hit_sl_S | hit_tp_S
        if not hit_any.any():
            continue

        # Fill price (with simple gap handling):
        fill = close_np[t].copy()

        # Long fills
        for j in np.where(hit_sl_L | hit_tp_L)[0]:
            sl_lvl = sl_level_L[j]
            tp_lvl = tp_level_L[j]
            o = open_np[t, j]
            sl_hit = hit_sl_L[j]
            tp_hit = hit_tp_L[j]

            if sl_hit and tp_hit:
                lvl = choose_fill(True, sl_lvl, tp_lvl)
            else:
                lvl = sl_lvl if sl_hit else tp_lvl

            # gap-through handling
            if sl_hit and o <= sl_lvl:
                fill[j] = o
            elif tp_hit and o >= tp_lvl:
                fill[j] = o
            else:
                fill[j] = lvl

        # Short fills
        for j in np.where(hit_sl_S | hit_tp_S)[0]:
            sl_lvl = sl_level_S[j]
            tp_lvl = tp_level_S[j]
            o = open_np[t, j]
            sl_hit = hit_sl_S[j]
            tp_hit = hit_tp_S[j]

            if sl_hit and tp_hit:
                lvl = choose_fill(False, sl_lvl, tp_lvl)
            else:
                lvl = sl_lvl if sl_hit else tp_lvl

            # gap-through handling
            if sl_hit and o >= sl_lvl:
                fill[j] = o
            elif tp_hit and o <= tp_lvl:
                fill[j] = o
            else:
                fill[j] = lvl

        # Write adjusted close for hit names
        close_adj.iloc[t, hit_any] = fill[hit_any]
        hit.iloc[t, hit_any] = True

        # Start cooldown for hit names: force weights to zero going forward
        cooldown[hit_any] = True
        target_adj.iloc[t, hit_any] = 0.0   # exit at today's close (now = stop fill)
        entry[hit_any] = np.nan
        best_fav[hit_any] = np.nan

    return close_adj, target_adj, hit

In [ ]:
predictions=cv_preds
#predictions=np.argmax(preds,axis=1)
#test_adv_probs=pd.Series(np.full(len(x_test),0),index=x_test.index)
#test_adv_probs=pd.Series(adv_res['val_probs_all'][-len(x_test):],index=x_test.index)
preds_s=rank_df=pd.DataFrame(predictions,index=x_test.index,columns=['pred_rank'])

In [ ]:
from sklearn.model_selection import cross_val_score
from scipy.stats import spearmanr
cv=create_cv_object(x_train)
model=LGBMRegressor(verbose=-1)
#cv_score=cross_val_score(model,X,y,cv=cv,error_score='raise')
cv_score=oof_spearman_score(model=model, X=x_train, y=y_train, cv=cv)
print(cv_score)

In [ ]:
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
model=LGBMRegressor(verbose=-1)
model.fit(x_train,y_train)
preds=model.predict(x_test)
#preds, cur_booster, cur_model=walk_forward_lgbm_warmstart_with_regressor_init_model(x_test,y_test,model,
 #                                                                                          params={'verbose':-1},trees_per_date=2)

In [ ]:
adv_res=adversarial_validation(x_train.fillna(-999),x_test.fillna(-999))
adv_series=pd.Series(adv_res['val_probs_all'][-len(x_test):],index=x_test.index)

In [ ]:
plot_returns(pf)
df=pf.stats()
print(df)

In [ ]:
from __future__ import annotations

from typing import Callable, Optional, Tuple, Dict, Any
import numpy as np
import pandas as pd

def build_xy_and_chrono_split_from_cfg(
    *,
    data_cfg,  # DataConfig
    params: Dict[str, Any],
    test_ratio: float = 0.2,
    fe_fn: Optional[Callable[[pd.DataFrame, pd.Series], Tuple[pd.DataFrame, pd.Series]]] = None,
    entry_price_col: str = "close",
    use_ohlc: bool = True,
    dropna_y: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """
    1) data_base, X_base = build_X_once(data_cfg)
    2) y = compute_y_for_params(data_base, fixed params)
    3) align X/y, optional FE, sort_index(level='date')
    4) chronological split with test_ratio by unique dates

    Returns: X_train, X_test, y_train, y_test
    """
    if not (0.0 < test_ratio < 1.0):
        raise ValueError(f"test_ratio must be in (0, 1); got {test_ratio}")

    # --- 1) Build base OHLCV + features once ---
    data_base, X_base = build_X_once(data_cfg)

    if not isinstance(X_base.index, pd.MultiIndex) or "date" not in X_base.index.names:
        raise ValueError("X_base must have a MultiIndex with a level named 'date'.")
    if not isinstance(data_base.index, pd.MultiIndex) or "date" not in data_base.index.names:
        raise ValueError("data_base must have a MultiIndex with a level named 'date'.")

    # --- 2) Build y from your fixed params ---
    y = compute_y_for_params(
        data_base,
        horizon=data_cfg.horizon,
        rank_ascending=data_cfg.rank_ascending,
        stop_pct=float(params["stop_pct"]),
        take_profit_pct=float(params["take_profit_pct"]),
        entry_price_col=entry_price_col,
        use_ohlc=use_ohlc,
    )

    # --- 3) Align X and y on common index ---
    common_idx = X_base.index.intersection(y.index)
    X = X_base.loc[common_idx]
    y = y.loc[common_idx]

    # Optional FE hook (your existing FE(X, y))
    if fe_fn is not None:
        X, y = fe_fn(X, y)

    # --- 4) Drop NaNs in y (profit engine often returns NaNs near the end) ---
    if dropna_y:
        m = np.isfinite(y.to_numpy(dtype=float))
        X = X.loc[m]
        y = y.loc[m]

    #y = y_to_ternary(y, q=0.4, by="date")

    # --- 5) Required: sort chronologically by date level ---
    X = X.sort_index(level="date")
    y = y.sort_index(level="date")

    # --- 6) Chronological split by unique dates ---
    dates = X.index.get_level_values("date")
    unique_dates = pd.Index(dates.unique()).sort_values()

    n_test_dates = int(np.ceil(len(unique_dates) * test_ratio))
    if n_test_dates < 1:
        raise ValueError("Not enough unique dates to create a test split.")
    if n_test_dates >= len(unique_dates):
        raise ValueError("test_ratio too large for the available number of unique dates.")

    split_date = unique_dates[-n_test_dates]  # first date in test set
    is_test = dates >= split_date

    X_train, X_test = X.loc[~is_test], X.loc[is_test]
    y_train, y_test = y.loc[~is_test], y.loc[is_test]

    return X_train, X_test, y_train, y_test,X_base

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Callable, Dict, Iterable, List, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

import tqdm

def _ensure_datetime(x: Union[str, pd.Timestamp]) -> pd.Timestamp:
    ts = pd.Timestamp(x)
    if ts.tzinfo is not None:
        ts = ts.tz_convert(None)
    return ts


def _weekly_dates(
    start: Union[str, pd.Timestamp],
    end: Union[str, pd.Timestamp],
    anchor: str = "FRI",
    include_start: bool = True,
) -> List[pd.Timestamp]:
    """
    Generate weekly dates between start and end (inclusive-ish).
    anchor: 'MON'...'SUN' (uses pandas weekly anchored frequency, e.g. 'W-FRI').
    """
    start_ts = _ensure_datetime(start)
    end_ts = _ensure_datetime(end)
    freq = f"W-{anchor.upper()}"
    dates = list(pd.date_range(start=start_ts, end=end_ts, freq=freq))

    # If start itself isn't on the anchor, and include_start=True, prepend it.
    if include_start and (len(dates) == 0 or dates[0].normalize() != start_ts.normalize()):
        dates = [start_ts] + dates

    # Ensure strictly increasing unique
    out = []
    seen = set()
    for d in dates:
        d = _ensure_datetime(d)
        if d not in seen and d <= end_ts:
            out.append(d)
            seen.add(d)
    out.sort()
    return out


def _numeric_feature_cols(df: pd.DataFrame) -> List[str]:
    return [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]


@dataclass
class RollingIFUniverseResult:
    weekly_df: pd.DataFrame                   # concatenated rows across dates (selected subset each week)
    selected_by_date: Dict[pd.Timestamp, List[str]]  # date -> selected tickers
    scores_by_date: Dict[pd.Timestamp, pd.Series]    # date -> score_samples for pool (indexed by ticker)


# --------- Main function ---------

def rolling_isoforest_weekly_universe(
    *,
    # Universe + date controls
    tickers: Sequence[str],
    initial_date: Union[str, pd.Timestamp],
    end_date: Union[str, pd.Timestamp],
    lookback_days: int,
    max_tickers: int,
    week_anchor: str = "FRI",

    # Step 1: initial anchor selection (your existing function)
    initial_selector: Callable[..., pd.DataFrame],
    initial_selector_kwargs: Optional[dict] = None,

    # Step 2: pool builder for "all available tickers" at each subsequent date
    # Must return a DataFrame with at least: ['date','ticker', <numeric feature cols...>]
    # One row per ticker for that date (or fewer if unavailable).
    pool_builder: Callable[..., pd.DataFrame] = None,
    pool_builder_kwargs: Optional[dict] = None,

    # IsolationForest knobs
    if_params: Optional[dict] = None,

    # Missing-value handling
    min_non_nan_frac_pool: float = 0.8,   # drop pool rows with too many NaNs in numeric features
    min_non_nan_frac_train: float = 0.8,  # drop training rows with too many NaNs in numeric features
) -> RollingIFUniverseResult:
    """
    Workflow (weekly):
      1) Build initial weekly_df using `initial_selector(...)` for `initial_date`
      2) For each next weekly date:
         a) Build pool (all tickers) for that date via `pool_builder(...)`
         b) Fit IsolationForest on ALL selected data up to now (history weekly_df)
         c) Score the pool with score_samples
         d) Select top `max_tickers` tickers by score (highest = most "similar"/normal under the fitted IF)
         e) Append those selected pool rows into weekly_df
      3) Return the concatenated weekly_df and selections.

    Requirements:
      - `initial_selector` returns a DataFrame with at least ['date','ticker', <numeric feature cols...>]
      - `pool_builder` returns a DataFrame with at least ['date','ticker', <numeric feature cols...>]
      - Feature columns should match across initial + pool rows (extra columns are allowed; intersection is used).
    """
    if initial_selector_kwargs is None:
        initial_selector_kwargs = {}
    if pool_builder_kwargs is None:
        pool_builder_kwargs = {}
    if if_params is None:
        if_params = {}

    if pool_builder is None:
        raise ValueError("You must pass a `pool_builder` function that builds the all-ticker feature pool per date.")

    initial_date_ts = _ensure_datetime(initial_date)
    end_date_ts = _ensure_datetime(end_date)

    if end_date_ts < initial_date_ts:
        raise ValueError("end_date must be >= initial_date.")

    if max_tickers <= 0:
        raise ValueError("max_tickers must be positive.")
    if max_tickers > len(tickers):
        max_tickers = len(tickers)

    # ---- 1) Initial anchor selection (your KNN function) ----
    init_df = initial_selector(
        tickers=list(tickers),
        initial_date=str(initial_date_ts.date()),
        lookback_days=lookback_days,
        max_tickers=max_tickers,
        **initial_selector_kwargs,
    ).copy()

    if not {"date", "ticker"}.issubset(init_df.columns):
        raise ValueError("initial_selector must return columns: at least ['date','ticker', ...features...]")

    init_df["date"] = pd.to_datetime(init_df["date"])
    init_df["ticker"] = init_df["ticker"].astype(str)

    weekly_df = init_df.sort_values(["date", "ticker"]).reset_index(drop=True)

    selected_by_date: Dict[pd.Timestamp, List[str]] = {
        initial_date_ts: sorted(weekly_df.loc[weekly_df["date"] == weekly_df["date"].min(), "ticker"].unique().tolist())
    }
    scores_by_date: Dict[pd.Timestamp, pd.Series] = {}

    # Generate weekly dates (including initial_date, but we’ll iterate from the next one)
    dates = _weekly_dates(initial_date_ts, end_date_ts, anchor=week_anchor, include_start=True)

    # If initial_date isn't exactly the first date in `weekly_df`, we still treat it as the anchor.
    # We'll iterate over dates strictly after initial_date_ts:
    future_dates = [d for d in dates if d > initial_date_ts]

    # Default IF parameters (overrideable)
    default_if = dict(
        n_estimators=500,
        contamination="auto",
        random_state=42,
        n_jobs=-1,
    )
    default_if.update(if_params)

    # ---- 2) Rolling weekly loop ----
    for dt in tqdm.tqdm(future_dates):
        # a) Pool for next date (all tickers)
        pool_df = pool_builder(
            tickers=list(tickers),
            asof_date=str(dt.date()),
            lookback_days=lookback_days,
            **pool_builder_kwargs,
        ).copy()

        if pool_df.empty:
            # Nothing available this week; skip
            continue

        if not {"date", "ticker"}.issubset(pool_df.columns):
            raise ValueError("pool_builder must return columns: at least ['date','ticker', ...features...]")

        pool_df["date"] = pd.to_datetime(pool_df["date"])
        pool_df["ticker"] = pool_df["ticker"].astype(str)

        # Keep only rows for this date (pool_builder might return multiple)
        pool_df = pool_df.loc[pool_df["date"] == pool_df["date"].max()].copy()
        if pool_df.empty:
            continue

        # b) Determine shared numeric feature columns
        train_cols = _numeric_feature_cols(weekly_df)
        pool_cols = _numeric_feature_cols(pool_df)
        feat_cols = [c for c in train_cols if c in pool_cols]

        if len(feat_cols) == 0:
            raise ValueError("No shared numeric feature columns between weekly_df and pool_df.")

        # c) Drop rows with too many NaNs
        def _filter_by_non_nan_frac(df: pd.DataFrame, cols: List[str], min_frac: float) -> pd.DataFrame:
            frac = df[cols].notna().mean(axis=1)
            return df.loc[frac >= min_frac].copy()

        # Fit only on the most recent selected set X_t (last stamped date in weekly_df)
        last_train_date = weekly_df["date"].max()
        train_df_latest = weekly_df.loc[weekly_df["date"] == last_train_date].copy()

        train_df = _filter_by_non_nan_frac(train_df_latest, feat_cols, min_non_nan_frac_train)

        pool_df_f = _filter_by_non_nan_frac(pool_df, feat_cols, min_non_nan_frac_pool)

        if train_df.empty or pool_df_f.empty:
            continue

        # d) Impute using training medians (keeps scoring consistent)
        med = train_df[feat_cols].median(numeric_only=True)
        X_train = train_df[feat_cols].fillna(med).to_numpy(dtype=float)

        X_pool = pool_df_f[feat_cols].fillna(med).to_numpy(dtype=float)

        # e) Fit IsolationForest on all history up to now
        iso = IsolationForest(**default_if)
        iso.fit(X_train)

        # f) Score the pool and select top max_tickers by score (higher = more "normal"/similar)
        scores = iso.score_samples(X_pool)
        score_s = pd.Series(scores, index=pool_df_f["ticker"].values, name="if_score").sort_values(ascending=False)

        scores_by_date[dt] = score_s

        chosen = score_s.head(max_tickers).index.tolist()
        selected_by_date[dt] = chosen

        chosen_rows = pool_df_f.loc[pool_df_f["ticker"].isin(chosen)].copy()
        chosen_rows["date"] = dt  # force the loop date as the stamped date

        # Append to weekly_df
        weekly_df = (
            pd.concat([weekly_df, chosen_rows], axis=0, ignore_index=True)
              .sort_values(["date", "ticker"])
              .reset_index(drop=True)
        )

    return RollingIFUniverseResult(
        weekly_df=weekly_df,
        selected_by_date=selected_by_date,
        scores_by_date=scores_by_date,
    )

In [ ]:
# ---- Example usage ----
# Assumes you already have:
#   1) initial_date_knn_maximin_universe(...)
#   2) SOME function that can build the full feature pool for a single date across tickers
#      (see `pool_builder` below)

initial_date = "2023-06-30"
end_date     = "2023-10-27"   # pick any weekly-end you want
lookback_days = 90
max_tickers = len(tickers) - 10

import pandas as pd
import numpy as np
from typing import Sequence, Dict, Any, Optional

def pool_builder(
    *,
    tickers: Sequence[str],
    asof_date: str | pd.Timestamp,
    lookback_days: int,
    feature_kwargs: Optional[Dict[str, Any]] = None,
    interval: str = "1d",
    auto_adjust: bool = False,
    threads: bool | int = True,
    include_end: bool = True,
    # If asof_date isn't a trading day (or data is missing), use last available <= asof_date
    allow_prior_date_fallback: bool = True,
) -> pd.DataFrame:
    """
    Build the "all ticker pool" feature table for a single weekly step.

    Returns: DataFrame with one row per ticker:
      columns include: ['date','ticker', <numeric engineered feature cols...>]

    Uses your existing:
      - fetch_ohlcv_window(...)
      - engineer_technical_features(...)
    """
    feature_kwargs = feature_kwargs or {}
    asof = pd.to_datetime(asof_date).normalize()

    # 1) Fetch windowed OHLCV up to asof_date (inclusive)
    ohlcv = fetch_ohlcv_window(
        tickers=list(tickers),
        end_date=asof,
        lookback_days=lookback_days,
        interval=interval,
        auto_adjust=auto_adjust,
        threads=threads,
        include_end=include_end,
    )
    if ohlcv.empty:
        return pd.DataFrame(columns=["date", "ticker"])

    # 2) Engineer features on the full window
    X_full = engineer_technical_features(ohlcv, **feature_kwargs)

    # 3) Extract one row per ticker "as of" the requested date
    # Prefer exact asof_date; if missing, optionally fall back to last available <= asof_date per ticker.
    dates = X_full.index.get_level_values("date")
    if asof in dates:
        X_date = X_full.xs(asof, level="date").copy()  # index=ticker
        used_date = asof
    else:
        if not allow_prior_date_fallback:
            return pd.DataFrame(columns=["date", "ticker"])

        # For each ticker, take its last available row with date <= asof
        xf = X_full.reset_index()  # columns: ticker,date,...
        xf = xf.loc[xf["date"] <= asof]
        if xf.empty:
            return pd.DataFrame(columns=["date", "ticker"])

        # last per ticker
        xf = xf.sort_values(["ticker", "date"])
        xf = xf.groupby("ticker", as_index=False).tail(1)
        used_date = asof  # we *stamp* the pool row with the loop date downstream
        X_date = xf.set_index("ticker").drop(columns=["date"])

    # 4) Final shape: one row per ticker, with ['date','ticker', ...features]
    out = X_date.apply(pd.to_numeric, errors="coerce").copy()
    out.insert(0, "ticker", out.index.astype(str))
    out.insert(0, "date", used_date)
    out = out.reset_index(drop=True)

    return out


# --- 2) Run rolling selection ---
res = rolling_isoforest_weekly_universe(
    tickers=tickers,
    initial_date=initial_date,
    end_date=end_date,
    lookback_days=lookback_days,
    max_tickers=max_tickers,
    week_anchor="FRI",

    initial_selector=initial_date_knn_maximin_universe,
    initial_selector_kwargs=dict(
        # KNN / universe-selection controls
        n_neighbors=8,
        seed="medoid",
        min_non_nan_frac=0.8,

        # Passed through to engineer_technical_features(...)
        feature_kwargs=dict(
            ma_windows=(5, 10, 20, 30),
            ema_windows=(12, 26),
            rsi_window=14,
            atr_window=14,
            bb_window=20,
            bb_n_std=2.0,
            roc_windows=(5, 10, 20),
            vol_windows=(10, 20, 30),
            lag_days=(1, 2, 5),
            drop_original_ohlcv=False,
            aroon_window=25,
            donchian_window=20,
            keltner_window=20,
            keltner_atr_mult=2.0,
            willr_window=14,
            stoch_window=14,
            stoch_smooth_k=3,
            adx_window=14,
            cci_window=20,
            uo_windows=(7, 14, 28),
            trix_window=15,
            ppo_fast=12,
            ppo_slow=26,
            ppo_signal=9,
            roll_mom_windows=(10, 20, 60),
            roll_shape_windows=(20, 60),
            vwap_window=20,
            cmf_window=20,
        ),
    ),

    pool_builder=pool_builder,
    pool_builder_kwargs=dict(
        feature_kwargs=dict(
            # IMPORTANT: keep this the same feature config as above
            ma_windows=(5, 10, 20, 30),
            ema_windows=(12, 26),
            rsi_window=14,
            atr_window=14,
            bb_window=20,
            bb_n_std=2.0,
            roc_windows=(5, 10, 20),
            vol_windows=(10, 20, 30),
            lag_days=(1, 2, 5),
            drop_original_ohlcv=False,
            aroon_window=25,
            donchian_window=20,
            keltner_window=20,
            keltner_atr_mult=2.0,
            willr_window=14,
            stoch_window=14,
            stoch_smooth_k=3,
            adx_window=14,
            cci_window=20,
            uo_windows=(7, 14, 28),
            trix_window=15,
            ppo_fast=12,
            ppo_slow=26,
            ppo_signal=9,
            roll_mom_windows=(10, 20, 60),
            roll_shape_windows=(20, 60),
            vwap_window=20,
            cmf_window=20,
        )
    ),

    if_params=dict(
        n_estimators=500,
        contamination="auto",
        random_state=42,
        n_jobs=-1,
    ),
)

# --- 3) Inspect outputs ---
weekly_df = res.weekly_df
print("Total selected rows:", len(weekly_df))
print("Unique dates:", weekly_df["date"].nunique())
print("Latest date:", weekly_df["date"].max())

# Which tickers were selected each week:
for d, chosen in res.selected_by_date.items():
    print(d.date(), len(chosen), chosen[:10], "..." if len(chosen) > 10 else "")

# The actual per-ticker isolation-forest scores for a specific week:
some_week = sorted(res.scores_by_date.keys())[0]
print("\nTop scores on", some_week.date())
print(res.scores_by_date[some_week].head(10))

In [ ]:
#weekly_df[['date','ticker']].to_csv('value_pairs.csv',index=False)

In [ ]:
value_pairs=pd.read_csv('value_pairs.csv')

In [ ]:
# --- original parameters ---
initial_date = "1999-01-01"
end_date = "2026-01-01"
lookback_weeks = 26
lookback_days_buffer = 7 * lookback_weeks  # IMPORTANT
max_tickers = 60

feature_kwargs = dict(
    ma_windows=(4, 8, 13, 26),
    ema_windows=(12, 26),
    rsi_window=14,
    atr_window=14,
    bb_window=20,
    bb_n_std=2.0,
    roc_windows=(4, 8, 13),
    vol_windows=(8, 13, 26),
    lag_days=(7, 14, 28),          # lag_weeks → lag_days
    roll_mom_windows=(8, 13, 26),
    roll_shape_windows=(13, 26),
    vwap_window=20,
    cmf_window=20,
    drop_original_ohlcv=False,
    include_slow_features=False,
)

rederived_df = rederive_dataset_from_pairs(
    pairs_df=value_pairs,
    lookback_days_buffer=lookback_days_buffer,
    feature_kwargs=feature_kwargs,
    interval="1wk",            # <-- change this
    auto_adjust=False,
    threads=True,
)

In [ ]:
import pandas as pd
from typing import Sequence, Union, Optional, Dict, Any

# This function assumes the same data/feature-generation logic
# as in your original pipeline (fetch_ohlcv_range + engineer_technical_features_fast).
# Given ONLY (ticker, date) pairs, it reconstructs the full dataset deterministically.

def rederive_dataset_from_pairs(
    pairs_df: pd.DataFrame,
    *,
    lookback_days_buffer: int,
    feature_kwargs: Optional[Dict[str, Any]] = None,
    interval: str = "1d",
    auto_adjust: bool = False,
    threads: bool | int = True,
) -> pd.DataFrame:
    """
    Re-derive the full engineered dataset from saved (ticker, date) pairs.

    Parameters
    ----------
    pairs_df : pd.DataFrame
        Must contain columns ['ticker', 'date'].
        These define the *final* rows you care about.
    lookback_days_buffer : int
        Number of calendar days of lookback required to recompute features
        (must be >= the max rolling window used originally).
    feature_kwargs : dict, optional
        Passed directly to engineer_technical_features_fast.
    interval, auto_adjust, threads :
        Passed through to fetch_ohlcv_range.

    Returns
    -------
    pd.DataFrame
        Fully re-derived dataset, identical (modulo data vendor revisions)
        to what the original pipeline would have produced for those rows.
    """

    required = {"ticker", "date"}
    if not required.issubset(pairs_df.columns):
        raise ValueError("pairs_df must contain columns ['ticker', 'date']")

    feature_kwargs = feature_kwargs or {}

    # Normalize inputs
    pairs = pairs_df.copy()
    pairs["ticker"] = pairs["ticker"].astype(str)
    pairs["date"] = pd.to_datetime(pairs["date"]).dt.normalize()

    tickers = sorted(pairs["ticker"].unique())
    min_date = pairs["date"].min()
    max_date = pairs["date"].max()

    # Expand the fetch window so rolling features are reproducible
    fetch_start = min_date - pd.Timedelta(days=int(lookback_days_buffer))
    fetch_end = max_date

    # --- Step 1: Re-fetch raw OHLCV once ---
    ohlcv = fetch_ohlcv_range(
        tickers=tickers,
        start_date=fetch_start,
        end_date=fetch_end,
        interval=interval,
        auto_adjust=auto_adjust,
        threads=threads,
        include_end=True,
    )

    if ohlcv.empty:
        raise ValueError("No OHLCV data returned; cannot re-derive dataset.")

    # --- Step 2: Recompute all features deterministically ---
    feats = engineer_technical_features_fast(
        ohlcv,
        **feature_kwargs,
    )

    pairs = pairs_df.copy()
    pairs["ticker"] = pairs["ticker"].astype(str)
    pairs["date"] = pd.to_datetime(pairs["date"]).dt.normalize()

    # map Fri (or any day) to that week’s Monday
    pairs["date"] = pairs["date"] - pd.to_timedelta(pairs["date"].dt.weekday, unit="D")


    # --- Step 3: Filter back down to exactly the saved (ticker, date) pairs ---
    out = (
        feats
        .reset_index()
        .merge(pairs, on=["ticker", "date"], how="inner")
        .sort_values(["date", "ticker"])
        .reset_index(drop=True)
    )


    feats_i = feats.reset_index()[["ticker", "date"]].drop_duplicates()
    pairs_i = pairs[["ticker", "date"]].drop_duplicates()

    print("pairs date sample:", sorted(pairs_i["date"].unique())[:10])
    print("feats  date sample:", sorted(feats_i["date"].unique())[:10])

    # how many pairs have an exact date match for that ticker?
    tmp = pairs_i.merge(feats_i, on=["ticker","date"], how="left", indicator=True)
    print(tmp["_merge"].value_counts())


    return out

In [ ]:
from __future__ import annotations

"""Fast weekly universe selection

Key speed changes vs your original:
- Fetch OHLCV ONCE for the full horizon (with a lookback buffer).
- Engineer TA features ONCE (vectorized groupby/transform where possible).
- Slice cross-sections by date from the cached feature panel.
- Replace full NxN Spearman+distance matrix with farthest-first (maximin)
  using on-demand correlations: O(k*n*p) instead of O(n^2*p).
- Reduce Python-level per-ticker loops; keep a small optional slow-features path.
- Remove duplicate resample_to_weekly and unreachable returns.

Notes:
- This keeps your overall workflow and outputs but is much faster for
  large ticker sets and many weekly steps.
"""

from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple, Union, Literal

import numpy as np
import pandas as pd

try:
    import yfinance as yf
except ImportError as e:
    raise ImportError("Please install yfinance: pip install yfinance") from e

try:
    # optional; used for better tie handling in ranks
    from scipy.stats import rankdata  # type: ignore

    _HAVE_SCIPY = True
except Exception:
    _HAVE_SCIPY = False

from sklearn.ensemble import IsolationForest
import tqdm


# ---------------------------------------------------------------------
# 1) Data fetch (single shot) + light caching
# ---------------------------------------------------------------------


def fetch_ohlcv_range(
    tickers: Sequence[str],
    start_date: Union[str, pd.Timestamp],
    end_date: Union[str, pd.Timestamp],
    *,
    interval: str = "1d",
    auto_adjust: bool = False,
    threads: bool | int = True,
    include_end: bool = True,
) -> pd.DataFrame:
    """Fetch OHLCV for a full date range once.

    Returns MultiIndex (ticker, date) with columns open/high/low/close/volume.
    """
    if not tickers:
        raise ValueError("tickers must be non-empty")

    start = pd.to_datetime(start_date).normalize()
    end = pd.to_datetime(end_date).normalize()
    yf_end = (end + pd.Timedelta(days=1)) if include_end else end

    raw = yf.download(
        list(tickers),
        start=start.strftime("%Y-%m-%d"),
        end=yf_end.strftime("%Y-%m-%d"),
        interval=interval,
        group_by="ticker",
        auto_adjust=auto_adjust,
        progress=False,
        threads=threads,
    )

    if raw is None or raw.empty:
        return pd.DataFrame(columns=["open", "high", "low", "close", "volume"]).set_index(
            pd.MultiIndex.from_arrays([[], []], names=["ticker", "date"])
        )

    frames: List[pd.DataFrame] = []
    if isinstance(raw.columns, pd.MultiIndex):
        for tkr in tickers:
            if tkr not in raw.columns.get_level_values(0):
                continue
            df_t = raw[tkr].copy().rename(columns=str.lower)
            keep = [c for c in ["open", "high", "low", "close", "volume"] if c in df_t.columns]
            df_t = df_t[keep].dropna(how="all")
            df_t.index = pd.to_datetime(df_t.index).normalize()
            df_t["ticker"] = tkr
            frames.append(df_t.reset_index(names="date"))
    else:
        df_t = raw.copy().rename(columns=str.lower)
        keep = [c for c in ["open", "high", "low", "close", "volume"] if c in df_t.columns]
        df_t = df_t[keep].dropna(how="all")
        df_t.index = pd.to_datetime(df_t.index).normalize()
        df_t["ticker"] = tickers[0]
        frames.append(df_t.reset_index(names="date"))

    if not frames:
        return pd.DataFrame(columns=["open", "high", "low", "close", "volume"]).set_index(
            pd.MultiIndex.from_arrays([[], []], names=["ticker", "date"])
        )

    out = (
        pd.concat(frames, ignore_index=True)
        .set_index(["ticker", "date"])
        .sort_index(level=["ticker", "date"])
    )

    for c in ["open", "high", "low", "close", "volume"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")

    return out


@dataclass
class FeatureCache:
    """Holds OHLCV + engineered features for fast slicing."""

    ohlcv: pd.DataFrame
    feats: pd.DataFrame

    @classmethod
    def build(
        cls,
        *,
        tickers: Sequence[str],
        start_date: Union[str, pd.Timestamp],
        end_date: Union[str, pd.Timestamp],
        lookback_days_buffer: int,
        feature_kwargs: Optional[Dict[str, Any]] = None,
        interval: str = "1d",
        auto_adjust: bool = False,
        threads: bool | int = True,
        include_end: bool = True,
        cache_parquet_path: Optional[str] = None,
    ) -> "FeatureCache":
        """Build cached features once.

        If cache_parquet_path is provided, will read/write a parquet file containing
        the engineered features (including OHLCV).
        """
        feature_kwargs = feature_kwargs or {}

        start = pd.to_datetime(start_date).normalize() - pd.Timedelta(days=int(lookback_days_buffer))
        end = pd.to_datetime(end_date).normalize()

        if cache_parquet_path is not None:
            try:
                df = pd.read_parquet(cache_parquet_path)
                # Expect MultiIndex columns? We'll store as regular with columns ticker/date.
                if {"ticker", "date"}.issubset(df.columns):
                    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
                    df = df.set_index(["ticker", "date"]).sort_index()
                    ohlcv = df[[c for c in ["open", "high", "low", "close", "volume"] if c in df.columns]].copy()
                    feats = df.drop(columns=[])
                    return cls(ohlcv=ohlcv, feats=feats)
            except Exception:
                pass

        ohlcv = fetch_ohlcv_range(
            tickers=tickers,
            start_date=start,
            end_date=end,
            interval=interval,
            auto_adjust=auto_adjust,
            threads=threads,
            include_end=include_end,
        )
        if ohlcv.empty:
            raise ValueError("No OHLCV data returned for the requested range.")

        feats = engineer_technical_features_fast(ohlcv, **feature_kwargs)

        if cache_parquet_path is not None:
            tmp = feats.reset_index()
            tmp.to_parquet(cache_parquet_path, index=False)

        return cls(ohlcv=ohlcv, feats=feats)


# ---------------------------------------------------------------------
# 2) Fast(er) technical features
# ---------------------------------------------------------------------


def _grouped(series: pd.Series) -> pd.core.groupby.SeriesGroupBy:
    return series.groupby(level="ticker", sort=False)


def _rsi_grouped(close: pd.Series, window: int) -> pd.Series:
    g = _grouped(close)
    delta = g.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = (-delta.where(delta < 0, 0.0))
    avg_gain = gain.groupby(level="ticker", sort=False).ewm(alpha=1 / window, adjust=False).mean().reset_index(level=0, drop=True)
    avg_loss = loss.groupby(level="ticker", sort=False).ewm(alpha=1 / window, adjust=False).mean().reset_index(level=0, drop=True)
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def _true_range_grouped(high: pd.Series, low: pd.Series, close: pd.Series) -> pd.Series:
    g = _grouped(close)
    prev_close = g.shift(1)
    tr1 = high - low
    tr2 = (high - prev_close).abs()
    tr3 = (low - prev_close).abs()
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    return tr


def _atr_grouped(high: pd.Series, low: pd.Series, close: pd.Series, window: int) -> pd.Series:
    tr = _true_range_grouped(high, low, close)
    atr = tr.groupby(level="ticker", sort=False).ewm(alpha=1 / window, adjust=False).mean().reset_index(level=0, drop=True)
    return atr


def _stoch_k_grouped(high: pd.Series, low: pd.Series, close: pd.Series, window: int) -> pd.Series:
    gH = _grouped(high)
    gL = _grouped(low)
    ll = gL.rolling(window, min_periods=window).min().reset_index(level=0, drop=True)
    hh = gH.rolling(window, min_periods=window).max().reset_index(level=0, drop=True)
    return 100 * (close - ll) / (hh - ll).replace(0, np.nan)


def _obv_grouped(close: pd.Series, volume: pd.Series) -> pd.Series:
    g = _grouped(close)
    direction = np.sign(g.diff()).fillna(0.0)
    signed = direction * volume
    return signed.groupby(level="ticker", sort=False).cumsum()


def _approx_slope_grouped(close: pd.Series, window: int) -> pd.Series:
    """Fast slope proxy: (last-first)/window. Much faster than polyfit."""
    g = _grouped(close)
    first = g.rolling(window, min_periods=window).apply(lambda x: x[0], raw=True).reset_index(level=0, drop=True)
    last = g.rolling(window, min_periods=window).apply(lambda x: x[-1], raw=True).reset_index(level=0, drop=True)
    return (last - first) / float(window)


def engineer_technical_features_fast(
    ohlcv: pd.DataFrame,
    *,
    ma_windows: Sequence[int] = (5, 10, 20, 30),
    ema_windows: Sequence[int] = (12, 26),
    rsi_window: int = 14,
    atr_window: int = 14,
    bb_window: int = 20,
    bb_n_std: float = 2.0,
    stoch_window: int = 14,
    stoch_smooth_k: int = 3,
    roc_windows: Sequence[int] = (5, 10, 20),
    vol_windows: Sequence[int] = (10, 20, 30),
    lag_days: Sequence[int] = (1, 2, 5, 10),
    roll_mom_windows: Sequence[int] = (10, 20, 60),
    roll_shape_windows: Sequence[int] = (20, 60),
    vwap_window: int = 20,
    cmf_window: int = 20,
    drop_original_ohlcv: bool = False,
    include_slow_features: bool = False,
) -> pd.DataFrame:
    """Vectorized TA features per ticker.

    include_slow_features:
      If True, computes a small set of more expensive rolling-apply features.
      Keep False for maximum speed.
    """

    if not isinstance(ohlcv.index, pd.MultiIndex) or ohlcv.index.names != ["ticker", "date"]:
        raise ValueError("ohlcv must have MultiIndex with names ['ticker', 'date']")

    required = {"open", "high", "low", "close"}
    missing = required - set(ohlcv.columns)
    if missing:
        raise ValueError(f"ohlcv missing required columns: {sorted(missing)}")

    df = ohlcv.copy().sort_index(level=["ticker", "date"])
    has_vol = "volume" in df.columns

    o = df["open"]
    h = df["high"]
    l = df["low"]
    c = df["close"]
    v = df["volume"] if has_vol else None

    gC = _grouped(c)

    feats = pd.DataFrame(index=df.index)

    # Returns / ranges
    feats["ret_1d"] = gC.pct_change(1)
    feats["log_ret_1d"] = np.log(c / gC.shift(1))
    feats["hl_range"] = (h - l) / c.replace(0, np.nan)
    feats["oc_return"] = (c - o) / o.replace(0, np.nan)
    feats["gap_return"] = (o - gC.shift(1)) / gC.shift(1).replace(0, np.nan)

    # Moving averages
    for w in ma_windows:
        sma = gC.rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
        feats[f"sma_{w}"] = sma
        feats[f"close_sma_ratio_{w}"] = c / sma.replace(0, np.nan)

    for w in ema_windows:
        ema = gC.ewm(span=w, adjust=False).mean().reset_index(level=0, drop=True)
        feats[f"ema_{w}"] = ema
        feats[f"close_ema_ratio_{w}"] = c / ema.replace(0, np.nan)

    # MACD
    ema12 = gC.ewm(span=12, adjust=False).mean().reset_index(level=0, drop=True)
    ema26 = gC.ewm(span=26, adjust=False).mean().reset_index(level=0, drop=True)
    macd = ema12 - ema26
    macd_signal = macd.groupby(level="ticker", sort=False).ewm(span=9, adjust=False).mean().reset_index(level=0, drop=True)
    feats["macd"] = macd
    feats["macd_signal"] = macd_signal
    feats["macd_hist"] = macd - macd_signal

    # RSI / ATR
    feats[f"rsi_{rsi_window}"] = _rsi_grouped(c, rsi_window)
    atr = _atr_grouped(h, l, c, atr_window)
    feats[f"atr_{atr_window}"] = atr
    feats[f"atrp_{atr_window}"] = atr / c.replace(0, np.nan)

    # Bollinger
    mid = gC.rolling(bb_window, min_periods=bb_window).mean().reset_index(level=0, drop=True)
    sd = gC.rolling(bb_window, min_periods=bb_window).std(ddof=0).reset_index(level=0, drop=True)
    upper = mid + bb_n_std * sd
    lower = mid - bb_n_std * sd
    feats[f"bb_mid_{bb_window}"] = mid
    feats[f"bb_upper_{bb_window}"] = upper
    feats[f"bb_lower_{bb_window}"] = lower
    feats[f"bb_width_{bb_window}"] = (upper - lower) / mid.replace(0, np.nan)
    feats[f"bb_pos_{bb_window}"] = (c - lower) / (upper - lower).replace(0, np.nan)

    # Stoch
    k = _stoch_k_grouped(h, l, c, stoch_window)
    feats[f"stoch_k_{stoch_window}"] = k
    feats[f"stoch_d_{stoch_window}"] = (
        k.groupby(level="ticker", sort=False)
        .rolling(stoch_smooth_k, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

    # ROC / momentum
    for w in roc_windows:
        feats[f"roc_{w}"] = gC.pct_change(w)
        feats[f"mom_{w}"] = c - gC.shift(w)

    # Volatility + rolling shape
    for w in vol_windows:
        feats[f"vol_{w}"] = (
            feats["log_ret_1d"].groupby(level="ticker", sort=False)
            .rolling(w, min_periods=max(2, w // 2))
            .std(ddof=0)
            .reset_index(level=0, drop=True)
        )

    for w in roll_shape_windows:
        rr = feats["log_ret_1d"].groupby(level="ticker", sort=False).rolling(w, min_periods=max(3, w // 2))
        feats[f"ret_skew_{w}"] = rr.skew().reset_index(level=0, drop=True)
        feats[f"ret_kurt_{w}"] = rr.kurt().reset_index(level=0, drop=True)

    # Rolling sharpe-ish + optional slope proxy
    for w in roll_mom_windows:
        mean = (
            feats["log_ret_1d"].groupby(level="ticker", sort=False)
            .rolling(w, min_periods=max(5, w // 2))
            .mean()
            .reset_index(level=0, drop=True)
        )
        std = (
            feats["log_ret_1d"].groupby(level="ticker", sort=False)
            .rolling(w, min_periods=max(5, w // 2))
            .std(ddof=0)
            .reset_index(level=0, drop=True)
        )
        feats[f"roll_sharpe_{w}"] = mean / std.replace(0, np.nan)

        if include_slow_features:
            # This is still much cheaper than polyfit, but uses rolling apply for first/last.
            feats[f"slope_{w}"] = _approx_slope_grouped(c, w)

    # Volume-derived features
    if has_vol and v is not None:
        gV = _grouped(v)
        vol_sma_10 = gV.rolling(10, min_periods=1).mean().reset_index(level=0, drop=True)
        feats["vol_sma_10"] = vol_sma_10
        feats["vol_rel_10"] = v / vol_sma_10.replace(0, np.nan)
        feats["obv"] = _obv_grouped(c, v)

        tp = (h + l + c) / 3.0
        pv = tp * v
        feats[f"vwap_{vwap_window}"] = (
            pv.groupby(level="ticker", sort=False).rolling(vwap_window, min_periods=vwap_window).sum().reset_index(level=0, drop=True)
            / gV.rolling(vwap_window, min_periods=vwap_window).sum().reset_index(level=0, drop=True).replace(0, np.nan)
        )
        feats[f"close_vwap_ratio_{vwap_window}"] = c / feats[f"vwap_{vwap_window}"].replace(0, np.nan)

        # CMF
        mfm = ((c - l) - (h - c)) / (h - l).replace(0, np.nan)
        mfv = mfm * v
        feats[f"cmf_{cmf_window}"] = (
            mfv.groupby(level="ticker", sort=False).rolling(cmf_window, min_periods=cmf_window).sum().reset_index(level=0, drop=True)
            / gV.rolling(cmf_window, min_periods=cmf_window).sum().reset_index(level=0, drop=True).replace(0, np.nan)
        )

    # Lags (only a compact set by default; extend if you want)
    lag_cols = [
        "ret_1d",
        "log_ret_1d",
        f"rsi_{rsi_window}",
        "macd",
        "macd_hist",
        f"atrp_{atr_window}",
        f"bb_pos_{bb_window}",
        f"stoch_k_{stoch_window}",
    ]
    if has_vol:
        lag_cols += ["obv"]

    lag_cols = [cname for cname in lag_cols if cname in feats.columns]
    for lag in lag_days:
        for cname in lag_cols:
            feats[f"{cname}_lag{lag}"] = feats[cname].groupby(level="ticker", sort=False).shift(lag)

    if drop_original_ohlcv:
        return feats

    return df.join(feats, how="left")


# ---------------------------------------------------------------------
# 3) Weekly resampling
# ---------------------------------------------------------------------


def resample_to_weekly(df: pd.DataFrame) -> pd.DataFrame:
    """Downsample a MultiIndex (ticker, date) DataFrame to weekly."""
    agg_dict: Dict[str, str] = {}
    for col in df.columns:
        if col == "open":
            agg_dict[col] = "first"
        elif col == "high":
            agg_dict[col] = "max"
        elif col == "low":
            agg_dict[col] = "min"
        elif col == "close":
            agg_dict[col] = "last"
        elif col == "volume":
            agg_dict[col] = "sum"
        else:
            agg_dict[col] = "last"

    weekly_df = (
        df.groupby(level="ticker", sort=False)
        .resample("W", level="date", label="right", closed="right")
        .agg(agg_dict)
    )
    return weekly_df


# ---------------------------------------------------------------------
# 4) Fast Spearman-on-rows + maximin without NxN distance matrix
# ---------------------------------------------------------------------


def _row_ranks(A: np.ndarray) -> np.ndarray:
    """Return within-row ranks for Spearman; shape (n, p)."""
    if _HAVE_SCIPY:
        # average ranks for ties
        return np.vstack([rankdata(row, method="average") for row in A]).astype(np.float32)

    # fast-ish fallback; tie handling is weaker
    order = np.argsort(A, axis=1)
    ranks = np.empty_like(order, dtype=np.float32)
    # ranks 0..p-1
    base = np.arange(A.shape[1], dtype=np.float32)
    for i in range(A.shape[0]):
        ranks[i, order[i]] = base
    return ranks


def _standardize_rows(X: np.ndarray) -> np.ndarray:
    """Center and scale each row; NaNs become 0 after standardization."""
    mu = np.nanmean(X, axis=1, keepdims=True)
    Z = X - mu
    sd = np.nanstd(Z, axis=1, ddof=0, keepdims=True)
    sd = np.where(sd == 0, np.nan, sd)
    Z = Z / sd
    Z = np.nan_to_num(Z, nan=0.0, posinf=0.0, neginf=0.0)
    return Z.astype(np.float32)


def _corr_to_dist(corr: np.ndarray) -> np.ndarray:
    """HRP-style dist = sqrt(0.5*(1-corr))."""
    corr = np.clip(corr, -1.0, 1.0)
    return np.sqrt(0.5 * (1.0 - corr)).astype(np.float32)


def maximin_select_by_spearman(
    X: pd.DataFrame,
    *,
    k: int,
    seed: Literal["medoid", "random", "first"] = "medoid",
    random_state: Optional[int] = 42,
) -> List[str]:
    """Select k tickers from a cross-sectional feature table (index=ticker).

    Uses Spearman similarity across feature vectors, but avoids building a full
    NxN matrix by computing correlations to the growing selected set on-demand.

    Complexity: O(k * n * p)
    """
    if k <= 0:
        return []

    Xn = X.apply(pd.to_numeric, errors="coerce")
    Xn = Xn.select_dtypes(include=[np.number])

    # Fill remaining NaNs with per-feature medians for stable ranks
    med = Xn.median(axis=0, skipna=True)
    A = Xn.fillna(med).to_numpy(dtype=np.float32)

    n, p = A.shape
    if n == 0:
        return []

    k = min(k, n)

    # Spearman via Pearson on row ranks
    R = _row_ranks(A)
    Z = _standardize_rows(R)

    # correlation(i,j) = (Zi·Zj)/p
    # We'll compute dot products to selected set incrementally.

    rng = np.random.default_rng(random_state)

    if seed == "first":
        seed_idx = 0
    elif seed == "random":
        seed_idx = int(rng.integers(0, n))
    elif seed == "medoid":
        # medoid approx: choose row with maximum average correlation
        # avg corr ~ (Z @ mean(Z).T)/p
        m = Z.mean(axis=0, keepdims=True)
        avg_corr = (Z @ m.T).reshape(-1) / max(p, 1)
        seed_idx = int(np.argmax(avg_corr))
    else:
        raise ValueError("seed must be one of {'medoid','random','first'}")

    selected: List[int] = [seed_idx]
    # Start with distances to seed
    corr_to_seed = (Z @ Z[seed_idx].T) / max(p, 1)
    min_d = _corr_to_dist(corr_to_seed)
    min_d[seed_idx] = -np.inf

    for _ in range(1, k):
        nxt = int(np.argmax(min_d))
        if not np.isfinite(min_d[nxt]):
            break
        selected.append(nxt)

        corr_to_new = (Z @ Z[nxt].T) / max(p, 1)
        d_to_new = _corr_to_dist(corr_to_new)
        min_d = np.minimum(min_d, d_to_new)
        for idx in selected:
            min_d[idx] = -np.inf

    tickers = Xn.index.astype(str).tolist()
    return [tickers[i] for i in selected]


def cross_section_asof(
    feats_panel: pd.DataFrame,
    asof_date: Union[str, pd.Timestamp],
    *,
    min_non_nan_frac: float = 0.8,
    allow_prior_date_fallback: bool = True,
) -> Tuple[pd.Timestamp, pd.DataFrame]:
    """Extract a 1-row-per-ticker feature table as of a date.

    Returns (used_date, X_date) where X_date index=ticker.
    """
    asof = pd.to_datetime(asof_date).normalize()
    dates = feats_panel.index.get_level_values("date")

    if asof in dates:
        used = asof
        X_date = feats_panel.xs(used, level="date").copy()
    else:
        if not allow_prior_date_fallback:
            return asof, pd.DataFrame()

        xf = feats_panel.reset_index()
        xf = xf.loc[xf["date"] <= asof]
        if xf.empty:
            return asof, pd.DataFrame()
        xf = xf.sort_values(["ticker", "date"]).groupby("ticker", as_index=False).tail(1)
        used = asof
        X_date = xf.set_index("ticker").drop(columns=["date"])

    # Keep numeric columns
    X_date = X_date.apply(pd.to_numeric, errors="coerce")

    # drop sparse columns
    keep_cols = X_date.columns[(X_date.isna().mean() <= (1.0 - min_non_nan_frac))]
    X_date = X_date[keep_cols]

    # drop sparse rows
    row_keep = X_date.isna().mean(axis=1) <= (1.0 - min_non_nan_frac)
    X_date = X_date.loc[row_keep]

    return used, X_date


# ---------------------------------------------------------------------
# 6) Initial selection (fast version)
# ---------------------------------------------------------------------


def initial_date_maximin_universe_from_cache(
    *,
    feats_panel: pd.DataFrame,
    initial_date: Union[str, pd.Timestamp],
    max_tickers: int,
    seed: Literal["medoid", "random", "first"] = "medoid",
    min_non_nan_frac: float = 0.8,
) -> pd.DataFrame:
    used_date, X_date = cross_section_asof(
        feats_panel,
        initial_date,
        min_non_nan_frac=min_non_nan_frac,
        allow_prior_date_fallback=True,
    )

    if X_date.shape[0] < 2:
        raise ValueError("Not enough tickers with valid features on initial_date after filtering.")

    chosen = maximin_select_by_spearman(X_date, k=min(max_tickers, X_date.shape[0]), seed=seed)

    # Build a weekly snapshot for the chosen tickers: use resample on the full panel subset.
    X_selected_daily = feats_panel.loc[chosen].sort_index()
    X_weekly = resample_to_weekly(X_selected_daily)

    # take latest weekly bar per ticker
    latest = X_weekly.index.to_frame().groupby(level="ticker")["date"].idxmax()
    X_weekly = X_weekly.loc[latest]

    return X_weekly.reset_index()


# ---------------------------------------------------------------------
# 7) Rolling weekly IsolationForest selection (cached pool)
# ---------------------------------------------------------------------


def _ensure_datetime(x: Union[str, pd.Timestamp]) -> pd.Timestamp:
    ts = pd.Timestamp(x)
    if ts.tzinfo is not None:
        ts = ts.tz_convert(None)
    return ts


def _weekly_dates(
    start: Union[str, pd.Timestamp],
    end: Union[str, pd.Timestamp],
    anchor: str = "FRI",
    include_start: bool = True,
) -> List[pd.Timestamp]:
    start_ts = _ensure_datetime(start)
    end_ts = _ensure_datetime(end)
    freq = f"W-{anchor.upper()}"
    dates = list(pd.date_range(start=start_ts, end=end_ts, freq=freq))

    if include_start and (len(dates) == 0 or dates[0].normalize() != start_ts.normalize()):
        dates = [start_ts] + dates

    out: List[pd.Timestamp] = []
    seen: set[pd.Timestamp] = set()
    for d in dates:
        d = _ensure_datetime(d).normalize()
        if d not in seen and d <= end_ts.normalize():
            out.append(d)
            seen.add(d)
    out.sort()
    return out


def _numeric_feature_cols(df: pd.DataFrame) -> List[str]:
    return [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]


@dataclass
class RollingIFUniverseResult:
    weekly_df: pd.DataFrame
    selected_by_date: Dict[pd.Timestamp, List[str]]
    scores_by_date: Dict[pd.Timestamp, pd.Series]


def rolling_isoforest_weekly_universe_cached(
    *,
    feats_panel: pd.DataFrame,
    tickers: Sequence[str],
    initial_date: Union[str, pd.Timestamp],
    end_date: Union[str, pd.Timestamp],
    max_tickers: int,
    week_anchor: str = "FRI",
    min_non_nan_frac: float = 0.8,
    if_params: Optional[dict] = None,
    train_on_last_n_weeks: int = 1,
) -> RollingIFUniverseResult:
    """Rolling weekly IF selection using cached feature panel.

    - Pool for each week is just the cached cross-section slice.
    - Fit IF on the most recent N weeks of selected history (default: 1 = latest week).
    """
    if if_params is None:
        if_params = {}

    initial_ts = _ensure_datetime(initial_date).normalize()
    end_ts = _ensure_datetime(end_date).normalize()

    if end_ts < initial_ts:
        raise ValueError("end_date must be >= initial_date")

    if max_tickers <= 0:
        raise ValueError("max_tickers must be positive")

    max_tickers = min(max_tickers, len(tickers))

    default_if = dict(
        n_estimators=150,  # reduced from 500 (usually plenty)
        contamination="auto",
        random_state=42,
        n_jobs=-1,
    )
    default_if.update(if_params)

    # ---- Initial selection ----
    init_df = initial_date_maximin_universe_from_cache(
        feats_panel=feats_panel,
        initial_date=initial_ts,
        max_tickers=max_tickers,
        seed="medoid",
        min_non_nan_frac=min_non_nan_frac,
    ).copy()

    init_df["date"] = pd.to_datetime(init_df["date"]).dt.normalize()
    init_df["ticker"] = init_df["ticker"].astype(str)

    weekly_df = init_df.sort_values(["date", "ticker"]).reset_index(drop=True)

    selected_by_date: Dict[pd.Timestamp, List[str]] = {
        initial_ts: sorted(weekly_df.loc[weekly_df["date"] == weekly_df["date"].min(), "ticker"].unique().tolist())
    }
    scores_by_date: Dict[pd.Timestamp, pd.Series] = {}

    # Weekly loop dates
    dates = _weekly_dates(initial_ts, end_ts, anchor=week_anchor, include_start=True)
    future_dates = [d for d in dates if d > initial_ts]

    for dt in tqdm.tqdm(future_dates):
        used_date, pool_X = cross_section_asof(
            feats_panel,
            dt,
            min_non_nan_frac=min_non_nan_frac,
            allow_prior_date_fallback=True,
        )
        if pool_X.empty:
            continue

        pool_df = pool_X.copy()
        pool_df.insert(0, "ticker", pool_df.index.astype(str))
        pool_df.insert(0, "date", dt)
        pool_df = pool_df.reset_index(drop=True)

        # Shared numeric columns
        train_cols = _numeric_feature_cols(weekly_df)
        pool_cols = _numeric_feature_cols(pool_df)
        feat_cols = [c for c in train_cols if c in pool_cols and c not in ("date",)]
        if not feat_cols:
            raise ValueError("No shared numeric feature columns between weekly_df and pool_df.")

        # Train on last N weeks of selected history
        all_dates_sorted = sorted(weekly_df["date"].unique())
        train_dates = all_dates_sorted[-max(1, train_on_last_n_weeks) :]
        train_df = weekly_df.loc[weekly_df["date"].isin(train_dates)].copy()

        # Drop sparse rows (again) on the selected train slice
        def _filter_by_non_nan_frac(df_: pd.DataFrame, cols: List[str], min_frac: float) -> pd.DataFrame:
            frac = df_[cols].notna().mean(axis=1)
            return df_.loc[frac >= min_frac].copy()

        train_df = _filter_by_non_nan_frac(train_df, feat_cols, min_non_nan_frac)
        pool_df_f = _filter_by_non_nan_frac(pool_df, feat_cols, min_non_nan_frac)

        if train_df.empty or pool_df_f.empty:
            continue

        # Median impute using training medians
        med = train_df[feat_cols].median(numeric_only=True)
        X_train = train_df[feat_cols].fillna(med).to_numpy(dtype=np.float32)
        X_pool = pool_df_f[feat_cols].fillna(med).to_numpy(dtype=np.float32)

        iso = IsolationForest(**default_if)
        iso.fit(X_train)

        scores = iso.score_samples(X_pool)
        score_s = pd.Series(scores, index=pool_df_f["ticker"].values, name="if_score").sort_values(ascending=False)
        scores_by_date[dt] = score_s

        chosen = score_s.head(max_tickers).index.tolist()
        selected_by_date[dt] = chosen

        chosen_rows = pool_df_f.loc[pool_df_f["ticker"].isin(chosen)].copy()
        chosen_rows["date"] = dt

        weekly_df = (
            pd.concat([weekly_df, chosen_rows], axis=0, ignore_index=True)
            .sort_values(["date", "ticker"])
            .reset_index(drop=True)
        )

    return RollingIFUniverseResult(
        weekly_df=weekly_df,
        selected_by_date=selected_by_date,
        scores_by_date=scores_by_date,
    )


if __name__ == "__main__":
    # Define tickers somewhere
    tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "META"]

    initial_date = "2023-06-30"
    end_date = "2023-10-27"
    lookback_days = 90
    max_tickers = max(1, len(tickers) - 1)

    feature_kwargs = dict(
        ma_windows=(5, 10, 20, 30),
        ema_windows=(12, 26),
        rsi_window=14,
        atr_window=14,
        bb_window=20,
        bb_n_std=2.0,
        roc_windows=(5, 10, 20),
        vol_windows=(10, 20, 30),
        lag_days=(1, 2, 5),
        roll_mom_windows=(10, 20, 60),
        roll_shape_windows=(20, 60),
        vwap_window=20,
        cmf_window=20,
        drop_original_ohlcv=False,
        include_slow_features=False,
    )

    cache = FeatureCache.build(
        tickers=tickers,
        start_date=initial_date,
        end_date=end_date,
        lookback_days_buffer=lookback_days,
        feature_kwargs=feature_kwargs,
        cache_parquet_path=None,  # e.g. "features_cache.parquet"
    )

    res = rolling_isoforest_weekly_universe_cached(
        feats_panel=cache.feats,
        tickers=tickers,
        initial_date=initial_date,
        end_date=end_date,
        max_tickers=max_tickers,
        week_anchor="FRI",
        min_non_nan_frac=0.8,
        if_params=dict(n_estimators=150, random_state=42, n_jobs=-1),
        train_on_last_n_weeks=1,
    )

    weekly_df = res.weekly_df
    print("Total selected rows:", len(weekly_df))
    print("Unique dates:", weekly_df["date"].nunique())
    print("Latest date:", weekly_df["date"].max())

    for d, chosen in res.selected_by_date.items():
        print(d.date(), len(chosen), chosen[:10], "..." if len(chosen) > 10 else "")


In [ ]:
x_train=rederived_df.copy()
y_train=y.copy()

y=y_train.copy().dropna()
X=x_train.copy().loc[y.index]

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# ---- 1) Hill-climb metric: hc_metric(X, y) ----
# X is (n_samples, n_features). We'll turn it into a single signal by averaging features.
# Then compute Spearman correlation vs y.
def hc_spearman_ic(X: np.ndarray, y: np.ndarray) -> float:
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float).reshape(-1)

    if X.ndim == 1:
        s = X
    else:
        # simple aggregation: mean signal across engineered features
        s = np.nanmean(X, axis=1)

    mask = np.isfinite(s) & np.isfinite(y)
    if mask.sum() < 10:
        return -np.inf  # treat as terrible

    rho, _ = spearmanr(s[mask], y[mask])
    return float(rho) if np.isfinite(rho) else -np.inf

'''
# ---- 2) Instantiate your GAFeatureEngineerDEAP in hill-climb mode ----
# Use either pure hill climbing or GA-backed hill climbing.
ge = GAFeatureEngineerDEAP(
    # IMPORTANT: set the correct index levels/cols used in your X data
    index_cols=("ticker", "date"),

    # hill climbing metric (separate from GA metric)
    hc_metric=hc_spearman_ic,
    maximize_hc_metric=True,

    # choose one:
    search_mode="ga_hill_climb",  # GA proposes, HC accepts
    # search_mode="hill_climb",   # purely HC proposal logic (your original)

    # GA knobs (these still affect ga_hill_climb)
    population_size=10,
    generations=2,
    tournament_size=7,
    cxpb=0.5,
    mutpb=0.3,

    # HC knobs
    hc_start_with_one_identity=True,
    hc_include_identity_candidates=True,
    hc_prune=True,
    hc_patience=5,
    checkpoint_path='testing_checkpointing.json',
    checkpoint_every_accepts=1,
)

# ---- 3) Fit + transform ----
ge.fit(x_train, y_train)

X_train_engineered = ge.transform(x_train)

print("Selected programs:", ge.selected_programs_)
print("Selected feature names:", ge.selected_feature_names_)
print("Best HC score (Spearman):", ge.selected_fitness_)
'''

In [ ]:
eng2 = GAFeatureEngineerDEAP(
    search_mode="ga_hill_climb",
    hc_metric=hc_spearman_ic,
    maximize_hc_metric=True,
    random_state=42,
    checkpoint_path="testing_checkpointing.json",
    checkpoint_every_accepts=2,
    population_size=30,
    generations=5,
)

eng2.fit(x_train,y_train)

In [ ]:
import json
import re
import operator
import random
import inspect
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple, Union, Literal

import numpy as np

try:
    import pandas as pd
except Exception:  # pragma: no cover
    pd = None

import polars as pl

try:
    from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin
except Exception:  # pragma: no cover
    class BaseEstimator:
        pass
    class TransformerMixin:
        pass
    class RegressorMixin:
        pass

from deap import base, creator, tools, gp

Arg = Union[str, "FeatureNode"]
OpType = Literal["element-wise", "groupby", "time-series", "target_encoding"]
SearchMode = Literal["ga", "hill_climb", "ga_hill_climb"]


@dataclass(frozen=True)
class OperatorSpec:
    name: str
    arity: int
    operator_type: OpType
    fn: Callable
    cat_pos: tuple[int, ...]
    num_pos: tuple[int, ...]
    target_pos: Optional[int] = None


@dataclass(frozen=True)
class OperationSpec:
    name: str
    arity: int
    kind: str
    builder: Callable[["GAFeatureEngineerDEAP", List[pl.Expr], Dict[str, Any]], pl.Expr]
    formatter: Callable[[List[str], Dict[str, Any]], str]
    requires_group_col: bool = False
    restrict_group_cols_to_categoricals: bool = False


@dataclass
class FeatureNode:
    op_token: str
    op_name: str
    args: List[Arg]
    params: Dict[str, Any]
    kind: str


@dataclass(frozen=True)
class NumE:
    expr: pl.Expr
    pre_cols: Tuple[Tuple[str, pl.Expr], ...] = field(default_factory=tuple)


@dataclass(frozen=True)
class CatE:
    expr: pl.Expr


class GAFeatureEngineerDEAP(BaseEstimator, TransformerMixin, RegressorMixin):
    def __init__(
        self,
        index_cols: Tuple[str, str] = ("ticker", "date"),
        categorical_cols: Optional[Sequence[str]] = None,
        numeric_cols: Optional[Sequence[str]] = None,
        max_colname_len: int = 160,
        random_state: int = 0,
        target_encoding_smoothing: float = 10.0,
        enable_impostor_op: bool = False,

        # GA metric: used ONLY in search_mode="ga"
        metric: Optional[Callable[[np.ndarray, np.ndarray], float]] = None,
        maximize_metric: bool = False,

        # Hill-climb metric: used in search_mode="hill_climb" or "ga_hill_climb"
        hc_metric: Optional[Callable[[np.ndarray, np.ndarray], float]] = None,
        maximize_hc_metric: Optional[bool] = None,

        # GA knobs
        population_size: int = 300,
        generations: int = 25,
        tournament_size: int = 7,
        cxpb: float = 0.5,
        mutpb: float = 0.3,
        init_min_depth: int = 1,
        init_max_depth: int = 4,
        max_tree_height: int = 10,
        hall_of_fame: int = 5,
        parsimony_coefficient: float = 1e-4,
        early_stop_rounds: Optional[int] = 7,
        early_stop_tol: float = 1e-6,

        # hill-climb knobs (still apply in ga_hill_climb)
        hc_max_steps: int = 25,
        hc_num_gp_candidates: int = 50,
        hc_include_identity_candidates: bool = True,
        hc_start_with_one_identity: bool = True,
        hc_prune: bool = True,
        hc_patience: int = 5,
        hc_tol: float = 1e-12,

        checkpoint_path=None,
        checkpoint_every_accepts =np.inf,

        # new
        search_mode: SearchMode = "ga",
        verbose: bool = False,
    ):


        self.checkpoint_path=checkpoint_path
        self.checkpoint_every_accepts=checkpoint_every_accepts
        self.index_cols = index_cols
        self.categorical_cols = list(categorical_cols) if categorical_cols else []
        self.numeric_cols = list(numeric_cols) if numeric_cols else []
        self.max_colname_len = int(max_colname_len)
        self.random_state = int(random_state)
        self.target_encoding_smoothing = float(target_encoding_smoothing)
        self.enable_impostor_op = bool(enable_impostor_op)
        self._tmp_col_counter = 0
        self.verbose = bool(verbose)

        # GA metric
        self.metric = metric
        self.maximize_metric = bool(maximize_metric)
        if isinstance(self.metric, (list, tuple)):
            raise TypeError("Pass exactly one GA metric callable (not a list/tuple).")
        if self.metric is not None and not callable(self.metric):
            raise TypeError("metric must be a callable or None.")

        # HC metric
        self.hc_metric = hc_metric
        if self.hc_metric is not None and not callable(self.hc_metric):
            raise TypeError("hc_metric must be a callable or None.")
        # If not specified, default to GA maximize flag for HC as well (keeps old behavior workable)
        self.maximize_hc_metric = bool(self.maximize_metric if maximize_hc_metric is None else maximize_hc_metric)

        self.search_mode: SearchMode = search_mode

        # DEAP knobs
        self.population_size = int(population_size)
        self.generations = int(generations)
        self.tournament_size = int(tournament_size)
        self.cxpb = float(cxpb)
        self.mutpb = float(mutpb)
        self.init_min_depth = int(init_min_depth)
        self.init_max_depth = int(init_max_depth)
        self.max_tree_height = int(max_tree_height)
        self.hall_of_fame = int(hall_of_fame)
        self.parsimony_coefficient = float(parsimony_coefficient)
        self.early_stop_rounds = None if early_stop_rounds is None else int(early_stop_rounds)
        self.early_stop_tol = float(early_stop_tol)

        self._rng: random.Random = random.Random(self.random_state)

        # registries (keep your existing ones)
        self._custom_ops: Dict[str, OperatorSpec] = {}
        self._ops_elementary: Dict[str, OperationSpec] = {}
        self._ops_groupby: Dict[str, OperationSpec] = {}
        self._ops_ts: Dict[str, OperationSpec] = {}
        self._ops_target: Dict[str, OperationSpec] = {}

        # target encoding
        self._te_maps: Dict[str, pl.DataFrame] = {}
        self._te_global_mean: Optional[float] = None
        self._te_target_col: Optional[str] = None

        # hill climb knobs
        self.hc_max_steps = int(hc_max_steps)
        self.hc_num_gp_candidates = int(hc_num_gp_candidates)
        self.hc_include_identity_candidates = bool(hc_include_identity_candidates)
        self.hc_start_with_one_identity = bool(hc_start_with_one_identity)
        self.hc_prune = bool(hc_prune)
        self.hc_patience = int(hc_patience)
        self.hc_tol = float(hc_tol)

        # outputs
        self.selected_programs_: Optional[List[str]] = None
        self.selected_fitness_: Optional[float] = None
        self.selected_feature_names_: Optional[List[str]] = None

        self.hof_: Optional[tools.HallOfFame] = None
        self.best_programs_: Optional[List[str]] = None
        self.best_fitness_: Optional[List[float]] = None
        self.best_feature_names_: Optional[List[str]] = None

        # internal
        self._pset: Optional[gp.PrimitiveSetTyped] = None
        self._toolbox: Optional[base.Toolbox] = None
        self._fitted: bool = False

        # keep your existing operator registration
        self._register_default_ops()

    # -------------------------
    # You should KEEP all your existing helpers:
    #   _register_default_ops, _build_pset, _ensure_deap, _to_polars, _infer_numeric_cols,
    #   _fit_target_encoders, _apply_target_encoders, _program_to_numpy,
    #   _compile_to_numexpr, _individual_to_symbolic, _sanitize_feature_name, _dedupe_names, etc.
    # -------------------------

    # ---- existing default metric for GA mode ----
    def _default_metric(self, y_true: np.ndarray, y_pred: np.ndarray) -> float:
        return float(np.mean((y_pred - y_true) ** 2))

    def _compute_single_metric(self, y_true: np.ndarray, y_pred: np.ndarray) -> float:
        fn = self.metric if self.metric is not None else self._default_metric
        val = fn(y_true, y_pred)
        if not isinstance(val, (int, float, np.number)):
            raise TypeError(f"Metric must return a single number, got {type(val)}")
        val = float(val)
        if not np.isfinite(val):
            raise ValueError("Metric must return a finite number.")
        return val

    # ---- HC scoring ----
    def _hc_score(self, Xmat: np.ndarray, y: np.ndarray) -> float:
        if self.hc_metric is None:
            raise ValueError("hc_metric must be provided for hill climbing modes.")
        val = float(self.hc_metric(Xmat, y))
        if not np.isfinite(val):
            raise ValueError("hc_metric must return a finite number.")
        return val

    def _hc_improved(self, new_score: float, old_score: float) -> bool:
        if self.maximize_hc_metric:
            return new_score > (old_score + self.hc_tol)
        return new_score < (old_score - self.hc_tol)

    # -------------------------
    # GA-backed hill climbing
    # -------------------------
    @dataclass
    class _HCState:
        selected_inds: List[Any]
        selected_names: List[str]
        best_X: np.ndarray
        best_score: float
        no_improve: int = 0

    def _make_identity_individual(self, col: str):
        self._ensure_deap()
        expr = f"el__identity(num__{col})"
        return creator.Individual(gp.PrimitiveTree.from_string(expr, self._pset))

    def _fit_ga_hill_climb(
          self,
          df: pl.DataFrame,
          y_np: np.ndarray,

      ) -> "GAFeatureEngineerDEAP":

        """Hybrid mode:
        - GA generates candidates via evolution.
        - Evaluation uses HC metric incrementally (add candidate feature to current selected set).
        - Acceptance updates the HC state (selected feature set).
        - GA's original y/y_pred evaluation is not performed.
        """
        self._ensure_deap()

        def _patch_legacy_constants(expr: str) -> str:
          # Replace bare legacy token "_create_lit" with a callable literal
          return re.sub(
              r'(?<![\w])_create_lit(?!\s*\()',
              "_create_lit(0.0)",
              expr
          )

        # ---- init HC state (supports resume) ----
        def build_X(ind_list: List[Any]) -> np.ndarray:
            cols = [self._program_to_numpy(df, ind) for ind in ind_list]
            return np.column_stack(cols) if cols else np.empty((df.height, 0), dtype=float)


        if self.checkpoint_path is not None:
            # Rebuild accepted features from their string programs
            with open(self.checkpoint_path, "r") as f:
              resume_checkpoint = json.load(f)


            selected_inds = [
                  creator.Individual(
                      gp.PrimitiveTree.from_string(
                          _patch_legacy_constants(expr),
                          self._pset
                      )
                  )
                  for expr in resume_checkpoint["selected_programs"]
              ]



            selected_names = list(resume_checkpoint["selected_programs"])


            best_X = build_X(selected_inds)
            best_score = float(resume_checkpoint["best_score"])

            mask = np.isfinite(y_np) & np.all(np.isfinite(best_X), axis=1)
            best_score = self._hc_score(best_X[mask], y_np[mask])
            print(best_score)


            no_improve0 = int(resume_checkpoint.get("no_improve", 0))
        else:
            remaining_identity = set(self.numeric_cols)
            selected_inds: List[Any] = []
            selected_names: List[str] = []

            if self.hc_start_with_one_identity and remaining_identity:
                c0 = self._rng.choice(list(remaining_identity))
                ind0 = self._make_identity_individual(c0)
                selected_inds.append(ind0)
                selected_names.append(str(ind0))
                remaining_identity.remove(c0)

            if not selected_inds:
                if not remaining_identity:
                    raise ValueError("No numeric columns available for hill-climb.")
                c0 = self._rng.choice(list(remaining_identity))
                ind0 = self._make_identity_individual(c0)
                selected_inds.append(ind0)
                selected_names.append(str(ind0))
                remaining_identity.remove(c0)

            best_X = build_X(selected_inds)
            best_score = self._hc_score(best_X, y_np)
            no_improve0 = 0

        state = GAFeatureEngineerDEAP._HCState(
            selected_inds=selected_inds,
            selected_names=selected_names,
            best_X=best_X,
            best_score=best_score,
            no_improve=no_improve0,
        )


        # ---- DEAP fitness: based on HC metric for THIS individual (incremental) ----
        def _eval_hc(individual):
            try:
                feat = self._program_to_numpy(df, individual)
                X_try = np.column_stack([state.best_X, feat])
            except Exception:
                return (1e18,)  # minimize

            mask = np.isfinite(y_np) & np.all(np.isfinite(X_try), axis=1)
            if mask.sum() < max(5, int(0.2 * len(y_np))):
                return (1e18,)

            sc = self._hc_score(X_try[mask], y_np[mask])

            # accept if improves the running HC solution
            if self._hc_improved(sc, state.best_score):
                state.selected_inds.append(individual)
                state.selected_names.append(str(individual))
                state.best_X = X_try
                state.best_score = sc
                state.no_improve = 0


                # optional prune (same semantics as your original)
                if self.hc_prune and len(state.selected_inds) > 1:
                    # Greedy redundancy pruning: drop any feature whose removal doesn't hurt.
                    changed = True
                    while changed and len(state.selected_inds) > 1:
                        changed = False
                        for j in range(len(state.selected_inds)):
                            trial_inds = state.selected_inds[:j] + state.selected_inds[j + 1 :]
                            X_trial = build_X(trial_inds)
                            m2 = np.isfinite(y_np) & np.all(np.isfinite(X_trial), axis=1)
                            if m2.sum() < max(5, int(0.2 * len(y_np))):
                                continue
                            sc2 = self._hc_score(X_trial[m2], y_np[m2])
                            not_worse = (
                                sc2 >= state.best_score - self.hc_tol
                                if self.maximize_hc_metric
                                else sc2 <= state.best_score + self.hc_tol
                            )
                            if not_worse:
                                state.selected_inds.pop(j)
                                state.selected_names.pop(j)
                                state.best_X = X_trial
                                state.best_score = sc2
                                changed = True
                                break

                accepted_count = len(state.selected_inds)

                # After acceptance + optional prune, sync HC state onto self so get_hc_checkpoint() works
                self._hc_selected_inds_ = list(state.selected_inds)
                self.selected_programs_ = list(state.selected_names)
                self.selected_fitness_ = float(state.best_score)
                self._hc_no_improve_ = int(state.no_improve)


                               # inside _eval_hc, after acceptance + optional prune
                if self.checkpoint_path is not None and (accepted_count % self.checkpoint_every_accepts == 0):
                    with open(self.checkpoint_path, "w") as f:
                        json.dump(self.get_hc_checkpoint(), f)

            # translate to DEAP FitnessMin
            fit_val = -sc if self.maximize_hc_metric else sc
            return (fit_val + self.parsimony_coefficient * len(individual),)

        self._toolbox.register("evaluate", _eval_hc)

        # ---- run GA (unchanged loop structure) ----
        pop = self._toolbox.population(n=self.population_size)
        hof = tools.HallOfFame(maxsize=self.hall_of_fame)

        # initial eval
        for ind, fit in zip(pop, map(self._toolbox.evaluate, pop)):
            ind.fitness.values = fit
        hof.update(pop)

        best_hist = [hof[0].fitness.values[0]]
        no_improve_ga = 0

        for _gen in range(1, self.generations + 1):
            offspring = list(map(self._toolbox.clone, self._toolbox.select(pop, len(pop))))

            for c1, c2 in zip(offspring[::2], offspring[1::2]):
                if self._rng.random() < self.cxpb:
                    self._toolbox.mate(c1, c2)
                    del c1.fitness.values, c2.fitness.values

            for mut in offspring:
                if self._rng.random() < self.mutpb:
                    self._toolbox.mutate(mut)
                    del mut.fitness.values

            invalid = [ind for ind in offspring if not ind.fitness.valid]
            for ind, fit in zip(invalid, map(self._toolbox.evaluate, invalid)):
                ind.fitness.values = fit

            pop[:] = offspring
            hof.update(pop)

            cur_best = hof[0].fitness.values[0]
            best_hist.append(cur_best)

            if self.early_stop_rounds is not None:
                if len(best_hist) >= 2 and (best_hist[-2] - best_hist[-1]) <= self.early_stop_tol:
                    no_improve_ga += 1
                else:
                    no_improve_ga = 0
                if no_improve_ga >= self.early_stop_rounds:
                    break

        # ---- store hybrid outputs ----
        self.hof_ = hof
        self.best_programs_ = [str(ind) for ind in hof]
        self.best_fitness_ = [float(ind.fitness.values[0]) for ind in hof]
        self.best_feature_names_ = self._dedupe_names(
            [self._sanitize_feature_name(self._individual_to_symbolic(ind)) for ind in hof]
        )

        # HC-selected (accepted) programs
        self.selected_programs_ = list(state.selected_names)
        raw = [self._individual_to_symbolic(ind) for ind in state.selected_inds]
        raw = [self._sanitize_feature_name(s) for s in raw]
        self.selected_feature_names_ = self._dedupe_names(raw)
        self.selected_fitness_ = float(state.best_score)
        self._hc_selected_inds_ = list(state.selected_inds)

        self._fitted = True
        if self.checkpoint_path is not None:
          with open(self.checkpoint_path, "w") as f:
              json.dump(self.get_hc_checkpoint(), f)

        return self

    # -------------------------
    # fit(): route modes
    # -------------------------
    def fit(
            self,
            X: Any,
            y: Any,
            target_col: Optional[str] = None,
        ):

        df = self._to_polars(X)
        self.numeric_cols = self._infer_numeric_cols(df)

        y_np = np.asarray(y).reshape(-1).astype(float)
        if y_np.shape[0] != df.height:
            raise ValueError("y length != number of rows in X")

        if target_col is None:
            target_col = "__target__"
        self._te_target_col = target_col

        if self.categorical_cols:
            df_te = df.with_columns(pl.Series(target_col, y_np))
            self._fit_target_encoders(df_te, target_col=target_col)
            df = self._apply_target_encoders(df)

        # rebuild GP primitives now that columns/TE are known
        self._pset = None
        self._toolbox = None
        self._ensure_deap()

        if self.search_mode in ("hill_climb", "ga_hill_climb"):
            # smoke test: hc_metric signature
            if self.hc_metric is None:
                raise ValueError("hc_metric must be provided when search_mode is hill_climb or ga_hill_climb.")
            try:
                _ = float(self.hc_metric(np.zeros((len(y_np), 1)), y_np))
            except TypeError as e:
                raise TypeError(
                    "In hill climbing modes, hc_metric must have signature hc_metric(X, y)."
                ) from e

        if self.search_mode == "hill_climb":
            return self._fit_hill_climb(df, y_np)  # keep your existing implementation

        if self.search_mode == "ga_hill_climb":

            return self._fit_ga_hill_climb(
                df,
                y_np,

            )

        # --------------------
        # Pure GA mode (unchanged): metric(y, y_pred)
        # --------------------
        if self.metric is not None:
            # smoke test ga metric
            try:
                _ = float(self.metric(np.zeros(3), np.zeros(3)))
            except TypeError as e:
                raise TypeError(
                    "In ga mode, metric must have signature metric(y_true, y_pred)."
                ) from e

        def _eval_ga(individual):
            compiled = gp.compile(expr=individual, pset=self._pset)
            out = compiled() if callable(compiled) else compiled

            lf = df
            if out.pre_cols:
                for name, e in out.pre_cols:
                    lf = lf.with_columns(e.alias(name))

            pred = (
                lf.select(out.expr.alias("__pred__"))
                .to_series()
                .to_numpy()
                .astype(float, copy=False)
            )

            mask = np.isfinite(pred) & np.isfinite(y_np)
            if mask.sum() < max(5, int(0.2 * len(y_np))):
                return (1e18,)

            metric_val = self._compute_single_metric(y_np[mask], pred[mask])
            fitness_val = -metric_val if self.maximize_metric else metric_val
            return (fitness_val + self.parsimony_coefficient * len(individual),)

        self._toolbox.register("evaluate", _eval_ga)

        pop = self._toolbox.population(n=self.population_size)
        hof = tools.HallOfFame(maxsize=self.hall_of_fame)

        for ind, fit in zip(pop, map(self._toolbox.evaluate, pop)):
            ind.fitness.values = fit
        hof.update(pop)

        best_hist = [hof[0].fitness.values[0]]
        no_improve = 0

        for _gen in range(1, self.generations + 1):
            offspring = list(map(self._toolbox.clone, self._toolbox.select(pop, len(pop))))

            for c1, c2 in zip(offspring[::2], offspring[1::2]):
                if self._rng.random() < self.cxpb:
                    self._toolbox.mate(c1, c2)
                    del c1.fitness.values, c2.fitness.values

            for mut in offspring:
                if self._rng.random() < self.mutpb:
                    self._toolbox.mutate(mut)
                    del mut.fitness.values

            invalid = [ind for ind in offspring if not ind.fitness.valid]
            for ind, fit in zip(invalid, map(self._toolbox.evaluate, invalid)):
                ind.fitness.values = fit

            pop[:] = offspring
            hof.update(pop)

            cur_best = hof[0].fitness.values[0]
            best_hist.append(cur_best)

            if self.early_stop_rounds is not None:
                if len(best_hist) >= 2 and (best_hist[-2] - best_hist[-1]) <= self.early_stop_tol:
                    no_improve += 1
                else:
                    no_improve = 0
                if no_improve >= self.early_stop_rounds:
                    break

        self.hof_ = hof
        self.best_programs_ = [str(ind) for ind in hof]
        self.best_feature_names_ = self._dedupe_names(
            [self._sanitize_feature_name(self._individual_to_symbolic(ind)) for ind in hof]
        )
        self.best_fitness_ = [float(ind.fitness.values[0]) for ind in hof]

        self._fitted = True
        return self

    # -------------------------
    # transform(): keep your existing logic, but in ga_hill_climb we emit selected HC features
    # -------------------------
    def transform(self, X: Any) -> pl.DataFrame:
        df = self._to_polars(X)
        if self._te_maps and self._te_target_col is not None:
            df = self._apply_target_encoders(df)

        self._ensure_deap()

        if self.search_mode in ("hill_climb", "ga_hill_climb"):
            if not getattr(self, "_hc_selected_inds_", None):
                raise RuntimeError("Call fit(X, y) before transform().")

            names = getattr(self, "selected_feature_names_", None)
            if not names:
                raw = [self._individual_to_symbolic(ind) for ind in self._hc_selected_inds_]
                names = self._dedupe_names([self._sanitize_feature_name(s) for s in raw])

            lf = df.lazy()
            for ind, nm in zip(self._hc_selected_inds_, names):
                out = self._compile_to_numexpr(ind)
                if out.pre_cols:
                    for name, e in out.pre_cols:
                        lf = lf.with_columns(e.alias(name))
                lf = lf.with_columns(out.expr.alias(nm))

            out_df = lf.collect()
            base_cols = list(self.index_cols) + [c for c in self.numeric_cols if c in out_df.columns]
            return out_df.select(base_cols + list(names))

        # pure GA: emit HOF
        if self.hof_ is None:
            raise RuntimeError("Call fit(X, y) before transform().")

        names = getattr(self, "best_feature_names_", None)
        if not names:
            raw = [self._individual_to_symbolic(ind) for ind in self.hof_]
            names = self._dedupe_names([self._sanitize_feature_name(s) for s in raw])

        lf = df.lazy()
        for ind, nm in zip(self.hof_, names):
            compiled = gp.compile(expr=ind, pset=self._pset)
            out = compiled() if callable(compiled) else compiled
            if getattr(out, "pre_cols", None):
                for name, e in out.pre_cols:
                    lf = lf.with_columns(e.alias(name))
            lf = lf.with_columns(out.expr.alias(nm))

        out_df = lf.collect()
        base_cols = list(self.index_cols) + [c for c in self.numeric_cols if c in out_df.columns]
        return out_df.select(base_cols + list(names))



    def _compile_to_numexpr(self, individual) -> NumE:
        self._ensure_deap()
        compiled = gp.compile(expr=individual, pset=self._pset)
        out = compiled() if callable(compiled) else compiled
        if not isinstance(out, NumE):
            raise TypeError("Program did not compile to NumE.")
        return out


    def _program_to_numpy(self, df: pl.DataFrame, individual) -> np.ndarray:
        out = self._compile_to_numexpr(individual)

        lf = df.lazy()

        if out.pre_cols:
            lf = lf.with_columns_seq([e.alias(name) for name, e in out.pre_cols])

        lf = lf.select(out.expr.alias("__feat__"))

        arr = (
            lf.collect(cluster_with_columns=False)
              .to_series()
              .to_numpy()
              .astype(float, copy=False)
        )
        return arr


    def _to_polars(self, X: Any) -> pl.DataFrame:
        if isinstance(X, pl.DataFrame):
            df = X
        elif pd is not None and isinstance(X, pd.DataFrame):
            df_pd = X.reset_index() if isinstance(X.index, pd.MultiIndex) else X.copy()
            df = pl.from_pandas(df_pd)
        elif isinstance(X, np.ndarray):
            if not self.numeric_cols:
                raise ValueError("If passing a numpy array, you must provide numeric_cols in __init__.")
            if X.shape[1] != len(self.numeric_cols):
                raise ValueError("numpy array column count != len(numeric_cols).")
            df = pl.DataFrame({c: X[:, i] for i, c in enumerate(self.numeric_cols)})
        else:
            raise TypeError(f"Unsupported input type: {type(X)}")

        for c in self.index_cols:
            if c not in df.columns:
                raise ValueError(f"Missing required index column '{c}'.")
        return df


    def _infer_numeric_cols(self, df: pl.DataFrame) -> List[str]:
        if self.numeric_cols:
            return list(self.numeric_cols)
        forbidden = set(self.index_cols) | set(self.categorical_cols)
        return [c for c in df.columns if c not in forbidden]


    def _fit_target_encoders(self, df: pl.DataFrame, target_col: str) -> None:
        self._te_global_mean = float(df.select(pl.col(target_col).mean()).item())
        smooth = float(self.target_encoding_smoothing)
        for cat in self.categorical_cols:
            if cat not in df.columns:
                continue
            tec = self._te_colname(cat, target_col)
            agg = (
                df.group_by(cat, maintain_order=True)
                  .agg(pl.col(target_col).mean().alias("__m"), pl.len().alias("__n"))
                  .with_columns(
                      ((pl.col("__m") * pl.col("__n") + smooth * self._te_global_mean)
                      / (pl.col("__n") + smooth)).alias(tec)
                  )
                  .select([cat, tec])
            )
            self._te_maps[cat] = agg


    def _apply_target_encoders(self, df: pl.DataFrame) -> pl.DataFrame:
        assert self._te_target_col is not None
        assert self._te_global_mean is not None
        out = df
        for cat, mdf in self._te_maps.items():
            tec = self._te_colname(cat, self._te_target_col)
            out = (
                out.join(mdf, on=cat, how="left")
                  .with_columns(pl.col(tec).fill_null(self._te_global_mean))
            )
        return out


    def _dedupe_names(self, names: list[str]) -> list[str]:
        seen = {}
        out = []
        for n in names:
            k = n
            if k in seen:
                seen[k] += 1
                k = f"{k}__{seen[n]}"
            else:
                seen[k] = 0
            out.append(k)
        return out


    def _sanitize_feature_name(self, s: str) -> str:
        s = s.strip()
        s = re.sub(r"\s+", "_", s)
        s = re.sub(r"[^0-9a-zA-Z_()\[\],.=+\-*/|:]", "_", s)
        if len(s) > self.max_colname_len:
            suffix = hex(abs(hash(s)) % (16**6))[2:]
            keep = max(8, self.max_colname_len - (2 + len(suffix)))
            s = f"{s[:keep]}__{suffix}"
        return s



    def _individual_to_symbolic(self, individual) -> str:
        """
        Convert a DEAP Individual / PrimitiveTree into a human-readable symbolic string
        using the registered formatter functions.
        """
        # DEAP Individuals are iterable over nodes (Primitive / Terminal)
        nodes = list(individual)
        stack: list[str] = []

        for node in reversed(nodes):  # process prefix tree right-to-left
            if isinstance(node, gp.Terminal):
                # Terminals in typed GP often stringify to their "name" already.
                # If yours stringify to something verbose, you can use node.name when available.
                stack.append(str(node.name))
                continue

            if isinstance(node, gp.Primitive):
                args = [stack.pop() for _ in range(node.arity)]  # already left->right
                op_name = node.name.split("__", 1)[1] if "__" in node.name else node.name

                if node.name.startswith("el__"):
                    spec = self._ops_elementary.get(op_name)
                elif node.name.startswith("gb__"):
                    spec = self._ops_groupby.get(op_name)
                elif node.name.startswith("ts__"):
                    spec = self._ops_ts.get(op_name)
                elif node.name.startswith("te__"):
                    spec = self._ops_target.get(op_name)
                else:
                    spec = None

                if spec is not None:
                    stack.append(spec.formatter(args, {}))
                else:
                    stack.append(f"{op_name}({', '.join(args)})")
                continue

            raise TypeError(f"Unknown node type inside tree: {type(node)}")

        if len(stack) != 1:
            # If this triggers, something is inconsistent in the tree representation.
            raise ValueError(f"Could not render individual; stack={stack}")

        return stack[0]



    def _register_default_ops(self) -> None:
        def fmt(name: str, args: List[str], params: Dict[str, Any]) -> str:
            if not params:
                return f"{name}({', '.join(args)})"
            p = ",".join([f"{k}={params[k]}" for k in sorted(params)])
            return f"{name}[{p}]({', '.join(args)})"

        def safe_div(a: pl.Expr, b: pl.Expr, eps: float = 1e-12) -> pl.Expr:
            return pl.when(b.abs() > eps).then(a / b).otherwise(None)

        def rolling_min_key(method: Callable) -> str:
            params = set(inspect.signature(method).parameters.keys())
            return "min_samples" if "min_samples" in params else "min_periods"

        # ---- elementary ----
        self.add_elementary_op("add", 2, builder=lambda eng, a, p: a[0] + a[1], formatter=lambda args, p: fmt("add", args, p))
        self.add_elementary_op("sub", 2, builder=lambda eng, a, p: a[0] - a[1], formatter=lambda args, p: fmt("sub", args, p))
        self.add_elementary_op("mul", 2, builder=lambda eng, a, p: a[0] * a[1], formatter=lambda args, p: fmt("mul", args, p))
        self.add_elementary_op("div", 2, builder=lambda eng, a, p: safe_div(a[0], a[1]), formatter=lambda args, p: fmt("div", args, p))
        self.add_elementary_op("neg", 1, builder=lambda eng, a, p: -a[0], formatter=lambda args, p: fmt("neg", args, p))
        self.add_elementary_op("abs", 1, builder=lambda eng, a, p: a[0].abs(), formatter=lambda args, p: fmt("abs", args, p))
        self.add_elementary_op("sqrt_abs", 1, builder=lambda eng, a, p: a[0].abs().sqrt(), formatter=lambda args, p: fmt("sqrt_abs", args, p))
        self.add_elementary_op("log1p_abs", 1, builder=lambda eng, a, p: a[0].abs().log1p(), formatter=lambda args, p: fmt("log1p_abs", args, p))
        self.add_elementary_op("clip", 1, builder=lambda eng, a, p: a[0].clip(p.get("lo", -3.0), p.get("hi", 3.0)), formatter=lambda args, p: fmt("clip", args, p))

        self.add_elementary_op(
            "identity", 1,
            builder=lambda eng, a, p: a[0],
            formatter=lambda args, p: fmt("identity", args, p),
        )

        # ---- groupby (defaults to date group) ----
        self.add_groupby_op("cs_rank", 1,
            builder=lambda eng, a, p: a[0].rank(method=p.get("method","average"), descending=p.get("descending",True))
                                  .over(p.get("group_col", eng.index_cols[1])),
            formatter=lambda args, p: fmt("cs_rank", args, p),
        )

        self.add_groupby_op("cs_zscore", 1,
            builder=lambda eng, a, p: safe_div(
                (a[0] - a[0].mean().over(p.get("group_col", eng.index_cols[1]))),
                a[0].std().over(p.get("group_col", eng.index_cols[1]))
            ),
            formatter=lambda args, p: fmt("cs_zscore", args, p),
        )
        self.add_groupby_op("grp_mean", 1,
            builder=lambda eng, a, p: a[0].mean().over(p.get("group_col", eng.index_cols[1])),
            formatter=lambda args, p: fmt("grp_mean", args, p),
        )

        # ---- time-series (over ticker) ----
        min_key = rolling_min_key(pl.Expr.rolling_mean)

        def ts_over_ticker(expr: pl.Expr, eng: "GAFeatureEngineerDEAP") -> pl.Expr:
            return expr.over(eng.index_cols[0])

        self.add_ts_op("ts_lag", 1, builder=lambda eng, a, p: ts_over_ticker(a[0].shift(int(p.get("n",1))), eng),
                      formatter=lambda args, p: fmt("ts_lag", args, p))
        self.add_ts_op("ts_diff", 1, builder=lambda eng, a, p: ts_over_ticker(a[0].diff(int(p.get("n",1))), eng),
                      formatter=lambda args, p: fmt("ts_diff", args, p))
        self.add_ts_op("ts_pct_change", 1, builder=lambda eng, a, p: ts_over_ticker(a[0].pct_change(int(p.get("n",1))), eng),
                      formatter=lambda args, p: fmt("ts_pct_change", args, p))
        self.add_ts_op("ts_mean", 1, builder=lambda eng, a, p: ts_over_ticker(
                          a[0].rolling_mean(window_size=int(p.get("window",5)), **{min_key:int(p.get("min_samples",1))}), eng),
                      formatter=lambda args, p: fmt("ts_mean", args, p))
        self.add_ts_op("ts_std", 1, builder=lambda eng, a, p: ts_over_ticker(
                          a[0].rolling_std(window_size=int(p.get("window",5)), **{min_key:int(p.get("min_samples",2))}), eng),
                      formatter=lambda args, p: fmt("ts_std", args, p))
        self.add_ts_op("ts_ewm_mean", 1, builder=lambda eng, a, p: ts_over_ticker(
                          a[0].ewm_mean(span=float(p.get("span",10.0)), adjust=bool(p.get("adjust",False)), ignore_nulls=bool(p.get("ignore_nulls",False))), eng),
                      formatter=lambda args, p: fmt("ts_ewm_mean", args, p))

        if self.enable_impostor_op:
            self.add_elementary_op(
                "impostor_noise", 0,
                builder=lambda eng, a, p: eng._impostor_expr(seed=int(p.get("seed", eng.random_state))),
                formatter=lambda args, p: fmt("impostor_noise", [], p),
            )


    def _new_tmp_col(self, base: str) -> str:
      self._tmp_col_counter += 1
      return self._safe_ident(f"__tmp__{base}__{self._tmp_col_counter}")

    # -------------------------
    # KEEP create_operator SYNTAX + 4 OP TYPES
    # -------------------------
    def create_operator(
        self,
        operator_function: Callable[..., pl.Expr],
        arity: int,
        name: str,
        operator_type: OpType,
        cat_col_args: Optional[List[int]] = None,
        num_col_args: Optional[List[int]] = None,
        target_col_arg: Optional[int] = None,
    ) -> None:
        cat_col_args = cat_col_args or []
        num_col_args = num_col_args or []

        allowed: set[str] = {"element-wise", "groupby", "time-series", "target_encoding"}
        if operator_type not in allowed:
            raise ValueError(f"operator_type must be one of {sorted(allowed)}")
        if not isinstance(arity, int) or arity < 0:
            raise ValueError("arity must be a non-negative int")

        all_pos = list(cat_col_args) + list(num_col_args)
        if target_col_arg is not None:
            all_pos.append(target_col_arg)

        for p in all_pos:
            if not isinstance(p, int):
                raise TypeError("arg positions must be integers (0-based)")
            if p < 0 or p >= arity:
                raise ValueError(f"arg position {p} out of bounds for arity={arity}")
        if len(set(all_pos)) != len(all_pos):
            raise ValueError("cat/num/target arg positions must be disjoint")

        # guards to preserve your conceptual op types
        if operator_type == "element-wise":
            if cat_col_args:
                raise ValueError("element-wise ops cannot use categorical args")
            if target_col_arg is not None:
                raise ValueError("element-wise ops cannot use target_col_arg")

        if operator_type == "groupby":
            if target_col_arg is not None:
                raise ValueError("groupby ops cannot use target_col_arg")
            if len(cat_col_args) == 0:
                raise ValueError("groupby ops must have at least one categorical arg (group key)")
            if len(num_col_args) == 0:
                raise ValueError("groupby ops must have at least one numeric arg (value)")

        if operator_type == "time-series":
            if cat_col_args:
                raise ValueError("time-series ops cannot use categorical args (initially)")
            if target_col_arg is not None:
                raise ValueError("time-series ops cannot use target_col_arg")

        if operator_type == "target_encoding":
            if target_col_arg is None:
                raise ValueError("target_encoding ops must set target_col_arg")
            if len(cat_col_args) == 0:
                raise ValueError("target_encoding ops must have at least one categorical arg")

        spec = OperatorSpec(
            name=name,
            arity=arity,
            operator_type=operator_type,
            fn=operator_function,
            cat_pos=tuple(cat_col_args),
            num_pos=tuple(num_col_args),
            target_pos=target_col_arg,
        )
        self._custom_ops[name] = spec

        kind_map = {
            "element-wise": "elementary",
            "groupby": "groupby",
            "time-series": "ts",
            "target_encoding": "target",
        }
        kind = kind_map[operator_type]

        def _builder(eng: "GAFeatureEngineerDEAP", arg_exprs: List[pl.Expr], p: Dict[str, Any]) -> pl.Expr:
            kwargs = {k: v for k, v in p.items() if k != "bound_cols"}
            return operator_function(*arg_exprs, **kwargs)

        def _formatter(args: List[str], p: Dict[str, Any]) -> str:
            pp = {k: v for k, v in p.items() if k != "bound_cols"}
            if not pp:
                return f"{name}({', '.join(args)})"
            param_str = ",".join([f"{k}={pp[k]}" for k in sorted(pp)])
            return f"{name}[{param_str}]({', '.join(args)})"

        self.add_custom_operation(
            name=name,
            arity=arity,
            kind=kind,
            builder=_builder,
            formatter=_formatter,
            restrict_group_cols_to_categoricals=(operator_type == "groupby"),
        )

        # new primitives => rebuild pset/toolbox next time
        self._pset = None
        self._toolbox = None

    # registry helpers
    def add_elementary_op(self, name: str, arity: int, builder, formatter) -> None:
        self._ops_elementary[name] = OperationSpec(name=name, arity=arity, kind="elementary", builder=builder, formatter=formatter)

    def add_groupby_op(self, name: str, arity: int, builder, formatter, restrict_group_cols_to_categoricals: bool = False) -> None:
        self._ops_groupby[name] = OperationSpec(name=name, arity=arity, kind="groupby", builder=builder, formatter=formatter,
                                                restrict_group_cols_to_categoricals=restrict_group_cols_to_categoricals)

    def add_ts_op(self, name: str, arity: int, builder, formatter) -> None:
        self._ops_ts[name] = OperationSpec(name=name, arity=arity, kind="ts", builder=builder, formatter=formatter)

    def add_target_op(self, name: str, arity: int, builder, formatter) -> None:
        self._ops_target[name] = OperationSpec(name=name, arity=arity, kind="target", builder=builder, formatter=formatter)

    def add_custom_operation(self, name: str, arity: int, kind: str, builder, formatter=None, restrict_group_cols_to_categoricals: bool = False) -> None:
        if formatter is None:
            formatter = lambda args, p: f"{name}({', '.join(args)})"
        if kind == "elementary":
            self.add_elementary_op(name, arity, builder, formatter)
        elif kind == "groupby":
            self.add_groupby_op(name, arity, builder, formatter, restrict_group_cols_to_categoricals)
        elif kind == "ts":
            self.add_ts_op(name, arity, builder, formatter)
        elif kind == "target":
            self.add_target_op(name, arity, builder, formatter)
        else:
            raise ValueError("kind must be one of: elementary, groupby, ts, target")

    # -------------------------
    # Target encoding: turned into GP primitives automatically
    # -------------------------
    def _te_colname(self, cat_col: str, target_col: str) -> str:
        return f"__te__mean__{cat_col}__{target_col}"

    # -------------------------
    # DEAP primitive set build
    # -------------------------
    def _safe_ident(self, s: str) -> str:
        s2 = "".join(ch if (ch.isalnum() or ch == "_") else "_" for ch in str(s))
        if not s2 or s2[0].isdigit():
            s2 = "c_" + s2
        return s2

    def _build_pset(self) -> gp.PrimitiveSetTyped:
        """
        Build a typed DEAP primitive set that produces a NumE (numeric Polars expression).

        Key points:
          - Column references must be *terminals* (pset.addTerminal), not 0-arity primitives.
          - Random constants should be *ephemeral terminals* (pset.addEphemeralConstant).
          - Operators remain primitives returning NumE.
        """
        # No GP input args: compiled programs are called with no parameters
        pset = gp.PrimitiveSetTyped("MAIN", [], NumE)

        # -------------------------
        # Terminals: numeric columns
        # -------------------------
        for c in self.numeric_cols:
            term = NumE(pl.col(c).cast(pl.Float64))
            pset.addTerminal(term, NumE, name=self._safe_ident(f"num__{c}"))


        # -----------------------------
        # Terminals: categorical columns
        # -----------------------------
        for c in self.categorical_cols:
            term = CatE(pl.col(c).cast(pl.Utf8))
            pset.addTerminal(term, CatE, name=self._safe_ident(f"cat__{c}"))


        # 1. Helper function that runs during eval() to create the actual NumE
        def _create_lit(x: float) -> NumE:
            return NumE(pl.lit(x))

        # Register it in the pset context so the compiled code can find it
        pset.context["_create_lit"] = _create_lit

        # 2. Wrapper class to control the string representation
        class LiteralWrapper:
            def __init__(self, val: float):
                self.val = val

            def __str__(self):
                return f"_create_lit({self.val})"

            __repr__ = __str__

        # 3. Add ephemeral constant returning the wrapper, but typed as NumE
        pset.addEphemeralConstant(
            "const",
            lambda: LiteralWrapper(self._rng.uniform(-1.0, 1.0)),
            NumE,
        )

        # -------------------------
        # Helper to add operations
        # -------------------------
        def add_op(
            tag: str,
            op: OperationSpec,
            arg_tys: List[type],
            default_params: Optional[Dict[str, Any]] = None,
        ) -> None:
            dp = dict(default_params or {})

            def _is_literal_expr(e: pl.Expr) -> bool:
                """
                True if this expr references no root columns (e.g. pl.lit(0.5)).
                We use Polars meta introspection when available.
                """
                try:
                    # For literal-only expressions, root_names() is empty
                    return len(e.meta.root_names()) == 0
                except Exception:
                    # If Polars meta isn't available/compatible, fall back to False
                    # (you can switch this to True for maximum safety at the cost of more temp cols)
                    return False

            def _prim(*args):
                # args are NumE/CatE wrappers
                pre_cols: List[Tuple[str, pl.Expr]] = []
                expr_args: List[pl.Expr] = []

                for a in args:
                    if isinstance(a, NumE):
                        pre_cols.extend(list(a.pre_cols))
                        expr_args.append(a.expr)
                    else:  # CatE
                        expr_args.append(a.expr)

                # ------------------------------------------------------------
                # FIX: Polars cannot aggregate/window on a pure literal.
                # If a ts/groupby op gets a literal argument, materialize it as
                # a temporary column first, then pass pl.col(tmp) to the builder.
                # ------------------------------------------------------------
                if op.kind in ("ts", "groupby"):
                    for j, ex in enumerate(expr_args):
                        if _is_literal_expr(ex):
                            tmp_in = self._new_tmp_col(f"{tag}__{op.name}__arg{j}")
                            pre_cols.append((tmp_in, ex))
                            expr_args[j] = pl.col(tmp_in)

                expr = op.builder(self, expr_args, dp)

                # MATERIALIZE: treat ts + groupby ops as staged expressions
                if op.kind in ("ts", "groupby"):
                    tmp = self._new_tmp_col(f"{tag}__{op.name}")
                    pre_cols.append((tmp, expr))
                    return NumE(pl.col(tmp), tuple(pre_cols))

                return NumE(expr, tuple(pre_cols))

            name = self._safe_ident(f"{tag}__{op.name}")
            pset.addPrimitive(_prim, arg_tys, NumE, name=name)

        # -------------------------
        # Element-wise ops (NumE -> NumE)
        # -------------------------
        for op in self._ops_elementary.values():
            add_op("el", op, [NumE] * op.arity)

        # -------------------------
        # Groupby ops
        #   - built-ins numeric-only
        #   - custom groupby ops respect Cat/Num positions
        # -------------------------
        for op in self._ops_groupby.values():
            if (
                op.name in self._custom_ops
                and self._custom_ops[op.name].operator_type == "groupby"
            ):
                spec = self._custom_ops[op.name]
                arg_tys = [(CatE if i in spec.cat_pos else NumE) for i in range(spec.arity)]
                add_op("gb", op, arg_tys)
            else:
                add_op("gb", op, [NumE] * op.arity)

        # -------------------------
        # Time-series ops (numeric-only)
        # -------------------------
        ts_defaults = {
            "ts_lag": {"n": 1},
            "ts_diff": {"n": 1},
            "ts_pct_change": {"n": 1},
            "ts_mean": {"window": 5, "min_samples": 1},
            "ts_std": {"window": 5, "min_samples": 2},
            "ts_ewm_mean": {"span": 10.0, "adjust": False, "ignore_nulls": True},
        }
        for op in self._ops_ts.values():
            add_op("ts", op, [NumE] * op.arity, default_params=ts_defaults.get(op.name, {}))

        # -------------------------
        # Target-encoding terminals + custom TE ops
        # -------------------------
        # These TE columns exist only after fit() has built self._te_maps; at that point
        # we can expose them as numeric terminals.
        if self._te_target_col is not None and self._te_maps:
            for cat in self.categorical_cols:
                tec = self._te_colname(cat, self._te_target_col)
                # Terminal that references the materialized TE column
                pset.addTerminal(
                    NumE(pl.col(tec).cast(pl.Float64)),
                    NumE,
                    name=self._safe_ident(f"te_mean__{cat}"),
                )

        # Custom target_encoding ops: typed via cat/num/target slots
        for op in self._ops_target.values():
            if (
                op.name in self._custom_ops
                and self._custom_ops[op.name].operator_type == "target_encoding"
            ):
                spec = self._custom_ops[op.name]
                arg_tys = [(CatE if i in spec.cat_pos else NumE) for i in range(spec.arity)]
                add_op("te", op, arg_tys)

        return pset


    def _ensure_deap(self):

        import builtins

        def _create_lit(x: float) -> NumE:
            return NumE(pl.lit(x))

        builtins._create_lit = _create_lit

        if self._pset is None:
            self._pset = self._build_pset()

        if self._toolbox is None:
            # global creator registry (avoid redef)
            if not hasattr(creator, "FitnessMin"):
                creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
            if not hasattr(creator, "Individual"):
                creator.create("Individual", gp.PrimitiveTree, fitness=creator.FitnessMin)

            toolbox = base.Toolbox()
            toolbox.register("expr", gp.genHalfAndHalf, pset=self._pset, min_=self.init_min_depth, max_=self.init_max_depth)
            toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.expr)
            toolbox.register("population", tools.initRepeat, list, toolbox.individual)

            toolbox.register("select", tools.selTournament, tournsize=self.tournament_size)
            toolbox.register("mate", gp.cxOnePoint)
            toolbox.register("expr_mut", gp.genFull, min_=0, max_=2)
            toolbox.register("mutate", gp.mutUniform, expr=toolbox.expr_mut, pset=self._pset)

            toolbox.decorate("mate", gp.staticLimit(key=operator.attrgetter("height"), max_value=self.max_tree_height))
            toolbox.decorate("mutate", gp.staticLimit(key=operator.attrgetter("height"), max_value=self.max_tree_height))

            self._toolbox = toolbox


    def get_hc_checkpoint(self) -> dict:
        if not hasattr(self, "_hc_selected_inds_"):

            raise RuntimeError("No hill-climb state to checkpoint.")

        return {
            "selected_programs": [str(ind) for ind in self._hc_selected_inds_],
            "best_score": float(self.selected_fitness_),
            "no_improve": getattr(self, "_hc_no_improve_", 0),
        }


    def load_hc_checkpoint(self, checkpoint: dict, df: pl.DataFrame, y_np: np.ndarray):
        self._ensure_deap()

        inds = [
            creator.Individual(
                gp.PrimitiveTree.from_string(expr, self._pset)
            )
            for expr in checkpoint["selected_programs"]
        ]

        def build_X(ind_list):
            cols = [self._program_to_numpy(df, ind) for ind in ind_list]
            return np.column_stack(cols) if cols else np.empty((df.height, 0))

        X = build_X(inds)
        score = checkpoint["best_score"]

        self._hc_selected_inds_ = inds
        self.selected_programs_ = checkpoint["selected_programs"]
        self.selected_feature_names_ = self._dedupe_names(
            [self._sanitize_feature_name(self._individual_to_symbolic(ind)) for ind in inds]
        )
        self.selected_fitness_ = score

        return X, score
